# Aplicación Meteorología Sinóptica — GOES, GFS, ECMWF y ASCAT

**Visor interactivo para Google Colab.** Incluye pronóstico numérico, Frentes fríos (FRAM con GFS 0.50°) y descargas de satélites europeos EUMETSAT/Metop/Sentinel-3, mapas GFS con barbas, GOES, vientos derivados DMW, GeoColor, GLM y consultas ASCAT históricas de NOAA STAR y ENDVI10 LSA SAF (NDVI por década, coordenadas y series).

Al iniciar, Colab mostrará el visor integrado y la celda imprimirá el enlace público de acceso cuando Gradio al final de esta aplicaicón. Mantenga la celda final en ejecución mientras utilice el visor. Los datos meteorológicos se descargan solamente al pulsar un botón.


Daniel Poleo Brito
dpoleo@imn.ac.cr



Joel Ixcamparij  López
cjixcamparij@insivumeh.gob.gt



In [ ]:
# Dependencias de Colab: Gradio 6.31.0 con soporte actualizado para el visor integrado.
# En un runtime limpio, ejecuta todas las celdas en orden. Si Colab ya tenía otra versión
# de Gradio importada antes de instalar, reinicia el runtime una vez y vuelve a ejecutar.
%pip install -q "gradio==6.31.0" herbie-data ecmwf-opendata eumdac xarray netCDF4 cfgrib eccodes cartopy metpy boto3 requests scipy pandas pillow


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Código de la aplicación

Las siguientes celdas definen las funciones. **La interfaz se inicia una sola vez** en la última celda.


In [ ]:
"""
Aplicación interactiva para los productos del notebook
"Prácticas_Meteorologia_Sinoptica_2_2026.ipynb".

Diseñada para ejecutarse en Google Colab. La fecha, el modelo, el período y
los productos se seleccionan desde una interfaz Gradio; no es necesario
editar las fechas dentro de las celdas originales.

Fuentes empleadas por los ejemplos originales:
    - GOES-R en los buckets públicos de NOAA/AWS.
    - MIMIC TPW v2 de CIMSS.
    - ALPW de NOAA/JPSS.
    - GFS mediante Herbie.
    - ECMWF IFS mediante ECMWF Open Data.

La aplicación no usa las celdas del notebook como código ejecutable. Reutiliza
sus productos, consultas y recetas de procesamiento en una capa de ejecución
parametrizada, lo que permite procesar fechas actuales o históricas por lotes.
"""

from __future__ import annotations

import json
import math
import re
import traceback
import zipfile
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable
from uuid import uuid4

import boto3
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import requests
import xarray as xr
from botocore import UNSIGNED
from botocore.config import Config
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap, Normalize
from scipy.ndimage import gaussian_filter, zoom

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# ============================================================================
# Configuración general
# ============================================================================

UTC = timezone.utc
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
APP_DIR = BASE_DIR / "meteorologia_sinoptica_app"
DATA_DIR = APP_DIR / "datos"
OUTPUT_DIR = APP_DIR / "salidas"
GOES_DIR = DATA_DIR / "goes"
MIMIC_DIR = DATA_DIR / "mimic"
ALPW_DIR = DATA_DIR / "alpw"
GFS_DIR = DATA_DIR / "gfs"
ECMWF_DIR = DATA_DIR / "ecmwf"

for _directory in (APP_DIR, DATA_DIR, OUTPUT_DIR, GOES_DIR, MIMIC_DIR, ALPW_DIR, GFS_DIR, ECMWF_DIR):
    _directory.mkdir(parents=True, exist_ok=True)


REGIONS: dict[str, list[float]] = {
    "Centroamérica y Caribe": [-100.0, -55.0, 0.0, 30.0],
    "Frentes fríos — dominio de referencia": [-120.0, -50.0, -5.0, 35.0],
    "Caribe": [-90.0, -55.0, 8.0, 25.0],
    "Centroamérica": [-95.0, -77.0, 5.0, 20.0],
    "Sudamérica": [-85.0, -30.0, -40.0, 15.0],
    "América": [-130.0, -30.0, -45.0, 45.0],
    "Personalizada": [-100.0, -55.0, 0.0, 30.0],
}

MODEL_CHOICES: list[tuple[str, str]] = [
    ("GFS 0.25°", "gfs025"),
    ("GFS 0.50°", "gfs050"),
    ("ECMWF IFS Open Data", "ecmwf"),
]

GFS_PRODUCTS = {
    "gfs025": "pgrb2.0p25",
    "gfs050": "pgrb2.0p50",
}

SCOPE_CHOICES: list[tuple[str, str]] = [
    ("Disco completo", "full"),
    ("CONUS", "conus"),
    ("Mesoescala M1", "m1"),
    ("Mesoescala M2", "m2"),
]

SCOPE_CONFIG = {
    "full": ("ABI-L2-CMIPF", None),
    "conus": ("ABI-L2-CMIPC", None),
    "m1": ("ABI-L2-CMIPM", "M1"),
    "m2": ("ABI-L2-CMIPM", "M2"),
}

APP_CSS = """
:root { color-scheme: dark; }
.gradio-container {
  max-width: 1500px !important; margin: 0 auto !important;
  padding: 22px clamp(14px, 3vw, 44px) 48px !important;
  background: #070b14 !important; color: #f5f7fb !important;
  font-family: -apple-system, BlinkMacSystemFont, "SF Pro Display", "SF Pro Text", "Helvetica Neue", Arial, sans-serif !important;
}
.app-header {
  position: relative; overflow: hidden;
  padding: 34px clamp(22px, 4vw, 48px) !important;
  margin: 5px 0 20px !important; border: 1px solid #263750 !important;
  border-radius: 28px !important; color: #fff !important;
  background:
    radial-gradient(ellipse at 88% 15%, rgba(10,132,255,.42), transparent 34%),
    radial-gradient(ellipse at 75% 115%, rgba(90,200,250,.18), transparent 30%),
    linear-gradient(118deg, #111827 0%, #0c1d35 58%, #102b4c 100%) !important;
  box-shadow: 0 22px 54px rgba(0,0,0,.34) !important;
}
.app-header h1 { color: #fff !important; font-size: clamp(32px, 4.2vw, 50px) !important; letter-spacing: -.04em !important; margin: 2px 0 8px !important; }
.app-header p { color: #d5e5f8 !important; font-size: 15px !important; margin: 0 !important; }
.eyebrow { color: #64d2ff !important; font-size: 11px !important; font-weight: 750 !important; letter-spacing: .14em !important; text-transform: uppercase !important; }
.nav-bar { gap: 9px !important; padding: 9px !important; margin: 0 0 23px !important; border: 1px solid #263246 !important; border-radius: 20px !important; background: rgba(18,26,40,.92) !important; box-shadow: 0 9px 28px rgba(0,0,0,.25) !important; }
.nav-item button { min-height: 48px !important; border-radius: 15px !important; border: 1px solid #34445c !important; font-size: 14px !important; font-weight: 680 !important; transition: transform .18s ease, background .18s ease, box-shadow .18s ease !important; }
.nav-item button:hover { transform: scale(1.035); box-shadow: 0 8px 20px rgba(10,132,255,.2) !important; }
.page-shell { padding: 4px 0 20px !important; }
.page-heading { margin: 4px 0 18px !important; }
.page-heading h2 { color: #f5f7fb !important; font-size: 29px !important; letter-spacing: -.03em !important; }
.page-heading p { color: #b5c0d0 !important; }
.panel, .product-group {
  padding: 18px !important; margin: 11px 0 !important;
  border: 1px solid #29374d !important; border-radius: 20px !important;
  background: linear-gradient(145deg, #121c2d, #0d1523) !important;
  color: #f5f7fb !important; box-shadow: 0 9px 24px rgba(0,0,0,.2) !important;
}
.product-group summary, .product-group button[aria-expanded] { color: #f5f7fb !important; font-weight: 700 !important; }
.dashboard-grid { gap: 16px !important; }
.dashboard-card button {
  position: relative; overflow: hidden; min-height: 158px !important;
  padding: 24px !important; border: 1px solid #31445e !important;
  border-radius: 23px !important;
  background:
    radial-gradient(circle at 88% 25%, rgba(90,200,250,.24), transparent 26%),
    linear-gradient(140deg, #1b2c45 0%, #111b2b 70%, #0d1522 100%) !important;
  color: #f8fbff !important; text-align: left !important;
  white-space: pre-wrap !important; font-size: 18px !important;
  font-weight: 720 !important; box-shadow: 0 12px 30px rgba(0,0,0,.28) !important;
  transition: transform .2s ease, box-shadow .2s ease, border-color .2s ease !important;
}
.dashboard-card button:hover, .dashboard-card button:focus-visible {
  transform: scale(1.035); border-color: #4fc3f7 !important;
  box-shadow: 0 18px 42px rgba(0,113,227,.25) !important;
}
.product-tiles .wrap {
  display: grid !important; grid-template-columns: repeat(auto-fill, minmax(205px, 1fr)) !important;
  gap: 12px !important; align-items: stretch !important;
}
.product-tiles .wrap label, .product-tiles label, .product-tiles [role="checkbox"] {
  position: relative !important; display: flex !important; align-items: flex-end !important;
  min-height: 102px !important; padding: 17px 18px !important; overflow: hidden !important;
  border: 1px solid #33435b !important; border-radius: 18px !important;
  color: #f5f7fb !important;
  background:
    radial-gradient(circle at 92% 18%, rgba(90,200,250,.2), transparent 25%),
    linear-gradient(145deg, #21334e, #121d2e 74%) !important;
  box-shadow: 0 7px 20px rgba(0,0,0,.2) !important;
  transition: transform .18s ease, border-color .18s ease, box-shadow .18s ease !important;
}
.product-tiles .wrap label:hover, .product-tiles label:hover, .product-tiles [role="checkbox"]:hover {
  transform: scale(1.025); border-color: #75d6ff !important;
}
.product-tiles label:has(input:checked), .product-tiles [role="checkbox"][aria-checked="true"] {
  border: 2px solid #0a84ff !important;
  background:
    radial-gradient(circle at 92% 18%, rgba(90,200,250,.34), transparent 28%),
    linear-gradient(145deg, #184b78, #122b49 78%) !important;
  box-shadow: 0 0 0 2px rgba(10,132,255,.18), 0 12px 28px rgba(0,113,227,.18) !important;
}
.product-tiles label span, .product-tiles [role="checkbox"] span { color: #f5f7fb !important; line-height: 1.35 !important; }
.product-tiles input[type="checkbox"] { accent-color: #0a84ff !important; }
.choice-cards .wrap {
  display: flex !important; flex-wrap: wrap !important; gap: 10px !important;
}
.choice-cards .wrap label, .choice-cards label,
.choice-cards [role="checkbox"] {
  display: flex !important; align-items: center !important; gap: 10px !important;
  min-height: 48px !important; padding: 11px 15px !important;
  border: 1px solid #3a4a62 !important; border-radius: 15px !important;
  color: #f5f7fb !important; background: linear-gradient(145deg, #1a2940, #111b2c) !important;
  cursor: pointer !important; pointer-events: auto !important;
  transition: transform .16s ease, border-color .16s ease, background .16s ease, box-shadow .16s ease !important;
}
.choice-cards .wrap label:hover, .choice-cards label:hover,
.choice-cards [role="checkbox"]:hover {
  transform: translateY(-1px); border-color: #64d2ff !important;
}
.choice-cards label:has(input:checked),
.choice-cards [role="checkbox"][aria-checked="true"] {
  border: 2px solid #0a84ff !important; background: #153d63 !important;
  box-shadow: 0 0 0 2px rgba(10,132,255,.18), 0 7px 18px rgba(0,113,227,.2) !important;
}
.choice-cards input[type="checkbox"] {
  width: 18px !important; height: 18px !important; margin: 0 !important;
  flex: 0 0 18px !important; accent-color: #0a84ff !important; cursor: pointer !important;
}
.choice-cards label span, .choice-cards [role="checkbox"] span {
  color: #f5f7fb !important; line-height: 1.25 !important;
}
button.primary, .gradio-button.primary {
  min-height: 50px !important; border: 1px solid #2d9bff !important;
  border-radius: 15px !important; background: #0a84ff !important; color: #fff !important;
  font-size: 15px !important; font-weight: 720 !important;
  box-shadow: 0 7px 19px rgba(10,132,255,.27) !important;
}
button.primary:hover, .gradio-button.primary:hover { background: #3297ff !important; transform: translateY(-1px); }
button.secondary, .gradio-button.secondary {
  min-height: 43px !important; border: 1px solid #35445a !important;
  border-radius: 13px !important; background: #202b3d !important;
  color: #f2f6fc !important; font-weight: 620 !important;
}
button.secondary:hover, .gradio-button.secondary:hover { background: #2a3b54 !important; }
input, textarea, select, .wrap, .form {
  border-radius: 12px !important; color: #f3f6fb !important;
}
input, textarea { background: #111b2b !important; border-color: #34445a !important; }
label, .label { color: #e8edf5 !important; font-weight: 620 !important; }
input[type="checkbox"], input[type="radio"] { accent-color: #0a84ff !important; }
input[type="date"], input[type="datetime-local"] { color-scheme: dark; }
.quick-date button { min-height: 40px !important; border-radius: 999px !important; }
.output-gallery { border: 1px solid #29374d !important; border-radius: 19px !important; background: #101827 !important; }
footer { display: none !important; }
"""


_ACTIVE_DEMO = None


@dataclass(frozen=True)
class MapSpec:
    identifier: str
    label: str
    family: str
    required_model: str | None = None
    query: str | None = None


# Productos derivados de modelo presentes en las series 3, 4, 5 y 6 del
# notebook. Todos los productos GFS pueden ejecutarse con 0.25° o 0.50°; la
# resolución se elige en la interfaz.
NWP_SPECS: tuple[MapSpec, ...] = (
    MapSpec("gfs_prmsl", "GFS — Presión a nivel medio del mar", "gfs", query=":PRMSL:"),
    MapSpec("gfs_pwat", "GFS — Agua precipitable", "gfs", query=":PWAT:"),
    MapSpec("gfs_rh925", "GFS — Humedad relativa a 925 hPa", "gfs", query=":RH:925 mb"),
    MapSpec("gfs_thickness", "GFS — Espesor 1000–925 hPa", "gfs", query=":HGT:(1000|925) mb"),
    MapSpec("gfs_delta_t", "GFS — Cambio de temperatura 1000 hPa (6 h)", "gfs", query=":TMP:1000 mb"),
    MapSpec("gfs_wind500", "GFS — Viento 500 hPa + líneas de corriente", "gfs", query=":(U|V)GRD:500 mb"),
    MapSpec("gfs_wind925", "GFS — Viento 925 hPa + vectores", "gfs", query=":(U|V)GRD:925 mb"),
    MapSpec("gfs_wind850", "GFS — Viento 850 hPa + barbas", "gfs", query=":(U|V)GRD:850 mb"),
    MapSpec("gfs_wind_mean", "GFS — Viento medio troposférico 850–300 hPa", "gfs", query=":(U|V)GRD:(850|800|750|700|650|600|550|500|450|400|350|300) mb"),
    MapSpec("gfs_shear", "GFS — Cizalla vertical 850–250 hPa", "gfs", query=":(U|V)GRD:(850|250) mb"),
    MapSpec("gfs_td1000", "GFS — Temperatura de rocío 1000 hPa", "gfs", query=":(TMP|RH):1000 mb"),
    MapSpec("gfs_vort925", "GFS — Vorticidad relativa 925 hPa", "gfs", query=":(U|V)GRD:925 mb"),
    MapSpec("gfs_div925", "GFS — Divergencia/convergencia 925 hPa + barbas", "gfs", query=":(U|V)GRD:925 mb"),
    MapSpec("gfs_divmean", "GFS — Divergencia/convergencia media 1000–925 hPa", "gfs", query=":(U|V)GRD:(1000|925) mb"),
    MapSpec("gfs_mfc", "GFS — Convergencia de flujo de humedad 1000–850 hPa", "gfs", query=":(U|V)GRD:(1000|850) mb"),
    MapSpec("gfs_gdi", "GFS — Índice Galvez-Davison (GDI)", "gfs"),
    MapSpec("gfs_fram", "Frentes fríos — FRAM con GFS 0.50°", "gfs", required_model="gfs050"),
    MapSpec("ecmwf_msl", "ECMWF IFS — Presión a nivel medio del mar", "ecmwf"),
    MapSpec("ecmwf_pwat", "ECMWF IFS — Agua precipitable", "ecmwf"),
    MapSpec("ecmwf_wind10", "ECMWF IFS — Viento a 10 m (velocidad y vectores)", "ecmwf"),
    MapSpec("ecmwf_t2m", "ECMWF IFS — Temperatura a 2 m", "ecmwf"),
)

SATELLITE_SPECS: tuple[tuple[str, str], ...] = (
    # El orden de los seis primeros productos coincide con las escalas
    # entregadas por el usuario: B13, B08, B09, B10, sandwich y rayos.
    ("abi_b13", "1 · GOES B13 IR limpio (10.3 µm) — °C"),
    ("abi_b08", "2 · GOES B08 vapor de agua (6.2 µm) — °C"),
    ("abi_b09", "3 · GOES B09 vapor de agua medio (6.9 µm) — °C"),
    ("abi_b10", "4 · GOES B10 vapor de agua bajo (7.3 µm) — °C"),
    ("sandwich", "5 · GOES sandwich B02 + B13 — escala IR °C"),
    ("glm_instant", "6 · GOES Rayos GLM — puntos pequeños"),
    ("geocolor", "GOES GeoColor — color diurno / IR nocturno"),
    ("abi_b02", "GOES B02 visible (0.64 µm) — alta resolución"),
    ("rgb_day", "GOES — RGB fase de nubes diurna"),
    ("rgb_day_glm", "GOES — RGB diurna + GLM acumulado"),
    ("rgb_night", "GOES — RGB microfísica nocturna"),
    ("glm_accum", "GOES — GLM acumulado (5 minutos)"),
    ("mimic_tpw", "MIMIC TPW v2 — Agua precipitable"),
    ("alpw", "CIRA/JPSS ALPW — Agua precipitable por capas"),
    ("fires", "GOES — Detección de incendios FDCF"),
    ("dmw", "GOES — Banda 13 + vientos derivados DMW"),
)

SATELLITE_LABELS = dict(SATELLITE_SPECS)
NWP_LABELS = {spec.identifier: spec.label for spec in NWP_SPECS}

SATELLITE_SOURCE_NOTES = {
    "abi_b02": "GOES ABI L2 CMIP — B02 reflectiva, recorte de alta resolución",
    "abi_b08": "GOES ABI L2 CMIP — B08 térmica convertida a °C",
    "abi_b09": "GOES ABI L2 CMIP — B09 térmica convertida a °C",
    "abi_b10": "GOES ABI L2 CMIP — B10 térmica convertida a °C",
    "abi_b13": "GOES ABI L2 CMIP — B13 térmica convertida a °C",
    "geocolor": "GOES ABI L2 CMIP — B01/B02/B03 de día + B07/B13 de noche",
    "rgb_day": "GOES ABI L2 CMIP — B13 + B02 + B05",
    "rgb_day_glm": "GOES ABI L2 CMIP + GOES GLM L2 LCFA",
    "rgb_night": "GOES ABI L2 CMIP — B07 + B13 + B15",
    "glm_instant": "GOES GLM L2 LCFA — rayos como puntos pequeños",
    "glm_accum": "GOES GLM L2 LCFA — acumulación temporal",
    "mimic_tpw": "MIMIC TPW v2 de CIMSS (producto separado de GOES)",
    "alpw": "NOAA/JPSS ALPW (producto separado de GOES)",
    "fires": "GOES ABI L2 FDCF/FDCC/FDCM — máscara y potencia de fuego",
    "sandwich": "GOES ABI L2 CMIP — B02 + B13 con paleta IR de la referencia",
    "dmw": "GOES ABI L2 CMIP B13 + ABI L2 DMW C14",
}

# Paletas copiadas de las barras de referencia adjuntas. Se mantienen como
# constantes en el código para que la aplicación sea autónoma en Colab y no
# dependa de que las imágenes de ejemplo sigan montadas.
ABI_C13_TICKS = (-90, -70, -50, -30, -15, 0, 15, 30)
ABI_WV_TICKS = (-100, -80, -60, -40, -20, 0, 20, 40)

ABI_C13_COLORS = (
    "#ffffff", "#ac2d9a", "#e4e4e4", "#232323", "#9d0000",
    "#ff5e00", "#dcff00", "#18ff00", "#006c42", "#000d7b",
    "#0072b2", "#00cde3", "#fafafa", "#ededed", "#e1e1e1",
    "#d4d4d4", "#c7c7c7", "#b9b9b9", "#a0a0a0", "#868686",
    "#6d6d6d", "#545454", "#3b3b3b", "#222222", "#090909",
)

ABI_WV_COLORS = (
    "#000032", "#000032", "#9d9d9d", "#58584c", "#efef07",
    "#af0000", "#01de00", "#008200", "#0000c1", "#4d7fb1",
    "#d1d1d1", "#959595", "#434343", "#9b3800", "#a00000",
    "#6c2626", "#c1c100", "#a9a900", "#919100", "#797900",
    "#616100", "#494900", "#313100", "#181800", "#000000",
)

ABI_C13_CMAP = LinearSegmentedColormap.from_list(
    "goes_c13_reference",
    list(zip(np.linspace(0, 1, len(ABI_C13_COLORS)), ABI_C13_COLORS)),
)
ABI_WV_CMAP = LinearSegmentedColormap.from_list(
    "goes_water_vapor_reference",
    list(zip(np.linspace(0, 1, len(ABI_WV_COLORS)), ABI_WV_COLORS)),
)

ABI_THERMAL_SCALES: dict[int, dict[str, Any]] = {
    8: {"vmin": -100.0, "vmax": 40.0, "ticks": ABI_WV_TICKS, "cmap": ABI_WV_CMAP},
    9: {"vmin": -100.0, "vmax": 40.0, "ticks": ABI_WV_TICKS, "cmap": ABI_WV_CMAP},
    10: {"vmin": -100.0, "vmax": 40.0, "ticks": ABI_WV_TICKS, "cmap": ABI_WV_CMAP},
    13: {"vmin": -90.0, "vmax": 30.0, "ticks": ABI_C13_TICKS, "cmap": ABI_C13_CMAP},
}

# La referencia de la sexta imagen identifica los rayos con un punto azul.
# Los tres niveles GLM conservan una pequeña diferencia cromática para que
# no se mezclen los instrumentos, pero todos se dibujan deliberadamente
# pequeños y con el azul de la leyenda como color dominante.
GLM_POINT_COLORS = {"event": "#6e83e8", "group": "#3f5fc9", "flash": "#2446b8"}


# ============================================================================
# Utilidades de fechas, región y archivos
# ============================================================================

def utc_now_naive() -> datetime:
    """Hora UTC sin tzinfo, formato compatible con APIs meteorológicas."""

    return datetime.now(UTC).replace(tzinfo=None, second=0, microsecond=0)


def parse_datetime(value: Any, *, allow_latest: bool = False) -> datetime | str:
    if isinstance(value, datetime):
        return value.replace(tzinfo=None)
    text = str(value or "").strip()
    if allow_latest and text.lower() == "latest":
        return "latest"
    text = text.replace("Z", "").strip()
    candidates = (
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M",
        "%Y-%m-%d",
        "%Y%m%d%H%M",
    )
    for fmt in candidates:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass
    try:
        parsed = datetime.fromisoformat(text)
        if parsed.tzinfo is not None:
            parsed = parsed.astimezone(UTC).replace(tzinfo=None)
        return parsed
    except ValueError:
        pass
    raise ValueError(
        f"Fecha no válida: {value!r}. Use YYYY-MM-DD o YYYY-MM-DD HH:MM (UTC)."
    )


def parse_forecast_run(date_text: Any, cycle: Any) -> datetime:
    parsed = parse_datetime(date_text)
    if isinstance(parsed, str):
        raise ValueError("La fecha de corrida no puede ser 'latest'.")
    try:
        hour = int(str(cycle).replace("Z", ""))
    except ValueError as exc:
        raise ValueError("El ciclo debe ser 00, 06, 12 o 18 UTC.") from exc
    if hour not in (0, 6, 12, 18):
        raise ValueError("El ciclo debe ser 00, 06, 12 o 18 UTC.")
    return parsed.replace(hour=hour, minute=0, second=0, microsecond=0)


def parse_hours(text: Any, *, default: Iterable[int] = (0,)) -> list[int]:
    if text is None or str(text).strip() == "":
        values = list(default)
    elif isinstance(text, (list, tuple, set)):
        values = []
        for item in text:
            try:
                values.append(int(float(item)))
            except (TypeError, ValueError) as exc:
                raise ValueError(f"Hora de pronóstico no válida: {item!r}.") from exc
    else:
        raw = str(text).replace(";", ",").replace(" ", ",")
        values = []
        for item in raw.split(","):
            item = item.strip()
            if not item:
                continue
            try:
                values.append(int(float(item)))
            except ValueError as exc:
                raise ValueError(f"Hora de pronóstico no válida: {item!r}.") from exc
    values = sorted(set(values))
    if not values:
        raise ValueError("Debe indicar al menos una hora de pronóstico.")
    if any(value < 0 or value > 384 for value in values):
        raise ValueError("Las horas de pronóstico deben estar entre 0 y 384.")
    return values


def resolve_extent(region: str, min_lon: Any, max_lon: Any, min_lat: Any, max_lat: Any) -> list[float]:
    if region != "Personalizada":
        return list(REGIONS.get(region, REGIONS["Centroamérica y Caribe"]))
    try:
        extent = [float(min_lon), float(max_lon), float(min_lat), float(max_lat)]
    except (TypeError, ValueError) as exc:
        raise ValueError("Los límites personalizados deben ser numéricos.") from exc
    if extent[0] >= extent[1] or extent[2] >= extent[3]:
        raise ValueError("Use min_lon < max_lon y min_lat < max_lat.")
    if extent[0] < -180 or extent[1] > 180 or extent[2] < -90 or extent[3] > 90:
        raise ValueError("Los límites deben estar dentro de longitudes ±180° y latitudes ±90°.")
    return extent


def iter_times(start: datetime, end: datetime, step: timedelta) -> list[datetime]:
    if end < start:
        raise ValueError("El período final debe ser posterior al inicial.")
    result: list[datetime] = []
    current = start
    while current <= end:
        result.append(current)
        current += step
    return result


def safe_name(value: Any) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "producto"


def make_run_dir(prefix: str) -> Path:
    name = f"{safe_name(prefix)}_{utc_now_naive().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:6]}"
    directory = OUTPUT_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def write_bundle(
    run_dir: Path,
    image_files: list[str],
    source_files: list[str],
    logs: list[str],
    configuration: dict[str, Any],
    archive_name: str,
) -> str | None:
    all_files: list[Path] = []
    seen: set[str] = set()
    for item in [*image_files, *source_files]:
        path = Path(item)
        if not path.exists() or not path.is_file() or str(path) in seen:
            continue
        seen.add(str(path))
        all_files.append(path)

    manifest = run_dir / "configuracion.json"
    manifest.write_text(json.dumps(configuration, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    log_file = run_dir / "registro_ejecucion.txt"
    log_file.write_text("\n".join(logs), encoding="utf-8")
    all_files.extend([manifest, log_file])

    if not all_files:
        return None
    archive = run_dir / f"{safe_name(archive_name)}.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in all_files:
            try:
                arcname = path.relative_to(run_dir)
            except ValueError:
                arcname = Path("datos") / path.name
            zf.write(path, arcname=str(arcname))
    return str(archive)


def result_tuple(image_files: list[str], archive: str | None, logs: list[str]) -> tuple[list[Any], str | None, str]:
    # La galería conserva el orden producto → fechas/plazos y rotula cada mapa.
    known_products = {**NWP_LABELS, **SATELLITE_LABELS}
    previews = []
    for filename in image_files[:60]:
        path = Path(filename)
        stem = path.stem
        caption = stem.replace("_", " ")
        for identifier, label in sorted(known_products.items(), key=lambda item: len(item[0]), reverse=True):
            if stem.startswith(identifier + "_"):
                caption = f"{label} · {stem[len(identifier) + 1:].replace('_', ' ')}"
                break
        previews.append((str(path), caption))
    return previews, archive, "\n".join(logs)


In [ ]:
# ============================================================================
# Descarga de GOES-R sin depender de GDAL
# ============================================================================

_S3_CLIENT = None
_S3_KEYS_CACHE: dict[tuple[str, str], list[str]] = {}


def goes_s3_client():
    global _S3_CLIENT
    if _S3_CLIENT is None:
        _S3_CLIENT = boto3.client(
            "s3",
            config=Config(
                signature_version=UNSIGNED,
                connect_timeout=30,
                read_timeout=120,
                retries={"max_attempts": 3, "mode": "standard"},
            ),
        )
    return _S3_CLIENT


def _looks_like_netcdf(path: Path) -> bool:
    """Comprueba que una descarga no sea un archivo parcial o una página HTML."""

    try:
        if not path.is_file() or path.stat().st_size < 1024:
            return False
        with path.open("rb") as stream:
            signature = stream.read(8)
        # NetCDF clásico: CDF; NetCDF4/HDF5: \x89HDF.
        return signature.startswith(b"CDF") or signature == b"\x89HDF\r\n\x1a\n"
    except OSError:
        return False


def _goes_keys(bucket: str, prefix: str) -> list[str]:
    cache_key = (bucket, prefix)
    if cache_key in _S3_KEYS_CACHE:
        return _S3_KEYS_CACHE[cache_key]
    keys: list[str] = []
    paginator = goes_s3_client().get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj.get("Key")
            if key:
                keys.append(key)
    _S3_KEYS_CACHE[cache_key] = keys
    return keys


def _goes_key_time(key: str) -> datetime | None:
    # En los nombres GOES la hora de inicio es YYYYJJJHHMMSS y, en muchos
    # productos, viene seguida por una décima de segundo. Por ejemplo:
    # s20262382100211 = 2026, día juliano 238, 21:00:21.1 UTC. La versión
    # anterior intentaba interpretar las 14 cifras como si fueran solo
    # segundos y por eso descartaba *todos* los archivos de NOAA.
    match = re.search(r"_s(\d{13,15})", key)
    if not match:
        return None
    try:
        return datetime.strptime(match.group(1)[:13], "%Y%j%H%M%S")
    except ValueError:
        return None


def _goes_patterns(product: str, satellite: str, band: int | None, mode: str, sector: str | None) -> list[str]:
    sat_number = satellite[1:]
    if product == "GLM-L2-LCFA":
        return [f"OR_{product}_G{sat_number}_s"]

    # La ruta S3 de mesoescala es ABI-L2-<producto>M, pero el nombre de
    # archivo lleva M1 o M2 (CMIPM1, FDCM1, DMWM1, ...).
    if product.endswith("M") and sector in ("M1", "M2"):
        product_tag = f"{product[:-1]}{sector}" if sector in ("M1", "M2") else product
    else:
        product_tag = product

    # CMIP y DMW tienen una variante por banda/componente en el nombre.
    # FDCF/FDCC/FDCM, en cambio, no llevan Cxx.
    if (product_tag.startswith("ABI-L2-CMIP") or product_tag.startswith("ABI-L2-DMW")) and band is not None:
        return [f"OR_{product_tag}-M{mode}C{int(band):02d}_G{sat_number}_s"]
    if product_tag.startswith("ABI-L2-DMW") and band is None:
        return [f"OR_{product_tag}-M{mode}C"]
    return [f"OR_{product_tag}-M{mode}_G{sat_number}_s"]


def _derived_product(scope: str, kind: str) -> tuple[str, str | None]:
    """Devuelve el producto GOES derivado correcto para el ámbito elegido.

    FDCF/DMWF son disco completo; FDCC/DMWC son CONUS; y FDCM/DMWM usan
    M1/M2. Mantener esta tabla separada evita pedir un producto ABI de una
    cobertura y luego graficarlo como si perteneciera a otra.
    """

    if scope not in SCOPE_CONFIG:
        raise ValueError(f"Ámbito ABI no válido: {scope!r}.")
    if kind == "fires":
        products = {
            "full": ("ABI-L2-FDCF", None),
            "conus": ("ABI-L2-FDCC", None),
            "m1": ("ABI-L2-FDCM", "M1"),
            "m2": ("ABI-L2-FDCM", "M2"),
        }
    elif kind == "dmw":
        products = {
            "full": ("ABI-L2-DMWF", None),
            "conus": ("ABI-L2-DMWC", None),
            "m1": ("ABI-L2-DMWM", "M1"),
            "m2": ("ABI-L2-DMWM", "M2"),
        }
    else:
        raise ValueError(f"Producto derivado no reconocido: {kind!r}.")
    return products[scope]


def download_goes(
    timestamp: datetime | str,
    satellite: str,
    product: str,
    band: int | None = None,
    *,
    sector: str | None = None,
    mode: str = "6",
    destination: Path | None = None,
    lookback_hours: int = 36,
    tolerance_minutes: float | None = None,
) -> str:
    """Descarga un archivo GOES-R y devuelve su ruta local.

    La búsqueda se realiza en el bucket público de NOAA y reutiliza archivos
    ya descargados. Para «latest» se buscan las últimas horas disponibles.
    """

    if satellite not in {"G16", "G17", "G18", "G19"}:
        raise ValueError("Satélite GOES no válido. Use G16, G17, G18 o G19.")
    destination = destination or GOES_DIR
    destination.mkdir(parents=True, exist_ok=True)
    bucket = f"noaa-goes{satellite[1:]}"
    is_latest = isinstance(timestamp, str) and timestamp.strip().lower() == "latest"
    target = utc_now_naive() if is_latest else parse_datetime(timestamp)
    if isinstance(target, str):
        raise ValueError("No se pudo interpretar la fecha GOES.")

    if tolerance_minutes is None:
        tolerance_minutes = 0.75 if product == "GLM-L2-LCFA" else (3.0 if product.endswith("M") else 12.0)

    # Se consultan una o varias horas. El caché evita repetir las listas S3.
    if is_latest:
        prefixes = []
        for offset in range(max(1, lookback_hours)):
            dt = target - timedelta(hours=offset)
            prefixes.append(f"{product}/{dt:%Y}/{dt:%j}/{dt:%H}/")
    else:
        prefixes = []
        for offset in (-1, 0, 1):
            dt = target + timedelta(hours=offset)
            prefixes.append(f"{product}/{dt:%Y}/{dt:%j}/{dt:%H}/")

    keys: list[str] = []
    for prefix in prefixes:
        keys.extend(_goes_keys(bucket, prefix))

    candidates: list[tuple[float, datetime, str]] = []
    # Los archivos públicos suelen existir en M3/M4/M6. Se permiten todos
    # esos modos para que una selección no falle por el modo de escaneo.
    modes = [str(mode), "6", "3", "4"]
    sectors = [sector] if sector else (["M1", "M2"] if product.endswith("M") else [None])
    patterns: list[str] = []
    for current_mode in dict.fromkeys(modes):
        for current_sector in sectors:
            patterns.extend(_goes_patterns(product, satellite, band, current_mode, current_sector))

    for key in keys:
        file_name = key.rsplit("/", 1)[-1]
        if not any(file_name.startswith(pattern) for pattern in patterns):
            continue
        file_time = _goes_key_time(key)
        if file_time is None:
            continue
        distance = abs((file_time - target).total_seconds()) / 60.0
        if is_latest:
            candidates.append((0.0, file_time, key))
        elif distance <= tolerance_minutes:
            candidates.append((distance, file_time, key))

    if not candidates:
        raise FileNotFoundError(
            f"No se encontró {product} para {target:%Y-%m-%d %H:%M:%S} UTC "
            f"en {satellite}. Revise la disponibilidad histórica o el sector."
        )

    if is_latest:
        _, _, selected_key = max(candidates, key=lambda item: item[1])
    else:
        _, _, selected_key = min(candidates, key=lambda item: (item[0], -item[1].timestamp()))
    local_path = destination / selected_key.rsplit("/", 1)[-1]
    if not _looks_like_netcdf(local_path):
        # Descargar con extensión temporal evita que una interrupción deje un
        # archivo aparentemente válido que luego bloquee todos los intentos.
        partial_path = local_path.with_name(local_path.name + ".part")
        if partial_path.exists():
            partial_path.unlink()
        goes_s3_client().download_file(bucket, selected_key, str(partial_path))
        if not _looks_like_netcdf(partial_path):
            if partial_path.exists():
                partial_path.unlink()
            raise IOError(f"La descarga GOES no produjo un NetCDF válido: {selected_key}")
        partial_path.replace(local_path)
    return str(local_path)


# ============================================================================
# Descargas MIMIC y ALPW
# ============================================================================

def _download_http_netcdf(url: str, local: Path, *, timeout: int | tuple[int, int]) -> None:
    """Descarga un NetCDF HTTP de forma atómica y valida su firma."""

    partial = local.with_name(local.name + ".part")
    if partial.exists():
        partial.unlink()
    with requests.get(url, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
    if not _looks_like_netcdf(partial):
        if partial.exists():
            partial.unlink()
        raise IOError(f"La URL no devolvió un NetCDF válido: {url}")
    partial.replace(local)


def download_mimic(timestamp: datetime | str, destination: Path | None = None) -> str:
    dt = parse_datetime(timestamp)
    if isinstance(dt, str):
        raise ValueError("MIMIC no admite 'latest'; indique una fecha.")
    # MIMIC v2 se publica cada hora. Se usa la hora más cercana hacia abajo.
    dt = dt.replace(minute=0, second=0, microsecond=0)
    destination = destination or MIMIC_DIR
    destination.mkdir(parents=True, exist_ok=True)
    filename = f"comp{dt:%Y%m%d}.{dt:%H%M}00.nc"
    local = destination / filename
    if _looks_like_netcdf(local):
        return str(local)
    url = f"https://bin.ssec.wisc.edu/pub/mtpw2/data/{dt:%Y%m}/{filename}"
    _download_http_netcdf(url, local, timeout=90)
    return str(local)


def download_alpw(timestamp: datetime | str, destination: Path | None = None) -> str:
    dt = parse_datetime(timestamp)
    if isinstance(dt, str):
        raise ValueError("ALPW no admite 'latest'; indique una fecha.")
    dt = dt.replace(minute=0, second=0, microsecond=0)
    destination = destination or ALPW_DIR
    destination.mkdir(parents=True, exist_ok=True)
    end_token = f"e{dt:%Y%m%d%H}"
    import xml.etree.ElementTree as ET

    namespace = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
    filename = None
    selected_prefix = None
    # El archivo que termina a las 00 UTC está dentro de la carpeta del día
    # anterior (cada ALPW cubre una ventana de nueve horas). Se revisan ambas
    # carpetas y se admite paginación del listado S3.
    for search_dt in (dt, dt - timedelta(days=1)):
        prefix = f"JPSS_Blended_Products/BHP_ALPW/{search_dt:%Y/%m/%d}/"
        continuation = None
        while True:
            params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
            if continuation:
                params["continuation-token"] = continuation
            response = requests.get("https://noaa-jpss.s3.amazonaws.com/", params=params, timeout=60)
            response.raise_for_status()
            root = ET.fromstring(response.content)
            elements = root.findall(".//s3:Key", namespace)
            if not elements:
                elements = [element for element in root.iter() if element.tag.endswith("Key")]
            for element in elements:
                name = element.text or ""
                if name.endswith(".nc") and end_token in name:
                    filename = name.rsplit("/", 1)[-1]
                    selected_prefix = prefix
                    break
            if filename is not None:
                break
            truncated = (root.findtext(".//s3:IsTruncated", default="false", namespaces=namespace) or "false").lower() == "true"
            continuation = root.findtext(".//s3:NextContinuationToken", default="", namespaces=namespace)
            if not truncated or not continuation:
                break
        if filename is not None:
            break
    if filename is None or selected_prefix is None:
        raise FileNotFoundError(f"No se encontró ALPW con fecha final {end_token}.")
    local = destination / filename
    if _looks_like_netcdf(local):
        return str(local)
    file_url = f"https://noaa-jpss.s3.amazonaws.com/{selected_prefix}{filename}"
    _download_http_netcdf(file_url, local, timeout=120)
    return str(local)


# ============================================================================
# Lectura y gráficos satelitales
# ============================================================================

def _satellite_info(path: str, variable: str = "CMI", extent: list[float] | None = None, max_pixels: int = 1300) -> dict[str, Any]:
    """Lee un producto ABI, recortando en la proyección nativa GOES.

    El recorte se hace antes del gráfico para no cargar a memoria todo el
    disco completo de alta resolución. No requiere GDAL.
    """

    ds = xr.open_dataset(path, mask_and_scale=True)
    try:
        if variable not in ds:
            lower = {name.lower(): name for name in ds.data_vars}
            selected = lower.get(variable.lower())
            if selected is None:
                raise KeyError(f"Variable {variable!r} no existe en {Path(path).name}.")
            variable = selected
        data_array = ds[variable].squeeze(drop=True)
        if data_array.ndim != 2:
            raise ValueError(f"La variable {variable} no es una matriz 2D.")
        # ABI estándar usa y/x. El uso de los dos últimos ejes como respaldo
        # permite abrir también productos con nombres de dimensión distintos.
        row_dim, col_dim = data_array.dims
        if "y" in data_array.dims and "x" in data_array.dims:
            row_dim, col_dim = "y", "x"
            data_array = data_array.transpose(row_dim, col_dim)
        x = np.asarray(ds["x"].values, dtype=float)
        y = np.asarray(ds["y"].values, dtype=float)
        if data_array.sizes[row_dim] != len(y) or data_array.sizes[col_dim] != len(x):
            raise ValueError(f"Grilla incompatible en {Path(path).name}: dato={data_array.shape}, x={len(x)}, y={len(y)}")
        projection_variable = ds.get("goes_imager_projection")
        attrs = dict(projection_variable.attrs) if projection_variable is not None else {}
        height = float(attrs.get("perspective_point_height", 35786023.0))
        central_longitude = float(attrs.get("longitude_of_projection_origin", -75.0))
        geo = ccrs.Geostationary(
            central_longitude=central_longitude,
            satellite_height=height,
            sweep_axis="x",
        )
        original_time = ds.attrs.get("time_coverage_start", "")

        x_m = x * height
        y_m = y * height
        row_indices = np.arange(len(y))
        col_indices = np.arange(len(x))
        if extent is not None:
            min_lon, max_lon, min_lat, max_lat = extent
            corner_lon = np.array([min_lon, max_lon, min_lon, max_lon], dtype=float)
            corner_lat = np.array([min_lat, min_lat, max_lat, max_lat], dtype=float)
            projected = geo.transform_points(ccrs.PlateCarree(), corner_lon, corner_lat)
            projected = projected[np.isfinite(projected).all(axis=1)]
            if len(projected):
                x_low, x_high = np.nanmin(projected[:, 0]), np.nanmax(projected[:, 0])
                y_low, y_high = np.nanmin(projected[:, 1]), np.nanmax(projected[:, 1])
                pad_x = max((x_high - x_low) * 0.04, 1000.0)
                pad_y = max((y_high - y_low) * 0.04, 1000.0)
                cols = np.where((x_m >= x_low - pad_x) & (x_m <= x_high + pad_x))[0]
                rows = np.where((y_m >= y_low - pad_y) & (y_m <= y_high + pad_y))[0]
                if len(cols) > 4:
                    col_indices = cols
                if len(rows) > 4:
                    row_indices = rows

        stride = max(1, int(math.ceil(max(len(row_indices), len(col_indices)) / max(1, int(max_pixels)))))
        row_indices = row_indices[::stride]
        col_indices = col_indices[::stride]
        # El recorte ocurre antes de .load(): una región pequeña ya no obliga a
        # materializar el disco completo en la RAM de Colab.
        data = np.asarray(
            data_array.isel({row_dim: row_indices, col_dim: col_indices}).load().values,
            dtype=np.float32,
        )
        x_selected = x_m[col_indices]
        y_selected = y_m[row_indices]
    finally:
        ds.close()

    if y_selected[0] > y_selected[-1]:
        image_extent = [float(x_selected[0]), float(x_selected[-1]), float(y_selected[-1]), float(y_selected[0])]
        origin = "upper"
    else:
        image_extent = [float(x_selected[0]), float(x_selected[-1]), float(y_selected[0]), float(y_selected[-1])]
        origin = "lower"
    return {
        "data": data,
        "geo": geo,
        "extent": image_extent,
        "origin": origin,
        "time": original_time,
        "x": x_selected,
        "y": y_selected,
        "height": height,
    }


def _satellite_time_text(path: str, fallback: str = "") -> str:
    try:
        ds = xr.open_dataset(path, decode_times=False)
        value = ds.attrs.get("time_coverage_start", "")
        ds.close()
        if value:
            value = str(value).replace("T", " ").replace("Z", "")
            return f"{value} UTC"
    except Exception:
        pass
    return fallback


def _base_map(extent: list[float], *, figsize: tuple[float, float] = (10.5, 7.5), facecolor: str = "white"):
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_facecolor(facecolor)
    ax.coastlines(resolution="110m", color="black", linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, edgecolor="black", linewidth=0.45)
    try:
        ax.add_feature(cfeature.LAKES, edgecolor="gray", facecolor="none", linewidth=0.25)
    except Exception:
        pass
    # Cartopy + Shapely recientes pueden fallar al dibujar gridlines con
    # etiquetas sobre una imagen en proyección geostacionaria. La malla sin
    # etiquetas conserva la referencia espacial y evita ese bloqueo en Colab.
    ax.gridlines(draw_labels=False, linestyle="--", linewidth=0.25, color="gray", alpha=0.65)
    return fig, ax


def _save_figure(fig, output: Path, *, dpi: int = 150) -> str:
    output.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.tight_layout()
        fig.savefig(output, dpi=dpi, bbox_inches="tight")
    finally:
        plt.close(fig)
    return str(output)


def _normalize_channel(data: np.ndarray, low: float, high: float, descending: bool = False) -> np.ndarray:
    values = np.asarray(data, dtype=np.float32)
    values = np.where(np.isfinite(values), values, np.nan)
    values = np.clip(values, min(low, high), max(low, high))
    if descending:
        result = (high - values) / (high - low)
    else:
        result = (values - low) / (high - low)
    return np.clip(result, 0, 1)


def _resize_channel(data: np.ndarray, target_shape: tuple[int, int]) -> np.ndarray:
    if data.shape == target_shape:
        return data
    factors = (target_shape[0] / data.shape[0], target_shape[1] / data.shape[1])
    resized = zoom(data, factors, order=1)
    return resized[: target_shape[0], : target_shape[1]]


def _abi_plot_values(data: np.ndarray, band: int) -> np.ndarray:
    """Convierte solo las bandas ABI infrarrojas a temperatura en °C.

    Las bandas 1–6 son reflectivas (0–1); las bandas 7–16 son térmicas y su
    variable CMI viene en kelvin. Aplicar Kelvin→C a una banda visible/NIR
    destruye el RGB, que era el error del producto diurno original.
    """

    values = np.asarray(data, dtype=np.float32)
    return values if band <= 6 else values - 273.15


def _get_goes_file(
    timestamp: datetime | str,
    satellite: str,
    scope: str,
    band: int | None,
    source_cache: dict[tuple[Any, ...], str],
    *,
    product_override: str | None = None,
    sector_override: str | None = None,
) -> str:
    product, sector = SCOPE_CONFIG[scope]
    product = product_override or product
    sector = sector_override if sector_override is not None else sector
    key = (str(timestamp), satellite, product, band, sector)
    if key not in source_cache:
        source_cache[key] = download_goes(timestamp, satellite, product, band, sector=sector)
    return source_cache[key]


def _plot_abi_channel(
    file_path: str,
    band: int,
    extent: list[float],
    output: Path,
    title: str,
    *,
    high_resolution: bool = False,
) -> str:
    use_high_resolution = high_resolution and band == 2
    info = _satellite_info(file_path, "CMI", extent, max_pixels=4200 if use_high_resolution else 1300)
    data = _abi_plot_values(info["data"], band)
    if band == 2:
        cmap, vmin, vmax, label = "Greys_r", 0.0, 1.0, "Reflectancia"
    else:
        scale = ABI_THERMAL_SCALES.get(band)
        if scale is None:
            scale = {"vmin": -100.0, "vmax": 40.0, "ticks": ABI_WV_TICKS, "cmap": ABI_WV_CMAP}
        cmap = scale["cmap"]
        vmin = scale["vmin"]
        vmax = scale["vmax"]
        label = "Temperatura de brillo (°C)"
    fig, ax = _base_map(extent)
    image = ax.imshow(
        data,
        origin=info["origin"],
        extent=info["extent"],
        transform=info["geo"],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="bilinear",
    )
    colorbar_kwargs: dict[str, Any] = {"label": label}
    if band != 2:
        colorbar_kwargs["ticks"] = scale["ticks"]
    fig.colorbar(image, ax=ax, pad=0.03, shrink=0.72, **colorbar_kwargs)
    ax.set_title(f"{title}\n{_satellite_time_text(file_path)}", loc="left", fontsize=11, fontweight="bold")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9)
    if use_high_resolution:
        ax.text(0.01, 0.02, "Recorte ABI de alta resolución", transform=ax.transAxes, fontsize=8, color="white", bbox={"facecolor": "#0b4ea2", "alpha": 0.75, "pad": 3})
    return _save_figure(fig, output, dpi=220 if use_high_resolution else 150)


def _plot_rgb(
    files: dict[int, str],
    recipe: str,
    extent: list[float],
    output: Path,
    title: str,
) -> str:
    infos = {band: _satellite_info(path, "CMI", extent) for band, path in files.items()}
    target_shape = infos[next(iter(infos))]["data"].shape
    channel: dict[int, np.ndarray] = {}
    for band, info in infos.items():
        data = _abi_plot_values(info["data"], band)
        channel[band] = _resize_channel(data, target_shape)

    if recipe == "day":
        red = _normalize_channel(channel[13], -53.5, 7.5, descending=True)
        green = _normalize_channel(channel[2], 0.0, 0.78)
        blue = _normalize_channel(channel[5], 0.01, 0.59)
    else:
        red = _normalize_channel(channel[15] - channel[13], -6.7, 2.6)
        green = _normalize_channel(channel[13] - channel[7], -3.1, 5.2)
        blue = _normalize_channel(channel[13], -29.6, 19.5)
    rgb = np.stack([red, green, blue], axis=2)
    rgb = np.nan_to_num(rgb, nan=0.0)
    fig, ax = _base_map(extent, facecolor="black")
    ax.imshow(
        rgb,
        origin=infos[next(iter(infos))]["origin"],
        extent=infos[next(iter(infos))]["extent"],
        transform=infos[next(iter(infos))]["geo"],
    )
    ax.set_title(f"{title}\n{_satellite_time_text(files[next(iter(files))])}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    return _save_figure(fig, output)


def _plot_rgb_with_glm(
    files: dict[int, str],
    glm_files: list[str],
    extent: list[float],
    output: Path,
    title: str,
) -> str:
    """Versión combinada de la celda RGB ABI + GLM del notebook."""

    infos = {band: _satellite_info(path, "CMI", extent) for band, path in files.items()}
    target_shape = infos[13]["data"].shape
    b13 = _resize_channel(_abi_plot_values(infos[13]["data"], 13), target_shape)
    b02 = _resize_channel(_abi_plot_values(infos[2]["data"], 2), target_shape)
    b05 = _resize_channel(_abi_plot_values(infos[5]["data"], 5), target_shape)
    red = _normalize_channel(b13, -53.5, 7.5, descending=True)
    green = _normalize_channel(b02, 0.0, 0.78)
    blue = _normalize_channel(b05, 0.01, 0.59)
    rgb = np.nan_to_num(np.stack([red, green, blue], axis=2), nan=0.0)
    e_lon, e_lat, _, _, f_lon, f_lat, time_label = _read_glm_points(glm_files)

    fig, ax = _base_map(extent, facecolor="black")
    ax.imshow(rgb, origin=infos[13]["origin"], extent=infos[13]["extent"], transform=infos[13]["geo"], zorder=1)
    mask = (
        np.isfinite(f_lon)
        & np.isfinite(f_lat)
        & (f_lon >= extent[0])
        & (f_lon <= extent[1])
        & (f_lat >= extent[2])
        & (f_lat <= extent[3])
    )
    if np.any(mask):
        ax.scatter(f_lon[mask], f_lat[mask], s=2.2, c=GLM_POINT_COLORS["flash"], edgecolors="none", alpha=0.9, transform=ccrs.PlateCarree(), zorder=5, label="Rayos")
        ax.legend(loc="upper right", fontsize=8)
    ax.set_title(f"{title}\n{_satellite_time_text(files[13])} — {time_label}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    return _save_figure(fig, output)


def _read_glm_points(files: list[str]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, str]:
    event_lon: list[np.ndarray] = []
    event_lat: list[np.ndarray] = []
    group_lon: list[np.ndarray] = []
    group_lat: list[np.ndarray] = []
    flash_lon: list[np.ndarray] = []
    flash_lat: list[np.ndarray] = []
    time_label = ""
    for path in files:
        ds = xr.open_dataset(path)
        try:
            time_label = _satellite_time_text(path, time_label)
            for target, name in (
                (event_lon, "event_lon"),
                (event_lat, "event_lat"),
                (group_lon, "group_lon"),
                (group_lat, "group_lat"),
                (flash_lon, "flash_lon"),
                (flash_lat, "flash_lat"),
            ):
                if name in ds:
                    target.append(np.asarray(ds[name].values).ravel())
        finally:
            ds.close()
    empty = np.array([], dtype=float)
    return (
        np.concatenate(event_lon) if event_lon else empty,
        np.concatenate(event_lat) if event_lat else empty,
        np.concatenate(group_lon) if group_lon else empty,
        np.concatenate(group_lat) if group_lat else empty,
        np.concatenate(flash_lon) if flash_lon else empty,
        np.concatenate(flash_lat) if flash_lat else empty,
        time_label,
    )


def _plot_glm(files: list[str], extent: list[float], output: Path, title: str) -> str:
    e_lon, e_lat, g_lon, g_lat, f_lon, f_lat, time_label = _read_glm_points(files)
    fig, ax = _base_map(extent, facecolor="black")
    for lon, lat, color, size, label in (
        (e_lon, e_lat, GLM_POINT_COLORS["event"], 0.8, "Eventos"),
        (g_lon, g_lat, GLM_POINT_COLORS["group"], 1.4, "Grupos"),
        (f_lon, f_lat, GLM_POINT_COLORS["flash"], 2.5, "Rayos"),
    ):
        mask = (
            np.isfinite(lon)
            & np.isfinite(lat)
            & (lon >= extent[0])
            & (lon <= extent[1])
            & (lat >= extent[2])
            & (lat <= extent[3])
        )
        if np.any(mask):
            ax.scatter(lon[mask], lat[mask], s=size, c=color, label=label, transform=ccrs.PlateCarree(), alpha=0.8, linewidths=0)
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(f"{title}\n{time_label}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    return _save_figure(fig, output)


def _read_geographic_grid(
    file_path: str,
    variables: tuple[str, ...],
) -> tuple[str, np.ndarray, np.ndarray, np.ndarray, str]:
    """Lee una grilla lat/lon de MIMIC o ALPW sin confundir metadatos."""

    ds = xr.open_dataset(file_path, mask_and_scale=True, decode_times=False)
    try:
        available = {name.lower(): name for name in ds.data_vars}
        variable = next((available[name.lower()] for name in variables if name.lower() in available), None)
        if variable is None:
            raise KeyError(
                f"No se encontró ninguna variable {variables} en {Path(file_path).name}. "
                f"Disponibles: {list(ds.data_vars)}"
            )
        data_array = ds[variable].squeeze(drop=True)
        if data_array.ndim != 2:
            raise ValueError(f"La variable geográfica {variable!r} no es una matriz 2D.")

        # Los dos productos usan lat/lon, pero MIMIC guarda sus coordenadas
        # como latArr/lonArr y ALPW las publica como lat/lon.
        lat_dim = next((dim for dim in data_array.dims if dim.lower() in {"lat", "latitude", "y"}), None)
        lon_dim = next((dim for dim in data_array.dims if dim.lower() in {"lon", "longitude", "x"}), None)
        if lat_dim is None or lon_dim is None:
            lat_dim, lon_dim = data_array.dims

        def coordinate_for(dim: str, aliases: tuple[str, ...]) -> np.ndarray:
            candidates = [*aliases, dim]
            for name in candidates:
                if name in ds.variables:
                    values = np.asarray(ds[name].values, dtype=float).squeeze()
                    if values.ndim == 1 and len(values) == data_array.sizes[dim]:
                        return values
            # Último recurso para archivos que omitan la variable de
            # coordenadas: usa índices solo si no hay otra información.
            return np.arange(data_array.sizes[dim], dtype=float)

        lat = coordinate_for(lat_dim, ("latArr", "latitude", "lat"))
        lon = coordinate_for(lon_dim, ("lonArr", "longitude", "lon"))
        data = np.asarray(data_array.transpose(lat_dim, lon_dim).load().values, dtype=float)
    finally:
        ds.close()

    if len(lat) != data.shape[0] or len(lon) != data.shape[1]:
        raise ValueError(f"Coordenadas incompatibles en {Path(file_path).name}: dato={data.shape}, lat={len(lat)}, lon={len(lon)}")
    if lat[0] > lat[-1]:
        lat = lat[::-1]
        data = data[::-1, :]
    if lon[0] > lon[-1]:
        lon = lon[::-1]
        data = data[:, ::-1]
    # Algunos proveedores expresan la longitud en 0–360; los gráficos de
    # esta aplicación usan −180–180.
    if np.nanmax(lon) > 180:
        lon = ((lon + 180) % 360) - 180
        order = np.argsort(lon)
        lon = lon[order]
        data = data[:, order]
    time_text = _satellite_time_text(file_path)
    return variable, data, lon, lat, time_text


def _plot_mimic(file_path: str, extent: list[float], output: Path) -> str:
    variable, data, lon, lat, time_text = _read_geographic_grid(file_path, ("tpwGrid",))
    colors = ["#bc8462", "#ae656f", "#a44a79", "#962e97", "#6158c5", "#2b8ffb", "#5fcdff", "#94fff0", "#a5ff94", "#fff88c", "#ffbf52", "#ec7b27", "#b84827", "#a1333d", "#bd5478", "#cc6a99", "#d982b8"]
    cmap = LinearSegmentedColormap.from_list("mimic_tpw", colors)
    fig, ax = _base_map(extent)
    image = ax.imshow(data, origin="lower", vmin=0, vmax=75, extent=[float(lon[0]), float(lon[-1]), float(lat[0]), float(lat[-1])], cmap=cmap, interpolation="bilinear", transform=ccrs.PlateCarree())
    fig.colorbar(image, ax=ax, pad=0.03, shrink=0.72, label="Agua precipitable total (mm)")
    subtitle = time_text or Path(file_path).name
    ax.set_title(f"MIMIC TPW v2 (CIMSS) — {variable}\n{subtitle}", loc="left", fontsize=11, fontweight="bold")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9)
    return _save_figure(fig, output)


def _plot_alpw(file_path: str, extent: list[float], output: Path) -> str:
    # Es la capa usada por la celda original del notebook. Las variables de
    # calidad se excluyen explícitamente para no graficarlas como agua.
    variable, data, lon, lat, time_text = _read_geographic_grid(file_path, ("ALPW_850_700",))
    colors = ["#000000", "#393939", "#818181", "#bec4ba", "#339831", "#056c47", "#1360a7", "#1b91fc", "#0ca7f0", "#00d6ef", "#5c92d8", "#8954d1", "#922fed", "#8b07a0", "#8d0069", "#8c0002", "#cc0000", "#e22a00", "#f56200", "#d25d00", "#9b3900", "#dc9e00", "#ffe300", "#fffa08", "#dd928c", "#e5a3a6"]
    cmap = LinearSegmentedColormap.from_list("alpw", colors)
    fig, ax = _base_map(extent, facecolor="black")
    image = ax.imshow(data, origin="lower", extent=[float(lon[0]), float(lon[-1]), float(lat[0]), float(lat[-1])], cmap=cmap, vmin=0, vmax=32, interpolation="bilinear", transform=ccrs.PlateCarree())
    fig.colorbar(image, ax=ax, pad=0.03, shrink=0.72, label="Agua precipitable por capa (mm)")
    subtitle = time_text or Path(file_path).name
    ax.set_title(f"CIRA/NOAA JPSS — {variable}\n{subtitle}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    return _save_figure(fig, output)


def _plot_fires(file_path: str, extent: list[float], output: Path) -> str:
    # FDCF es una detección puntual: no se puede reducir la matriz antes de
    # buscar los fuegos porque una máscara de 1 píxel podría desaparecer.
    mask_info = _satellite_info(file_path, "Mask", extent, max_pixels=6000)
    power_info = _satellite_info(file_path, "Power", extent, max_pixels=6000)
    mask = mask_info["data"]
    power = power_info["data"]
    valid = np.isfinite(mask) & (mask >= 10) & (mask <= 15) & np.isfinite(power)
    rows, cols = np.where(valid)
    points = ccrs.PlateCarree().transform_points(mask_info["geo"], mask_info["x"][cols], mask_info["y"][rows])
    lons, lats = points[:, 0], points[:, 1]
    point_valid = np.isfinite(lons) & np.isfinite(lats)
    point_valid &= (lons >= extent[0]) & (lons <= extent[1]) & (lats >= extent[2]) & (lats <= extent[3])
    frp = np.maximum(power[rows, cols], 0)[point_valid]
    lons, lats = lons[point_valid], lats[point_valid]
    fig, ax = _base_map(extent, facecolor="black")
    if len(frp):
        sizes = 18 + 70 * frp / max(float(np.nanpercentile(frp, 95)), 1.0)
        image = ax.scatter(lons, lats, c=frp, s=sizes, cmap="hot", edgecolors="white", linewidths=0.25, transform=ccrs.PlateCarree(), zorder=4)
        fig.colorbar(image, ax=ax, pad=0.03, shrink=0.72, label="Potencia radiativa del fuego (MW)")
        subtitle = f"{len(frp)} detecciones"
    else:
        subtitle = "Sin detecciones válidas en la región"
    product_match = re.search(r"ABI-L2-(FDC[FCM]\d?)", Path(file_path).name)
    fire_product = f"ABI-L2-{product_match.group(1)}" if product_match else "ABI-L2-FDCF"
    ax.set_title(f"GOES — Detección de incendios {fire_product}\n{subtitle} — {_satellite_time_text(file_path)}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    return _save_figure(fig, output)


def _plot_sandwich(
    files: dict[int, str],
    extent: list[float],
    output: Path,
    *,
    high_resolution: bool = False,
) -> str:
    max_pixels = 4200 if high_resolution else 1300
    info_vis = _satellite_info(files[2], "CMI", extent, max_pixels=max_pixels)
    info_ir = _satellite_info(files[13], "CMI", extent, max_pixels=max_pixels)
    vis = info_vis["data"]
    ir = _abi_plot_values(info_ir["data"], 13)
    fig, ax = _base_map(extent, facecolor="black")
    ax.imshow(vis, origin=info_vis["origin"], extent=info_vis["extent"], transform=info_vis["geo"], cmap="Greys_r", vmin=0, vmax=1, interpolation="bilinear")
    normalized = Normalize(vmin=-90.0, vmax=30.0, clip=True)(ir)
    rgba = ABI_C13_CMAP(np.nan_to_num(normalized, nan=0.0))
    rgba[..., 3] = np.where(np.isfinite(ir) & (ir < -40.0), 0.72, 0.0)
    ax.imshow(rgba, origin=info_ir["origin"], extent=info_ir["extent"], transform=info_ir["geo"], interpolation="bilinear")
    sm = plt.cm.ScalarMappable(norm=Normalize(vmin=-90.0, vmax=30.0), cmap=ABI_C13_CMAP)
    sm.set_array(ir)
    fig.colorbar(sm, ax=ax, pad=0.03, shrink=0.72, ticks=ABI_C13_TICKS, label="Temperatura de brillo B13 (°C)")
    ax.set_title(f"GOES — Sandwich B02 + B13 frío (< −40 °C)\n{_satellite_time_text(files[13])}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    if high_resolution:
        ax.text(0.01, 0.02, "B02/B13: recorte de alta resolución", transform=ax.transAxes, fontsize=8, color="white", bbox={"facecolor": "#0b4ea2", "alpha": 0.75, "pad": 3})
    return _save_figure(fig, output, dpi=220 if high_resolution else 150)


def _plot_dmw(files: dict[str, str], extent: list[float], output: Path) -> str:
    info = _satellite_info(files["b13"], "CMI", extent)
    ds = xr.open_dataset(files["dmw"], mask_and_scale=True, decode_times=False)
    try:
        def read_vector(name: str) -> np.ndarray:
            if name not in ds.variables:
                raise KeyError(f"El producto DMW no contiene la variable {name!r}.")
            return np.asarray(ds[name].values, dtype=float).ravel()

        lat = read_vector("lat")
        lon = read_vector("lon")
        speed = read_vector("wind_speed")
        direction = read_vector("wind_direction")
        pressure = read_vector("pressure") if "pressure" in ds.variables else np.full_like(speed, np.nan)
        dqf = read_vector("DQF") if "DQF" in ds.variables else np.zeros_like(speed)
    finally:
        ds.close()
    size = min(len(lat), len(lon), len(speed), len(direction), len(pressure), len(dqf))
    lat, lon, speed, direction, pressure, dqf = (array[:size] for array in (lat, lon, speed, direction, pressure, dqf))
    valid = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(speed) & np.isfinite(direction)
    valid &= np.isfinite(dqf) & (dqf <= 1)
    valid &= (lon >= extent[0]) & (lon <= extent[1]) & (lat >= extent[2]) & (lat <= extent[3])
    u = -speed * np.sin(np.deg2rad(direction))
    v = -speed * np.cos(np.deg2rad(direction))
    fig, ax = _base_map(extent)
    image = ax.imshow(_abi_plot_values(info["data"], 13), origin=info["origin"], extent=info["extent"], transform=info["geo"], cmap=ABI_C13_CMAP, vmin=-90, vmax=30, interpolation="bilinear")
    pressure_values = pressure.copy()
    finite_pressure = pressure_values[np.isfinite(pressure_values)]
    if finite_pressure.size and np.nanmedian(finite_pressure) > 2000:
        pressure_values = pressure_values / 100.0
    pressure_ranges = [
        (100, 249, "#3155d9", "100–249 hPa"),
        (250, 399, "#27a8e0", "250–399 hPa"),
        (400, 549, "#13a66b", "400–549 hPa"),
        (550, 699, "#d4bd14", "550–699 hPa"),
        (700, 849, "#ed6b32", "700–849 hPa"),
        (850, 1000, "#d43d8f", "850–1000 hPa"),
    ]
    drawn = 0
    for p_min, p_max, color, label in pressure_ranges:
        idx = valid & np.isfinite(pressure_values) & (pressure_values >= p_min) & (pressure_values <= p_max)
        indices = np.flatnonzero(idx)
        if indices.size == 0:
            continue
        indices = indices[::max(1, int(np.ceil(indices.size / 2200)))]
        ax.barbs(lon[indices], lat[indices], u[indices], v[indices],
                 length=4.5, linewidth=0.65, pivot="middle",
                 barbcolor=color, flagcolor=color,
                 sizes={"emptybarb": 0.25, "spacing": 0.18, "height": 0.5},
                 transform=ccrs.PlateCarree(), zorder=5)
        ax.plot([], [], color=color, label=label, linewidth=2.4)
        drawn += int(indices.size)
    if drawn == 0 and np.any(valid):
        indices = np.flatnonzero(valid)
        indices = indices[::max(1, int(np.ceil(indices.size / 2200)))]
        ax.barbs(lon[indices], lat[indices], u[indices], v[indices],
                 length=4.5, linewidth=0.65, pivot="middle", color="#e84a5f",
                 transform=ccrs.PlateCarree(), zorder=5)
        drawn = int(indices.size)
        ax.plot([], [], color="#e84a5f", label="Presión no disponible", linewidth=2.4)
    if drawn:
        ax.legend(loc="upper right", fontsize=8, framealpha=0.94, title="Presión DMW")
    ax.coastlines(resolution="110m", color="white", linewidth=0.65, zorder=6)
    ax.add_feature(cfeature.BORDERS, edgecolor="white", linewidth=0.4, zorder=6)
    fig.colorbar(image, ax=ax, pad=0.03, shrink=0.72, ticks=ABI_C13_TICKS, label="Temperatura de brillo B13 (°C)")
    ax.set_title(f"GOES Banda 13 + Vientos derivados DMW\n{drawn} barbas — {_satellite_time_text(files['b13'])}", loc="left", fontsize=11, fontweight="bold")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9)
    return _save_figure(fig, output)


def render_satellite_map(
    map_id: str,
    timestamp: datetime | str,
    satellite: str,
    scope: str,
    extent: list[float],
    output_dir: Path,
    source_cache: dict[tuple[Any, ...], str],
    logs: list[str],
    high_resolution: bool = False,
) -> tuple[str, list[str]]:
    """Descarga los archivos necesarios y genera un único mapa satelital."""

    label = SATELLITE_LABELS[map_id]
    output_dir.mkdir(parents=True, exist_ok=True)
    time_tag = safe_name(timestamp.strftime("%Y%m%dT%H%M") if isinstance(timestamp, datetime) else timestamp)
    source_files: list[str] = []
    logs.append(f"    Fuente: {SATELLITE_SOURCE_NOTES.get(map_id, map_id)}")

    if map_id in {"abi_b02", "abi_b08", "abi_b09", "abi_b10", "abi_b13"}:
        band = {"abi_b02": 2, "abi_b08": 8, "abi_b09": 9, "abi_b10": 10, "abi_b13": 13}[map_id]
        path = _get_goes_file(timestamp, satellite, scope, band, source_cache)
        source_files.append(path)
        result = _plot_abi_channel(
            path,
            band,
            extent,
            output_dir / f"{map_id}_{time_tag}.png",
            label,
            high_resolution=high_resolution,
        )
        return result, source_files

    if map_id == "geocolor":
        files: dict[int, str] = {}
        # B02 y B13 son obligatorias. Las otras bandas mejoran el color
        # verdadero, pero el gráfico conserva una salida útil si un proveedor
        # no tiene una de ellas para la fecha histórica seleccionada.
        for band in (1, 2, 3, 7, 13):
            try:
                files[band] = _get_goes_file(timestamp, satellite, scope, band, source_cache)
            except Exception as exc:
                if band in {2, 13}:
                    raise
                logs.append(f"    Aviso GeoColor B{band:02d}: {type(exc).__name__}: se usará sustituto.")
        source_files.extend(files.values())
        result = _plot_geocolor(
            files,
            extent,
            output_dir / f"geocolor_{time_tag}.png",
            label,
            high_resolution=high_resolution,
        )
        return result, source_files

    if map_id == "rgb_day":
        files = {band: _get_goes_file(timestamp, satellite, scope, band, source_cache) for band in (2, 5, 13)}
        source_files.extend(files.values())
        result = _plot_rgb(files, "day", extent, output_dir / f"rgb_day_{time_tag}.png", label)
        return result, source_files

    if map_id == "rgb_day_glm":
        files = {band: _get_goes_file(timestamp, satellite, scope, band, source_cache) for band in (2, 5, 13)}
        center = timestamp if isinstance(timestamp, datetime) else utc_now_naive()
        glm_files: list[str] = []
        for index in range(16):
            current = center - timedelta(seconds=20 * index)
            try:
                path = download_goes(current, satellite, "GLM-L2-LCFA", None, destination=GOES_DIR, tolerance_minutes=0.8)
                if path not in glm_files:
                    glm_files.append(path)
            except Exception:
                continue
        if not glm_files:
            raise FileNotFoundError("No se encontraron archivos GLM para construir el producto combinado.")
        source_files.extend([*files.values(), *glm_files])
        result = _plot_rgb_with_glm(files, glm_files, extent, output_dir / f"rgb_day_glm_{time_tag}.png", label)
        return result, source_files

    if map_id == "rgb_night":
        files = {band: _get_goes_file(timestamp, satellite, scope, band, source_cache) for band in (7, 13, 15)}
        source_files.extend(files.values())
        result = _plot_rgb(files, "night", extent, output_dir / f"rgb_night_{time_tag}.png", label)
        return result, source_files

    if map_id in {"glm_instant", "glm_accum"}:
        if map_id == "glm_instant":
            glm_files = [download_goes(timestamp, satellite, "GLM-L2-LCFA", None, destination=GOES_DIR)]
        else:
            center = timestamp if isinstance(timestamp, datetime) else utc_now_naive()
            glm_files = []
            for index in range(16):
                current = center - timedelta(seconds=20 * index)
                try:
                    path = download_goes(current, satellite, "GLM-L2-LCFA", None, destination=GOES_DIR, tolerance_minutes=0.8)
                    if path not in glm_files:
                        glm_files.append(path)
                except Exception:
                    continue
            glm_files.sort()
        source_files.extend(glm_files)
        result = _plot_glm(glm_files, extent, output_dir / f"{map_id}_{time_tag}.png", label)
        return result, source_files

    if map_id == "mimic_tpw":
        path = download_mimic(timestamp)
        source_files.append(path)
        result = _plot_mimic(path, extent, output_dir / f"mimic_tpw_{time_tag}.png")
        return result, source_files

    if map_id == "alpw":
        path = download_alpw(timestamp)
        source_files.append(path)
        result = _plot_alpw(path, extent, output_dir / f"alpw_{time_tag}.png")
        return result, source_files

    if map_id == "fires":
        product, sector = _derived_product(scope, "fires")
        path = _get_goes_file(timestamp, satellite, scope, None, source_cache, product_override=product, sector_override=sector)
        source_files.append(path)
        result = _plot_fires(path, extent, output_dir / f"fires_{time_tag}.png")
        return result, source_files

    if map_id == "sandwich":
        files = {band: _get_goes_file(timestamp, satellite, scope, band, source_cache) for band in (2, 13)}
        source_files.extend(files.values())
        result = _plot_sandwich(files, extent, output_dir / f"sandwich_{time_tag}.png", high_resolution=high_resolution)
        return result, source_files

    if map_id == "dmw":
        b13 = _get_goes_file(timestamp, satellite, scope, 13, source_cache)
        product, sector = _derived_product(scope, "dmw")
        # DMW publica varias componentes C02/C07/C08/C09/C10/C14; C14 es la
        # componente usada junto con la B13 en la receta original.
        dmw = _get_goes_file(timestamp, satellite, scope, 14, source_cache, product_override=product, sector_override=sector)
        source_files.extend([b13, dmw])
        result = _plot_dmw({"b13": b13, "dmw": dmw}, extent, output_dir / f"dmw_{time_tag}.png")
        return result, source_files

    raise ValueError(f"Producto satelital no implementado: {map_id}")

def _enhance_visible(data: np.ndarray) -> np.ndarray:
    """Contraste suave para reflectancias ABI en una composición de color."""

    values = np.asarray(data, dtype=np.float32)
    values = np.nan_to_num(values, nan=0.0, posinf=1.0, neginf=0.0)
    values = np.clip((values - 0.004) / 0.55, 0.0, 1.0)
    return np.power(values, 0.55)


def _plot_geocolor(
    files: dict[int, str],
    extent: list[float],
    output: Path,
    title: str,
    *,
    high_resolution: bool = False,
) -> str:
    """GeoColor aproximado de NOAA: color verdadero de día e IR de noche.

    De día se emplean B01/B02/B03, con verde simulado; de noche se emplean
    B07 y B13, que es la separación publicada para GeoColor. B02 determina la
    transición día/noche por píxel y se usa como malla de alta resolución.
    """

    max_pixels = 4200 if high_resolution else 1300
    info02 = _satellite_info(files[2], "CMI", extent, max_pixels=max_pixels)
    target_shape = info02["data"].shape

    def channel(band: int, default: np.ndarray | None = None) -> np.ndarray:
        if band not in files:
            if default is None:
                raise KeyError(f"GeoColor requiere la banda {band}.")
            return default.copy()
        info = _satellite_info(files[band], "CMI", extent, max_pixels=max_pixels)
        return _resize_channel(info["data"], target_shape)

    b02 = _resize_channel(info02["data"], target_shape)
    b01 = channel(1, b02)
    b03 = channel(3, b02)
    b07 = channel(7, np.full(target_shape, 245.0, dtype=np.float32))
    b13 = channel(13)

    # Componente diurna: B02 roja, verde simulado y B01 azul.
    red_day = _enhance_visible(b02)
    green_day = _enhance_visible(0.45 * b01 + 0.45 * b02 + 0.10 * b03)
    blue_day = _enhance_visible(1.12 * b01)
    day_rgb = np.stack([red_day, green_day, blue_day], axis=2)

    # Componente nocturna: B13 controla la nubosidad fría y B07-B13 añade el
    # matiz azul de nubes bajas/agua del producto GeoColor.
    b13_c = _abi_plot_values(b13, 13)
    b07_c = _abi_plot_values(b07, 7)
    ir = _normalize_channel(b13_c, -90.0, 30.0, descending=True)
    water = _normalize_channel(b13_c - b07_c, -12.0, 5.0)
    night_base = np.clip(0.10 + 0.86 * ir, 0.0, 1.0)
    night_rgb = np.stack(
        [0.58 * night_base, 0.69 * night_base + 0.10 * water, 0.90 * night_base + 0.34 * water],
        axis=2,
    )
    night_rgb = np.clip(night_rgb, 0.0, 1.0)

    # En el océano la reflectancia es baja incluso de día; un umbral muy alto
    # produce una imagen completamente azul. Este umbral conserva superficies
    # oscuras diurnas y sigue dejando B02≈0 como noche.
    day_weight = np.clip((b02 - 0.003) / 0.025, 0.0, 1.0)
    rgb = np.clip(day_weight[..., None] * day_rgb + (1.0 - day_weight[..., None]) * night_rgb, 0.0, 1.0)
    rgb = np.nan_to_num(rgb, nan=0.0)

    fig, ax = _base_map(extent, facecolor="black")
    ax.imshow(
        rgb,
        origin=info02["origin"],
        extent=info02["extent"],
        transform=info02["geo"],
        interpolation="bilinear",
    )
    ax.text(
        0.01,
        0.02,
        "GeoColor: B01/B02/B03 día · B07/B13 noche",
        transform=ax.transAxes,
        fontsize=8,
        color="white",
        bbox={"facecolor": "#0b4ea2", "alpha": 0.78, "pad": 3},
    )
    ax.set_title(f"{title}\n{_satellite_time_text(files[2])}", loc="left", fontsize=11, fontweight="bold", color="white")
    ax.set_title("Aplicación Meteorología Sinóptica", loc="right", fontsize=9, color="white")
    return _save_figure(fig, output, dpi=220 if high_resolution else 150)


In [ ]:
# ============================================================================
# Lectura de GFS y ECMWF
# ============================================================================

def _coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in ds.coords:
            return name
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"No se encontró una coordenada entre {candidates}.")


def normalize_nwp(ds: xr.Dataset) -> xr.Dataset:
    longitude = _coord_name(ds, ("longitude", "lon"))
    latitude = _coord_name(ds, ("latitude", "lat"))
    if float(ds[longitude].max()) > 180:
        ds = ds.assign_coords({longitude: ((ds[longitude] + 180) % 360) - 180}).sortby(longitude)
    return ds


def subset_nwp(ds: xr.Dataset, extent: list[float]) -> xr.Dataset:
    ds = normalize_nwp(ds)
    longitude = _coord_name(ds, ("longitude", "lon"))
    latitude = _coord_name(ds, ("latitude", "lat"))
    min_lon, max_lon, min_lat, max_lat = extent
    lon_values = np.asarray(ds[longitude].values)
    lat_values = np.asarray(ds[latitude].values)
    lon_slice = slice(min_lon, max_lon) if lon_values[0] < lon_values[-1] else slice(max_lon, min_lon)
    lat_slice = slice(min_lat, max_lat) if lat_values[0] < lat_values[-1] else slice(max_lat, min_lat)
    return ds.sel({longitude: lon_slice, latitude: lat_slice})


def field_data(ds: xr.Dataset, names: tuple[str, ...], level: float | None = None) -> tuple[np.ndarray, xr.DataArray]:
    found = None
    for name in names:
        if name in ds.data_vars:
            found = ds[name]
            break
    if found is None:
        raise KeyError(f"No se encontró ninguna variable {names}. Disponibles: {list(ds.data_vars)}")
    da = found
    if level is not None:
        for dim in ("isobaricInhPa", "isobaricInPa", "pressure", "level"):
            if dim in da.dims:
                da = da.sel({dim: level}, method="nearest")
                break
    for dim in list(da.dims):
        if dim not in {"latitude", "longitude", "lat", "lon"} and da.sizes.get(dim, 1) == 1:
            da = da.isel({dim: 0})
    return np.asarray(da.squeeze().values, dtype=float), da


def grid_coordinates(ds: xr.Dataset) -> tuple[np.ndarray, np.ndarray]:
    longitude = _coord_name(ds, ("longitude", "lon"))
    latitude = _coord_name(ds, ("latitude", "lat"))
    return np.asarray(ds[longitude].values, dtype=float), np.asarray(ds[latitude].values, dtype=float)


def grid_metrics(lon: np.ndarray, lat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Calcula dx/dy aproximados en metros para una malla regular lat/lon."""

    earth_radius = 6_371_000.0
    lon2d, lat2d = np.meshgrid(lon, lat)
    dlon = np.gradient(np.deg2rad(lon2d), axis=1)
    dlat = np.gradient(np.deg2rad(lat2d), axis=0)
    dx = earth_radius * np.cos(np.deg2rad(lat2d)) * dlon
    dy = earth_radius * dlat
    dx = np.where(np.abs(dx) < 1.0, np.nan, dx)
    dy = np.where(np.abs(dy) < 1.0, np.nan, dy)
    return dx, dy


def relative_vorticity(u: np.ndarray, v: np.ndarray, lon: np.ndarray, lat: np.ndarray) -> np.ndarray:
    dx, dy = grid_metrics(lon, lat)
    dvdx = np.gradient(v, axis=1) / dx
    dudy = np.gradient(u, axis=0) / dy
    return (dvdx - dudy) * 1e5


def horizontal_divergence(u: np.ndarray, v: np.ndarray, lon: np.ndarray, lat: np.ndarray) -> np.ndarray:
    dx, dy = grid_metrics(lon, lat)
    dudx = np.gradient(u, axis=1) / dx
    dvdy = np.gradient(v, axis=0) / dy
    return (dudx + dvdy) * 1e5


def _titles(ax, left: str, run_dt: datetime, fxx: int, right_prefix: str = "Pronóstico"):
    valid = run_dt + timedelta(hours=fxx)
    ax.set_title(f"{left}\nCorrida: {run_dt:%Y-%m-%d %H:%M} UTC", loc="left", fontsize=10.5, fontweight="bold")
    ax.set_title(f"{right_prefix} +{fxx} h — Válido: {valid:%Y-%m-%d %H:%M} UTC", loc="right", fontsize=9)


def _scalar_map(
    lon: np.ndarray,
    lat: np.ndarray,
    data: np.ndarray,
    extent: list[float],
    output: Path,
    title: str,
    run_dt: datetime,
    fxx: int,
    *,
    levels: np.ndarray,
    cmap: str | Any,
    colorbar_label: str,
    contours: bool = True,
    contour_color: str = "lightgray",
    contour_width: float = 0.45,
) -> str:
    values = np.asarray(data, dtype=float)
    if values.ndim != 2:
        values = np.squeeze(values)
    if values.shape != (len(lat), len(lon)):
        raise ValueError(f"Dimensiones incompatibles para {title}: dato={values.shape}, grilla={(len(lat), len(lon))}.")
    if not np.isfinite(values).any():
        raise ValueError(f"No hay valores válidos para graficar {title}.")
    fig, ax = _base_map(extent)
    lon2d, lat2d = np.meshgrid(lon, lat)
    cf = ax.contourf(lon2d, lat2d, np.ma.masked_invalid(values), levels=levels, cmap=cmap, extend="both", transform=ccrs.PlateCarree())
    if contours:
        try:
            cs = ax.contour(lon2d, lat2d, np.ma.masked_invalid(values), levels=levels[::2], colors=contour_color, linewidths=contour_width, transform=ccrs.PlateCarree())
            ax.clabel(cs, fontsize=6, inline=True, fmt="%g")
        except Exception:
            pass
    fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72, label=colorbar_label)
    _titles(ax, title, run_dt, fxx)
    return _save_figure(fig, output)


def _wind_arrays(ds: xr.Dataset, level: float) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    u, _ = field_data(ds, ("u", "ugrd", "UGRD"), level)
    v, _ = field_data(ds, ("v", "vgrd", "VGRD"), level)
    lon, lat = grid_coordinates(ds)
    return u, v, lon, lat


def _plot_wind(
    ds: xr.Dataset,
    level: float,
    extent: list[float],
    output: Path,
    title: str,
    run_dt: datetime,
    fxx: int,
    style: str,
) -> str:
    u, v, lon, lat = _wind_arrays(ds, level)
    speed = np.hypot(u, v) * 1.94384
    fig, ax = _base_map(extent)
    lon2d, lat2d = np.meshgrid(lon, lat)
    cf = ax.contourf(lon2d, lat2d, speed, levels=np.arange(0, 85, 5), cmap="YlOrBr", extend="both", transform=ccrs.PlateCarree())
    cs = ax.contour(lon2d, lat2d, speed, levels=np.arange(10, 85, 10), colors="lightgray", linewidths=0.45, transform=ccrs.PlateCarree())
    ax.clabel(cs, fontsize=6, fmt="%d")
    step = max(1, int(max(len(lon), len(lat)) / 24))
    u_kt, v_kt = u * 1.94384, v * 1.94384
    if style == "stream":
        plot_lat = lat
        plot_u = u_kt
        plot_v = v_kt
        if plot_lat[0] > plot_lat[-1]:
            plot_lat = plot_lat[::-1]
            plot_u = plot_u[::-1, :]
            plot_v = plot_v[::-1, :]
        try:
            ax.streamplot(lon[::step], plot_lat[::step], plot_u[::step, ::step], plot_v[::step, ::step], density=1.5, color="dimgray", linewidth=0.65, arrowsize=0.9, transform=ccrs.PlateCarree())
        except Exception:
            pass
    elif style == "vector":
        ax.quiver(lon2d[::step, ::step], lat2d[::step, ::step], u_kt[::step, ::step], v_kt[::step, ::step], scale=800, color="black", width=0.002, transform=ccrs.PlateCarree())
    else:
        threshold = speed >= 10
        ax.barbs(lon2d[::step, ::step], lat2d[::step, ::step], np.where(threshold, u_kt, np.nan)[::step, ::step], np.where(threshold, v_kt, np.nan)[::step, ::step], length=5.2, linewidth=0.55, transform=ccrs.PlateCarree())
    fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72, label="Velocidad del viento (kt)")
    _titles(ax, title, run_dt, fxx)
    return _save_figure(fig, output)


class GFSContext:
    def __init__(self, run_dt: datetime, fxx: int, model_id: str, extent: list[float]):
        from herbie import Herbie

        self.run_dt = run_dt
        self.fxx = fxx
        self.model_id = model_id
        self.extent = extent
        self.used_queries: set[str] = set()
        self.cache: dict[str, xr.Dataset] = {}
        self.herbie = Herbie(
            run_dt.strftime("%Y-%m-%d %H:%M"),
            model="gfs",
            product=GFS_PRODUCTS[model_id],
            fxx=fxx,
            save_dir=str(GFS_DIR),
            verbose=False,
        )

    def query(self, search: str) -> xr.Dataset:
        if search not in self.cache:
            self.used_queries.add(search)
            ds = self.herbie.xarray(search)
            self.cache[search] = subset_nwp(ds, self.extent)
        return self.cache[search]

    def source_files(self) -> list[str]:
        files: list[str] = []
        for search in sorted(self.used_queries):
            try:
                result = self.herbie.download(search=search)
                if result is not None:
                    path = Path(result)
                    if path.exists():
                        files.append(str(path))
            except Exception:
                # Los mapas ya pudieron descargarse mediante H.xarray; la
                # inclusión de fuente es opcional y no debe borrar el mapa.
                continue
        return list(dict.fromkeys(files))



def latest_ecmwf_run(fxx: int) -> datetime:
    """Obtener fecha/ciclo realmente publicado para el plazo solicitado.

    La API Open Data solo conserva las últimas corridas. No usar el día de hoy
    con ciclo 00 por suposición: puede no estar aún publicado.
    """
    from ecmwf.opendata import Client

    client = Client(source="ecmwf", model="ifs", resol="0p25")
    latest = client.latest(stream="oper", type="fc", step=fxx, param="msl")
    if hasattr(latest, "datetime"):
        latest = latest.datetime
    if isinstance(latest, str):
        latest = datetime.fromisoformat(latest.replace("Z", "+00:00"))
    if not isinstance(latest, datetime):
        raise RuntimeError(f"ECMWF devolvió un ciclo inesperado: {latest!r}")
    if latest.tzinfo is not None:
        latest = latest.astimezone(timezone.utc)
    return latest.replace(tzinfo=None, minute=0, second=0, microsecond=0)


def check_ecmwf_manual_run(run_dt: datetime) -> None:
    """Evita solicitar datos borrados del portal público de tiempo real."""
    age = utc_now_naive() - run_dt
    if age > timedelta(days=3):
        raise ValueError(
            f"La corrida ECMWF {run_dt:%Y-%m-%d %H} UTC tiene "
            f"{age.days} días de antigüedad. Open Data solo conserva las "
            "últimas ~12 corridas (2–3 días). Active 'ECMWF automático' "
            "o consulte un archivo histórico por separado (MARS/u otra fuente)."
        )
    if age < timedelta(hours=-1):
        raise ValueError("No se puede consultar una corrida ECMWF futura. Active ECMWF automático.")


class ECMWFContext:
    def __init__(self, run_dt: datetime, fxx: int, extent: list[float]):
        self.run_dt = run_dt
        self.fxx = fxx
        self.extent = extent
        self.cache: dict[str, xr.Dataset] = {}
        self.sources: list[str] = []

    def field(self, parameter: str) -> xr.Dataset:
        if parameter in self.cache:
            return self.cache[parameter]
        from ecmwf.opendata import Client

        target = ECMWF_DIR / f"ecmwf_{self.run_dt:%Y%m%d%H}_f{self.fxx:03d}_{safe_name(parameter)}.grib2"
        if not target.exists() or target.stat().st_size < 1024:
            client = Client(source="ecmwf", model="ifs", resol="0p25")
            try:
                client.retrieve(
                    date=self.run_dt.strftime("%Y-%m-%d"),
                    time=self.run_dt.hour,
                    step=self.fxx,
                    type="fc",
                    stream="oper",
                    levtype="sfc",
                    param=parameter,
                    target=str(target),
                )
            except Exception:
                # Un archivo parcial por error de red no debe conservarse como caché.
                if target.exists():
                    target.unlink()
                raise
        ds = xr.open_dataset(str(target), engine="cfgrib", backend_kwargs={"indexpath": ""})
        ds = subset_nwp(ds, self.extent)
        self.cache[parameter] = ds
        self.sources.append(str(target))
        return ds


def render_nwp_map(
    map_id: str,
    context: GFSContext | ECMWFContext,
    output_dir: Path,
    logs: list[str],
    *,
    include_sources: bool = False,
) -> tuple[str, list[str]]:
    """Genera un producto NWP a partir de un contexto ya parametrizado."""

    run_dt = context.run_dt
    fxx = context.fxx
    extent = context.extent
    model_tag = context.model_id if isinstance(context, GFSContext) else "ecmwf"
    time_tag = f"{run_dt:%Y%m%dT%H}Z_f{fxx:03d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    sources: list[str] = []

    if map_id.startswith("ecmwf_"):
        if not isinstance(context, ECMWFContext):
            raise ValueError("El producto ECMWF requiere el modelo ECMWF IFS.")
        if map_id == "ecmwf_wind10":
            # El IFS expone por separado las componentes 10u/10v (m/s).
            # En cfgrib las variables suelen llamarse u10/v10.
            u_ds = context.field("10u")
            v_ds = context.field("10v")
            u, _ = field_data(u_ds, ("u10", "10u"))
            v, _ = field_data(v_ds, ("v10", "10v"))
            lon, lat = grid_coordinates(u_ds)
            if u.shape != v.shape:
                raise ValueError(f"Mallas ECMWF incompatibles para 10u/10v: {u.shape} y {v.shape}.")
            speed = np.hypot(u, v) * 1.943844  # nudos
            fig, ax = _base_map(extent)
            xx, yy = np.meshgrid(lon, lat)
            cf = ax.contourf(xx, yy, np.ma.masked_invalid(speed),
                             levels=np.arange(0, 85, 5), cmap="YlOrBr",
                             extend="max", transform=ccrs.PlateCarree())
            skip = max(1, int(max(len(lon), len(lat)) / 24))
            ax.quiver(xx[::skip, ::skip], yy[::skip, ::skip],
                      u[::skip, ::skip] * 1.943844,
                      v[::skip, ::skip] * 1.943844,
                      scale=650, width=0.002, color="black",
                      transform=ccrs.PlateCarree())
            fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72,
                         label="Viento a 10 m (kt)")
            _titles(ax, "ECMWF IFS — Viento a 10 m", run_dt, fxx)
            path = _save_figure(fig, output_dir / f"{map_id}_{time_tag}.png")
        else:
            parameter = {"ecmwf_msl": "msl", "ecmwf_pwat": "tcw", "ecmwf_t2m": "2t"}.get(map_id)
            if parameter is None:
                raise ValueError(f"Mapa ECMWF no implementado: {map_id}")
            ds = context.field(parameter)
            lon, lat = grid_coordinates(ds)
            names = ("t2m", "2t") if parameter == "2t" else (parameter,)
            data, _ = field_data(ds, names)
            if map_id == "ecmwf_msl":
                data = data / 100.0
                levels = np.arange(980, 1040, 2)
                cmap, label = "RdBu_r", "Presión a nivel medio del mar (hPa)"
                title = "ECMWF IFS — Presión a nivel medio del mar"
            elif map_id == "ecmwf_pwat":
                levels = np.arange(0, 81, 4)
                cmap, label = "gist_ncar", "Agua precipitable (kg m⁻² ≈ mm)"
                title = "ECMWF IFS — Agua precipitable"
            else:
                data = data - 273.15
                levels = np.arange(-10, 46, 2)
                cmap, label = "coolwarm", "Temperatura a 2 m (°C)"
                title = "ECMWF IFS — Temperatura a 2 m"
            path = _scalar_map(lon, lat, data, extent, output_dir / f"{map_id}_{time_tag}.png", title, run_dt, fxx, levels=levels, cmap=cmap, colorbar_label=label)
        if include_sources:
            sources.extend(context.sources)
        return path, list(dict.fromkeys(sources))

    if not isinstance(context, GFSContext):
        raise ValueError("El producto GFS requiere un contexto GFS.")

    if map_id == "gfs_prmsl":
        ds = context.query(":PRMSL:")
        data, _ = field_data(ds, ("prmsl", "mslet", "msl"))
        data = data / 100.0
        lon, lat = grid_coordinates(ds)
        path = _scalar_map(lon, lat, data, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Presión a nivel medio del mar", run_dt, fxx, levels=np.arange(980, 1040, 2), cmap="RdBu_r", colorbar_label="Presión (hPa)")

    elif map_id == "gfs_pwat":
        ds = context.query(":PWAT:")
        data, _ = field_data(ds, ("pwat", "PWAT"))
        lon, lat = grid_coordinates(ds)
        path = _scalar_map(lon, lat, data, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Agua precipitable", run_dt, fxx, levels=np.arange(0, 81, 4), cmap="gist_ncar", colorbar_label="Agua precipitable (mm)")

    elif map_id == "gfs_rh925":
        ds = context.query(":RH:925 mb")
        data, _ = field_data(ds, ("r", "rh"), 925)
        lon, lat = grid_coordinates(ds)
        path = _scalar_map(lon, lat, data, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Humedad relativa a 925 hPa", run_dt, fxx, levels=np.arange(0, 101, 10), cmap="Blues", colorbar_label="Humedad relativa (%)")

    elif map_id == "gfs_thickness":
        ds = context.query(":HGT:(1000|925) mb")
        z1000, _ = field_data(ds, ("gh", "hgt", "z"), 1000)
        z925, _ = field_data(ds, ("gh", "hgt", "z"), 925)
        lon, lat = grid_coordinates(ds)
        path = _scalar_map(lon, lat, z925 - z1000, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Espesor 1000–925 hPa", run_dt, fxx, levels=np.arange(500, 750, 5), cmap="RdYlBu_r", colorbar_label="Espesor geopotencial (gpm)")

    elif map_id == "gfs_delta_t":
        ds_now = context.query(":TMP:1000 mb")
        previous_context = GFSContext(context.run_dt - timedelta(hours=6), 0, context.model_id, context.extent)
        ds_previous = previous_context.query(":TMP:1000 mb")
        data_now, _ = field_data(ds_now, ("t", "tmp"), 1000)
        data_previous, _ = field_data(ds_previous, ("t", "tmp"), 1000)
        lon, lat = grid_coordinates(ds_now)
        path = _scalar_map(lon, lat, data_now - data_previous, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Cambio de temperatura a 1000 hPa (6 h)", run_dt, fxx, levels=np.arange(-8, 9, 1), cmap="RdBu_r", colorbar_label="Cambio de temperatura (K)")

    elif map_id == "gfs_wind500":
        ds = context.query(":(U|V)GRD:500 mb")
        path = _plot_wind(ds, 500, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Viento a 500 hPa + líneas de corriente", run_dt, fxx, "stream")

    elif map_id == "gfs_wind925":
        ds = context.query(":(U|V)GRD:925 mb")
        path = _plot_wind(ds, 925, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Viento a 925 hPa + vectores", run_dt, fxx, "vector")

    elif map_id == "gfs_wind850":
        ds = context.query(":(U|V)GRD:850 mb")
        path = _plot_wind(ds, 850, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Viento a 850 hPa + barbas", run_dt, fxx, "barbs")

    elif map_id == "gfs_wind_mean":
        ds = context.query(":(U|V)GRD:(850|800|750|700|650|600|550|500|450|400|350|300) mb")
        u_da = next(ds[name] for name in ("u", "ugrd", "UGRD") if name in ds.data_vars)
        v_da = next(ds[name] for name in ("v", "vgrd", "VGRD") if name in ds.data_vars)
        level_dim = next((name for name in ("isobaricInhPa", "isobaricInPa", "pressure", "level") if name in u_da.dims), None)
        if level_dim is None:
            raise ValueError("No se encontró la dimensión vertical para el viento medio.")
        available = np.asarray(u_da[level_dim].values, dtype=float)
        wanted = np.array([850, 800, 750, 700, 650, 600, 550, 500, 450, 400, 350, 300], dtype=float)
        levels_available = [float(level) for level in wanted if np.any(np.isclose(available, level))]
        if not levels_available:
            raise ValueError("No hay niveles entre 850 y 300 hPa.")
        u = np.nanmean(np.stack([np.asarray(u_da.sel({level_dim: level}, method="nearest").squeeze().values, dtype=float) for level in levels_available]), axis=0)
        v = np.nanmean(np.stack([np.asarray(v_da.sel({level_dim: level}, method="nearest").squeeze().values, dtype=float) for level in levels_available]), axis=0)
        lon, lat = grid_coordinates(ds)
        speed = np.hypot(u, v) * 1.94384
        fig, ax = _base_map(extent)
        lon2d, lat2d = np.meshgrid(lon, lat)
        cf = ax.contourf(lon2d, lat2d, speed, levels=np.arange(0, 65, 5), cmap="YlOrBr", extend="both", transform=ccrs.PlateCarree())
        step = max(1, int(max(len(lon), len(lat)) / 24))
        ax.barbs(lon2d[::step, ::step], lat2d[::step, ::step], (u * 1.94384)[::step, ::step], (v * 1.94384)[::step, ::step], length=5, linewidth=0.5, transform=ccrs.PlateCarree())
        fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72, label="Velocidad media (kt)")
        _titles(ax, "GFS — Viento medio troposférico 850–300 hPa", run_dt, fxx)
        path = _save_figure(fig, output_dir / f"{map_id}_{model_tag}_{time_tag}.png")

    elif map_id == "gfs_shear":
        ds = context.query(":(U|V)GRD:(850|250) mb")
        u850, v850, lon, lat = _wind_arrays(ds, 850)
        u250, _ = field_data(ds, ("u", "ugrd", "UGRD"), 250)
        v250, _ = field_data(ds, ("v", "vgrd", "VGRD"), 250)
        shear = np.hypot(u250 - u850, v250 - v850) * 1.94384
        fig, ax = _base_map(extent)
        lon2d, lat2d = np.meshgrid(lon, lat)
        cf = ax.contourf(lon2d, lat2d, shear, levels=np.arange(0, 81, 5), cmap="RdYlGn_r", extend="both", transform=ccrs.PlateCarree())
        step = max(1, int(max(len(lon), len(lat)) / 24))
        ax.quiver(lon2d[::step, ::step], lat2d[::step, ::step], (u250 - u850)[::step, ::step], (v250 - v850)[::step, ::step], scale=700, color="black", transform=ccrs.PlateCarree())
        fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72, label="Cizalla vertical 850–250 hPa (kt)")
        _titles(ax, "GFS — Cizalla vertical 850–250 hPa", run_dt, fxx)
        path = _save_figure(fig, output_dir / f"{map_id}_{model_tag}_{time_tag}.png")

    elif map_id == "gfs_td1000":
        ds = context.query(":(TMP|RH):1000 mb")
        temperature, _ = field_data(ds, ("t", "tmp"), 1000)
        rh, _ = field_data(ds, ("r", "rh"), 1000)
        temp_c = temperature - 273.15
        rh = np.clip(rh, 1, 100)
        gamma = np.log(rh / 100.0) + 17.625 * temp_c / (243.04 + temp_c)
        dewpoint = 243.04 * gamma / (17.625 - gamma)
        lon, lat = grid_coordinates(ds)
        path = _scalar_map(lon, lat, dewpoint, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Temperatura de rocío a 1000 hPa", run_dt, fxx, levels=np.arange(-5, 31, 2), cmap="YlGnBu_r", colorbar_label="Temperatura de rocío (°C)")

    elif map_id in {"gfs_vort925", "gfs_div925"}:
        ds = context.query(":(U|V)GRD:925 mb")
        u, v, lon, lat = _wind_arrays(ds, 925)
        data = relative_vorticity(u, v, lon, lat) if map_id == "gfs_vort925" else horizontal_divergence(u, v, lon, lat)
        title = "GFS — Vorticidad relativa a 925 hPa" if map_id == "gfs_vort925" else "GFS — Divergencia/convergencia a 925 hPa"
        label = "Vorticidad relativa (×10⁻⁵ s⁻¹)" if map_id == "gfs_vort925" else "Divergencia (+) / convergencia (−) (×10⁻⁵ s⁻¹)"
        fig, ax = _base_map(extent)
        lon2d, lat2d = np.meshgrid(lon, lat)
        cf = ax.contourf(lon2d, lat2d, data, levels=np.linspace(-20, 20, 21), cmap="RdBu_r" if map_id == "gfs_vort925" else "BrBG_r", extend="both", transform=ccrs.PlateCarree())
        if map_id == "gfs_div925":
            step = max(1, int(max(len(lon), len(lat)) / 24))
            ax.barbs(lon2d[::step, ::step], lat2d[::step, ::step], (u * 1.94384)[::step, ::step], (v * 1.94384)[::step, ::step], length=5, linewidth=0.5, transform=ccrs.PlateCarree())
        fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72, label=label)
        _titles(ax, title, run_dt, fxx)
        path = _save_figure(fig, output_dir / f"{map_id}_{model_tag}_{time_tag}.png")

    elif map_id == "gfs_divmean":
        ds = context.query(":(U|V)GRD:(1000|925) mb")
        u1000, v1000, lon, lat = _wind_arrays(ds, 1000)
        u925, v925, _, _ = _wind_arrays(ds, 925)
        data = (horizontal_divergence(u1000, v1000, lon, lat) + horizontal_divergence(u925, v925, lon, lat)) / 2.0
        u_mean, v_mean = (u1000 + u925) / 2.0, (v1000 + v925) / 2.0
        fig, ax = _base_map(extent)
        lon2d, lat2d = np.meshgrid(lon, lat)
        cf = ax.contourf(lon2d, lat2d, data, levels=np.linspace(-10, 10, 21), cmap="BrBG_r", extend="both", transform=ccrs.PlateCarree())
        step = max(1, int(max(len(lon), len(lat)) / 24))
        ax.barbs(lon2d[::step, ::step], lat2d[::step, ::step], (u_mean * 1.94384)[::step, ::step], (v_mean * 1.94384)[::step, ::step], length=5, linewidth=0.5, transform=ccrs.PlateCarree())
        fig.colorbar(cf, ax=ax, pad=0.03, shrink=0.72, label="Divergencia (+) / convergencia (−) (×10⁻⁵ s⁻¹)")
        _titles(ax, "GFS — Divergencia/convergencia media 1000–925 hPa", run_dt, fxx)
        path = _save_figure(fig, output_dir / f"{map_id}_{model_tag}_{time_tag}.png")

    elif map_id == "gfs_mfc":
        ds_wind = context.query(":(U|V)GRD:(1000|850) mb")
        ds_q = context.query("SPFH:(1000|850) mb")
        u1000, v1000, lon, lat = _wind_arrays(ds_wind, 1000)
        u850, v850, _, _ = _wind_arrays(ds_wind, 850)
        q1000, _ = field_data(ds_q, ("q", "spfh"), 1000)
        q850, _ = field_data(ds_q, ("q", "spfh"), 850)
        # signo cambiado: positivo representa convergencia de humedad, como
        # en la celda 4.5 del notebook.
        data = -(horizontal_divergence(q1000 * u1000, q1000 * v1000, lon, lat) + horizontal_divergence(q850 * u850, q850 * v850, lon, lat)) / 2.0
        path = _scalar_map(lon, lat, gaussian_filter(data, sigma=0.6), extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Convergencia de flujo de humedad 1000–850 hPa", run_dt, fxx, levels=np.linspace(-30, 30, 21), cmap="BrBG", colorbar_label="Convergencia de humedad (×10⁻⁵)")

    elif map_id == "gfs_gdi":
        ds_t = context.query("TMP:(950|850|700|500) mb")
        ds_q = context.query("SPFH:(950|850|700|500) mb")
        ds_p = context.query(":PRES:surface")
        t950, _ = field_data(ds_t, ("t", "tmp"), 950)
        t850, _ = field_data(ds_t, ("t", "tmp"), 850)
        t700, _ = field_data(ds_t, ("t", "tmp"), 700)
        t500, _ = field_data(ds_t, ("t", "tmp"), 500)
        q950, _ = field_data(ds_q, ("q", "spfh"), 950)
        q850, _ = field_data(ds_q, ("q", "spfh"), 850)
        q700, _ = field_data(ds_q, ("q", "spfh"), 700)
        q500, _ = field_data(ds_q, ("q", "spfh"), 500)
        sfc, _ = field_data(ds_p, ("sp", "pres", "pres_surface"))
        sfc = sfc / 100.0 if np.nanmedian(sfc) > 2000 else sfc
        theta_a = t950 * (1000 / 950) ** (2 / 7)
        theta_b = 0.5 * (t850 * (1000 / 850) ** (2 / 7) + t700 * (1000 / 700) ** (2 / 7))
        theta_c = t500 * (1000 / 500) ** (2 / 7)
        r_a = q950
        r_b = 0.5 * (q850 + q700)
        r_c = q500
        p1, p2, p3 = 2690000 * r_a, 2690000 * r_b, 2690000 * r_c
        p4 = 1005.7 * t850
        ept_a = theta_a * np.exp(np.clip(p1 / p4, -50, 50))
        ept_b = theta_b * np.exp(np.clip(p2 / p4, -50, 50)) - 10
        ept_c = theta_c * np.exp(np.clip(p3 / p4, -50, 50)) - 10
        me, le = ept_c - 303, ept_a - 303
        eci = np.where(le <= 0, 0, 0.065 * le * me)
        mwi = np.where(t500 - 263.15 <= 0, 0, (t500 - 263.15) * -7)
        inversion = 1.5 * (t950 - t700) + 1.5 * (ept_b - ept_a)
        ii = np.where(inversion <= 0, inversion, 0)
        correction = 18 - 9000 / np.where(np.abs(sfc - 500) < 1, np.nan, sfc - 500)
        gdi = gaussian_filter(eci + mwi + ii + correction, sigma=0.6)
        lon, lat = grid_coordinates(ds_t)
        gdi_colors = ["#323232", "#646464", "#737373", "#7e7e7e", "#909090", "#a3a3a3", "#b1b1b1", "#bcbcbc", "#bbc7cb", "#b2d2dd", "#90d5bb", "#55d065", "#5acf28", "#bad411", "#ffcc00", "#ffa900", "#fc8106", "#eb4722", "#d8133a", "#ac0a1d"]
        path = _scalar_map(lon, lat, gdi, extent, output_dir / f"{map_id}_{model_tag}_{time_tag}.png", "GFS — Índice Galvez-Davison (GDI)", run_dt, fxx, levels=np.arange(-30, 71, 5), cmap=ListedColormap(gdi_colors), colorbar_label="GDI")

    elif map_id == "gfs_fram":
        # Implementación FRAM portada del script de referencia adjunto.
        # La figura usa el dominio, campos, filtros, escalas y capas del original.
        import metpy.calc as mpcalc
        from metpy.units import units

        ds_t = context.query("TMP:(1000|925|850|700) mb")
        ds_z = context.query("HGT:(1000|925|850|700) mb")
        ds_q = context.query("SPFH:(1000|925|850) mb")
        ds_u = context.query("UGRD:(1000|925) mb")
        ds_v = context.query("VGRD:(1000|925) mb")

        t1000, _ = field_data(ds_t, ("t", "tmp"), 1000)
        t925, _ = field_data(ds_t, ("t", "tmp"), 925)
        t850, _ = field_data(ds_t, ("t", "tmp"), 850)
        t700, _ = field_data(ds_t, ("t", "tmp"), 700)
        z1000, _ = field_data(ds_z, ("gh", "hgt", "z"), 1000)
        z925, _ = field_data(ds_z, ("gh", "hgt", "z"), 925)
        z850, _ = field_data(ds_z, ("gh", "hgt", "z"), 850)
        z700, _ = field_data(ds_z, ("gh", "hgt", "z"), 700)
        q1000, _ = field_data(ds_q, ("q", "spfh"), 1000)
        q925, _ = field_data(ds_q, ("q", "spfh"), 925)
        q850, _ = field_data(ds_q, ("q", "spfh"), 850)
        u1000, _ = field_data(ds_u, ("u", "ugrd", "UGRD"), 1000)
        u925, _ = field_data(ds_u, ("u", "ugrd", "UGRD"), 925)
        v1000, _ = field_data(ds_v, ("v", "vgrd", "VGRD"), 1000)
        v925, _ = field_data(ds_v, ("v", "vgrd", "VGRD"), 925)

        lon, lat = grid_coordinates(ds_t)
        lon2d, lat2d = np.meshgrid(lon, lat)
        dx, dy = mpcalc.lat_lon_grid_deltas(lon2d, lat2d)
        esp_1000_925 = z925 - z1000
        esp_1000_850 = z850 - z1000
        esp_1000_700 = z700 - z1000
        sfc = z1000

        q1000_q = q1000 * units.dimensionless
        q925_q = q925 * units.dimensionless
        q850_q = q850 * units.dimensionless
        dewp1000 = mpcalc.dewpoint_from_specific_humidity(
            1000 * units.hPa, t1000 * units.kelvin, q1000_q
        ).to(units.kelvin)
        dewp925 = mpcalc.dewpoint_from_specific_humidity(
            925 * units.hPa, t925 * units.kelvin, q925_q
        ).to(units.kelvin)
        dewp850 = mpcalc.dewpoint_from_specific_humidity(
            850 * units.hPa, t850 * units.kelvin, q850_q
        ).to(units.kelvin)
        dewp1000_k = dewp1000.magnitude
        dewp925_k = dewp925.magnitude
        dewp850_k = dewp850.magnitude
        thetae1000 = mpcalc.equivalent_potential_temperature(
            1000 * units.hPa, t1000 * units.kelvin, dewp1000
        ).to(units.kelvin).magnitude

        def fram_smooth(data, sigma=1.0):
            return gaussian_filter(np.asarray(data, dtype=float), sigma=sigma)

        def fram_mag_grad(data, field_unit):
            grad_y, grad_x = mpcalc.gradient(
                np.asarray(data, dtype=float) * field_unit, deltas=(dy, dx)
            )
            return np.hypot(np.asarray(grad_y.magnitude), np.asarray(grad_x.magnitude))

        # Máscaras topográficas y térmicas FRAM.
        FR_LMSK = (sfc < 1000.0).astype(int)
        FR_HMSK = (sfc >= 1000.0).astype(int)
        gradient_magnitude = fram_mag_grad(sfc, units.meter)
        FR_MSKG_A = (gradient_magnitude * 1000.0) - 10.0
        FR_MSKG_B = ((np.where(FR_MSKG_A >= 0, FR_MSKG_A, 0.0)) * -1.0) + 1.0
        FR_MSKG = np.where(FR_MSKG_B >= 0, FR_MSKG_B, 0.0)
        FR_MSKE_A = sfc - 3000.0
        FR_MSKE_B = np.where(FR_MSKE_A >= 0, FR_MSKE_A, 0.0) - 1.0
        FR_MSKE = np.where(FR_MSKE_B <= 0, FR_MSKE_B, 0.0) * -1.0
        FR_MSTK_A = fram_smooth(esp_1000_925, sigma=1.0) - 673.0
        FR_MSTK_B = ((np.where(FR_MSTK_A <= 0, FR_MSTK_A, 0.0)) * -1.0) - 1.0
        FR_MSTK = np.where(FR_MSTK_B <= 0, FR_MSTK_B, 0.0) + 1.0
        FR_MSKS = FR_MSKG * FR_MSKE * FR_MSTK

        # Índice FRAM de masa de aire.
        FR_LTHK = esp_1000_925 / 23.0
        FR_LTHKF = FR_LTHK ** 2
        FR_HTHK = (z700 - z925) / 80.0
        FR_HTHKF = FR_HTHK ** 2
        FR_THK = FR_LTHKF * FR_LMSK + FR_HTHKF * FR_HMSK
        FR_LDW = (dewp1000_k * dewp925_k) / 100.0
        FR_HDW = (dewp925_k * dewp850_k) / 100.0
        FR_DW_p = (FR_LDW * FR_LMSK) + (FR_HDW * FR_HMSK)
        FR_DW = fram_smooth(FR_DW_p, sigma=1.0)
        FR_AMF = (FR_DW / 50.0) * (FR_THK / 58.0)

        # Gradientes de espesores, theta-e e índice FRAM.
        FR_GR192 = fram_mag_grad(esp_1000_925, units.meter) * 80000.0
        FR_GR185 = fram_mag_grad(esp_1000_850, units.meter) * 50000.0
        FR_GR170 = fram_mag_grad(esp_1000_700, units.meter) * 2000.0
        FR_THEFC = fram_mag_grad(thetae1000, units.kelvin) * 700000.0
        FR_GRAIR = fram_mag_grad(FR_AMF, units.dimensionless) * 70000.0
        FR_GRADNT = (FR_GR192 + FR_GR185 + FR_GR170 + FR_THEFC + FR_GRAIR) * FR_MSKS
        FR_GRADNT = fram_smooth(FR_GRADNT, sigma=0.7)

        # Flujo de humedad y convergencia media 1000–925 hPa.
        mixr1000 = mpcalc.mixing_ratio_from_specific_humidity(q1000_q)
        mixr925 = mpcalc.mixing_ratio_from_specific_humidity(q925_q)
        mixru1000 = mixr1000 * u1000 * units("m/s")
        mixrv1000 = mixr1000 * v1000 * units("m/s")
        mixru925 = mixr925 * u925 * units("m/s")
        mixrv925 = mixr925 * v925 * units("m/s")
        conv1000 = mpcalc.divergence(mixru1000, mixrv1000, dx=dx, dy=dy).to("1/s").magnitude
        conv925 = mpcalc.divergence(mixru925, mixrv925, dx=dx, dy=dy).to("1/s").magnitude
        COMA_1000_925 = ((conv925 + conv1000) / 2.0) * 20000000.0
        FR_MCOCOR = fram_smooth(np.where(COMA_1000_925 < 0, COMA_1000_925, 0.0), sigma=1.0) * FR_MSKS
        mask_conv = FR_MCOCOR < -1.4
        x_pts = lon2d[mask_conv].ravel()
        y_pts = lat2d[mask_conv].ravel()
        mask_orog = np.where(sfc > 2500.0, 1.0, np.nan)

        # Viento medio en nudos, con las dos clases de barbas del script.
        u_1000_925_kt = (((u1000 + u925) / 2.0) * units("m/s")).to("knots").magnitude
        v_1000_925_kt = (((v1000 + v925) / 2.0) * units("m/s")).to("knots").magnitude
        wspd = np.hypot(u_1000_925_kt, v_1000_925_kt)
        m25 = (wspd >= 25.0).astype(int)
        m10 = ((wspd >= 10.0) & (wspd < 25.0)).astype(int)
        u_m25_kt, v_m25_kt = u_1000_925_kt * m25, v_1000_925_kt * m25
        u_m10_kt, v_m10_kt = u_1000_925_kt * m10, v_1000_925_kt * m10
        flip_flag = np.zeros_like(lat2d)
        flip_flag[lat2d < 0.0] = 1

        # Paleta discreta original y niveles FRAM de 174–258.
        fram_rgb = [
            (185,185,200),(175,175,195),(165,165,190),(155,155,185),(145,145,180),
            (140,140,180),(160,150,160),(155,145,155),(150,140,150),(144,134,144),
            (136,126,136),(128,118,128),(120,110,120),(140,110,130),(130,105,135),
            (120,100,139),(110,95,143),(100,90,147),(94,92,152),(93,99,156),
            (92,106,160),(91,113,164),(90,120,168),(70,130,154),(71,135,153),
            (72,145,152),(73,155,151),(74,165,150),(77,162,120),(80,157,94),
            (105,164,92),(130,168,90),(150,172,88),(188,170,82),(186,161,85),
            (184,152,88),(182,143,91),(180,130,104),(178,120,106),(176,115,110),
            (174,110,114),(172,98,95),
        ]
        colors = np.asarray(fram_rgb, dtype=float) / 255.0
        fram_cmap = ListedColormap(colors, name="FRAM")
        fram_cmap.set_under(colors[0])
        fram_cmap.set_over(colors[-1])
        levels_AMF = list(range(174, 260, 2))
        fram_norm = BoundaryNorm(levels_AMF, ncolors=fram_cmap.N, clip=False)
        levels2 = [100] + levels_AMF + [300]

        # Composición gráfica del script de referencia.
        fig = plt.figure(figsize=(20, 15))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        cf = ax.contourf(
            lon2d, lat2d, np.ma.masked_invalid(FR_AMF), levels=levels2,
            cmap=fram_cmap, norm=fram_norm, transform=ccrs.PlateCarree(), extend="both",
        )
        plt.colorbar(
            cf, orientation="vertical", pad=0.01, shrink=0.6, aspect=50,
            ticks=levels_AMF[::2], format="%d",
        )

        cian = (0, 1, 1)
        ax.contour(lon2d, lat2d, esp_1000_925, levels=np.arange(500, 649, 2),
                   colors=[cian], linewidths=0.5, linestyles="dashed", transform=ccrs.PlateCarree())
        ax.contour(lon2d, lat2d, esp_1000_925, levels=[623], colors=[cian],
                   linewidths=1.2, transform=ccrs.PlateCarree())
        verd = (0, 1, 0)
        ax.contour(lon2d, lat2d, esp_1000_925, levels=np.arange(650, 680, 2),
                   colors=[verd], linewidths=0.5, linestyles="dashed", transform=ccrs.PlateCarree())
        ax.contour(lon2d, lat2d, esp_1000_925, levels=[650], colors=[verd],
                   linewidths=1.2, transform=ccrs.PlateCarree())

        face_rgba = ax.get_facecolor()
        for level, inner_color, lw_inner, lw_border in (
            (25, face_rgba, 1.6, 2.4), (40, "red", 2.0, 3.0), (70, "yellow", 2.6, 3.6)
        ):
            ax.contour(lon2d, lat2d, FR_GRADNT, levels=[level], colors="black", linewidths=lw_border,
                       transform=ccrs.PlateCarree())
            ax.contour(lon2d, lat2d, FR_GRADNT, levels=[level], colors=[inner_color], linewidths=lw_inner,
                       transform=ccrs.PlateCarree())

        skip = 3
        bcolor = "0.2"
        barb_args = dict(
            length=5.0, sizes=dict(emptybarb=0.0, spacing=0.2, height=0.5),
            linewidth=0.6, pivot="middle", flip_barb=flip_flag[::skip, ::skip],
            transform=ccrs.PlateCarree(),
        )
        ax.barbs(lon2d[::skip, ::skip], lat2d[::skip, ::skip],
                 u_m10_kt[::skip, ::skip], v_m10_kt[::skip, ::skip],
                 barbcolor=bcolor, **barb_args)
        ax.barbs(lon2d[::skip, ::skip], lat2d[::skip, ::skip],
                 u_m25_kt[::skip, ::skip], v_m25_kt[::skip, ::skip],
                 barbcolor="k", **barb_args)
        try:
            ax.streamplot(lon2d, lat2d, u_1000_925_kt, v_1000_925_kt,
                          density=[1, 1], linewidth=0.8, color=bcolor)
        except Exception:
            lat_order = np.argsort(lat)
            ax.streamplot(lon, lat[lat_order], u_1000_925_kt[lat_order, :], v_1000_925_kt[lat_order, :],
                          density=[1, 1], linewidth=0.8, color=bcolor, transform=ccrs.PlateCarree())

        ax.scatter(x_pts, y_pts, s=10, marker="s", edgecolors="k", linewidths=0.5,
                   color="0.2", transform=ccrs.PlateCarree())
        cmap_gray = ListedColormap(["0.6"])
        cmap_gray.set_bad(alpha=0)
        ax.pcolormesh(lon2d, lat2d, mask_orog, cmap=cmap_gray, shading="auto",
                      transform=ccrs.PlateCarree(), zorder=10)
        ax.coastlines(resolution="10m", color="black", linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, edgecolor="black", linewidth=0.5)
        gl = ax.gridlines(
            crs=ccrs.PlateCarree(), color="gray", linestyle="--", linewidth=0.1,
            xlocs=np.arange(-180, 181, 5), ylocs=np.arange(-90, 91, 5), draw_labels=True,
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xpadding = -5
        gl.ypadding = -5
        gl.ylabel_style = {"color": "black", "size": 6, "weight": "bold"}
        gl.xlabel_style = {"color": "black", "size": 6, "weight": "bold"}

        valid_dt = run_dt + timedelta(hours=fxx)
        ax.set_title(
            f"GFS - FRAM (Herramienta de Caracterización de Frentes y Masas de Aire) - "
            f"{run_dt:%Y-%m-%d %H:%M} UTC",
            fontweight="bold", fontsize=12, loc="left",
        )
        ax.set_title(
            f"Pronóstico +{fxx}h - Válido: {valid_dt:%Y-%m-%d %H:%M} UTC",
            fontsize=10, loc="right",
        )
        fig.tight_layout()
        path = _save_figure(fig, output_dir / f"{map_id}_{model_tag}_{time_tag}.png")

    else:
        raise ValueError(f"Producto NWP no implementado: {map_id}")

    if include_sources:
        sources.extend(context.source_files())
    return path, list(dict.fromkeys(sources))


def _dewpoint_from_specific_humidity(temperature_k: np.ndarray, q: np.ndarray, pressure_hpa: float) -> np.ndarray:
    mixing_ratio = np.maximum(q, 1e-8) / np.maximum(0.622 - q, 1e-8)
    vapor_pressure = pressure_hpa * mixing_ratio / (0.622 + mixing_ratio)
    gamma = np.log(np.maximum(vapor_pressure, 1e-4) / 6.112)
    return 243.5 * gamma / (17.67 - gamma)




In [ ]:
# ============================================================================
# Ejecuciones solicitadas desde la interfaz
# ============================================================================

def run_forecast(
    date_text: str,
    cycle: str,
    fxx_text: str,
    models: list[str] | None,
    maps: list[str] | None,
    region: str,
    min_lon: float,
    max_lon: float,
    min_lat: float,
    max_lat: float,
    include_sources: bool,
    auto_ecmwf: bool = True,
):
    logs = ["=== EJECUCIÓN DE PRONÓSTICO ==="]
    images: list[str] = []
    sources: list[str] = []
    try:
        run_dt = parse_forecast_run(date_text, cycle)
        fxxs = parse_hours(fxx_text)
        selected_models = models or []
        selected_maps = maps or []
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        if not selected_models:
            raise ValueError("Seleccione al menos un modelo numérico.")
        if not selected_maps:
            raise ValueError("Seleccione al menos un mapa.")
        needs_gfs = any(item.startswith("gfs_") for item in selected_maps)
        needs_ecmwf = any(item.startswith("ecmwf_") for item in selected_maps)
        has_gfs = any(item in selected_models for item in ("gfs025", "gfs050"))
        missing_models = []
        if needs_gfs and not has_gfs:
            missing_models.append("GFS 0.50° (requerido por FRAM)" if "gfs_fram" in selected_maps else "GFS 0.25° o GFS 0.50°")
        elif "gfs_fram" in selected_maps and "gfs050" not in selected_models:
            missing_models.append("GFS 0.50° (requerido por FRAM)")
        if needs_ecmwf and "ecmwf" not in selected_models:
            missing_models.append("ECMWF IFS")
        if missing_models:
            raise ValueError(
                "Los mapas seleccionados requieren " + " y ".join(missing_models)
                + ". Marque ese modelo en la selección para continuar."
            )
        ecmwf_run_dt = None
        if "ecmwf" in selected_models:
            if auto_ecmwf:
                # Un ciclo común para todos los productos y plazos del pedido.
                try:
                    ecmwf_run_dt = latest_ecmwf_run(max(fxxs))
                except Exception as exc:
                    raise RuntimeError(
                        f"No se pudo identificar la última corrida ECMWF disponible: "
                        f"{type(exc).__name__}: {exc}"
                    ) from exc
                logs.append(f"ECMWF automático: corrida publicada {ecmwf_run_dt:%Y-%m-%d %H:%M} UTC")
            else:
                check_ecmwf_manual_run(run_dt)
                ecmwf_run_dt = run_dt
                logs.append(f"ECMWF manual: corrida {ecmwf_run_dt:%Y-%m-%d %H:%M} UTC")
        if any(m != "ecmwf" for m in selected_models):
            logs.append(f"GFS manual: corrida {run_dt:%Y-%m-%d %H:%M} UTC")
        logs.append(f"Plazos: {fxxs} h | región: {extent}")
        run_dir = make_run_dir("pronostico")
        context_cache = {}
        # Orden de salida: producto y, dentro de cada producto, plazo creciente.
        for model_id in selected_models:
            model_maps = [
                item for item in selected_maps
                if NWP_LABELS.get(item, "").startswith("ECMWF") == (model_id == "ecmwf")
                and (item != "gfs_fram" or model_id == "gfs050")
            ]
            for map_id in model_maps:
                logs.append(f"\\n[Producto] {NWP_LABELS[map_id]}")
                for fxx in fxxs:
                    logs.append(f"  +{fxx} h")
                    try:
                        cache_key = (model_id, fxx)
                        if cache_key not in context_cache:
                            context_cache[cache_key] = (
                                ECMWFContext(ecmwf_run_dt, fxx, extent)
                                if model_id == "ecmwf"
                                else GFSContext(run_dt, fxx, model_id, extent)
                            )
                        context = context_cache[cache_key]
                        image, raw = render_nwp_map(
                            map_id, context, run_dir, logs,
                            include_sources=include_sources,
                        )
                        images.append(image)
                        sources.extend(raw)
                        logs.append(f"  OK: {Path(image).name}")
                    except Exception as exc:
                        logs.append(f"  ERROR en {map_id} +{fxx} h: {type(exc).__name__}: {exc}")
        if include_sources:
            for context in context_cache.values():
                if isinstance(context, ECMWFContext):
                    sources.extend(context.sources)
        archive = None
        if images:
            archive = write_bundle(
                run_dir,
                images,
                list(dict.fromkeys(sources)),
                logs,
                {"modo": "pronóstico", "corrida_utc_gfs": run_dt,
                 "corrida_utc_ecmwf": ecmwf_run_dt,
                 "ecmwf_automatico": auto_ecmwf,
                 "horas": fxxs, "modelos": selected_models, "mapas": selected_maps,
                 "region": extent, "archivos_fuente": include_sources},
                "pronostico_meteorologia_sinoptica",
            )
        else:
            logs.append("No se generaron mapas: no se creará un ZIP vacío. Revise los errores anteriores.")
        logs.append(f"\nMapas generados: {len(images)}")
        if archive:
            logs.append(f"Paquete: {archive}")
        return result_tuple(images, archive, logs)
    except Exception as exc:
        logs.append(f"ERROR: {type(exc).__name__}: {exc}")
        return result_tuple([], None, logs)


def diagnostic_times(mode: str, start_text: str, end_text: str, hours_back: float, cadence_minutes: int) -> list[datetime]:
    if mode == "Últimas horas (automático)":
        end = utc_now_naive()
        end = end.replace(minute=(end.minute // cadence_minutes) * cadence_minutes, second=0, microsecond=0)
        start = end - timedelta(hours=float(hours_back))
    else:
        start = parse_datetime(start_text)
        end = parse_datetime(end_text)
        if isinstance(start, str) or isinstance(end, str):
            raise ValueError("Indique fechas históricas válidas.")
    return iter_times(start, end, timedelta(minutes=int(cadence_minutes)))


def run_diagnostic(
    time_mode: str,
    start_text: str,
    end_text: str,
    hours_back: float,
    cadence_minutes: str,
    satellite: str,
    scope: str,
    maps: list[str] | None,
    region: str,
    min_lon: float,
    max_lon: float,
    min_lat: float,
    max_lat: float,
    include_sources: bool,
    high_resolution: bool = False,
):
    logs = ["=== EJECUCIÓN DE DIAGNÓSTICO SATELITAL ==="]
    images: list[str] = []
    sources: list[str] = []
    try:
        cadence = int(float(cadence_minutes))
        if cadence not in (10, 20, 30, 60):
            raise ValueError("La cadencia debe ser 10, 20, 30 o 60 minutos.")
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        selected_maps = maps or []
        if not selected_maps:
            raise ValueError("Seleccione al menos un producto satelital.")
        times = diagnostic_times(time_mode, start_text, end_text, hours_back, cadence)
        if len(times) > 240:
            raise ValueError("El período genera más de 240 imágenes. Aumente la cadencia o divídalo en varios períodos.")
        logs.append(f"Satélite: {satellite} | pasos: {len(times)} | cadencia: {cadence} min | región: {extent}")
        logs.append("Alta resolución B02/sandwich/GeoColor: " + ("sí" if high_resolution else "no"))
        run_dir = make_run_dir("diagnostico")
        source_cache: dict[tuple[Any, ...], str] = {}
        # Agrupa todos los horarios de un producto antes de pasar al siguiente.
        for map_id in selected_maps:
            logs.append(f"\\n[Producto] {SATELLITE_LABELS[map_id]}")
            for index, timestamp in enumerate(times, start=1):
                try:
                    logs.append(f"  [{index}/{len(times)}] {timestamp:%Y-%m-%d %H:%M} UTC")
                    image, raw = render_satellite_map(
                        map_id, timestamp, satellite, scope, extent,
                        run_dir, source_cache, logs,
                        high_resolution=high_resolution,
                    )
                    images.append(image)
                    sources.extend(raw)
                    logs.append(f"  OK: {Path(image).name}")
                except Exception as exc:
                    logs.append(f"  ERROR en {map_id} {timestamp:%Y-%m-%d %H:%M}: {type(exc).__name__}: {exc}")
        archive = write_bundle(
            run_dir,
            images,
            list(dict.fromkeys(sources)) if include_sources else [],
            logs,
            {"modo": "diagnóstico", "periodo_utc": [times[0], times[-1]], "cadencia_minutos": cadence, "satellite": satellite, "scope": scope, "mapas": selected_maps, "region": extent, "alta_resolucion": high_resolution, "archivos_fuente": include_sources},
            "diagnostico_satelital",
        )
        logs.append(f"\nMapas generados: {len(images)}")
        if archive:
            logs.append(f"Paquete: {archive}")
        return result_tuple(images, archive, logs)
    except Exception as exc:
        logs.append(f"ERROR: {type(exc).__name__}: {exc}")
        return result_tuple([], None, logs)


def run_reanalysis(
    start_text: str,
    end_text: str,
    cadence_hours: str,
    fxx_text: str,
    models: list[str] | None,
    nwp_maps: list[str] | None,
    satellite: str,
    scope: str,
    satellite_maps: list[str] | None,
    region: str,
    min_lon: float,
    max_lon: float,
    min_lat: float,
    max_lat: float,
    output_mode: str,
    high_resolution: bool = False,
):
    logs = ["=== EJECUCIÓN HISTÓRICA / REANÁLISIS POR PERÍODO ==="]
    images: list[str] = []
    sources: list[str] = []
    try:
        start = parse_datetime(start_text)
        end = parse_datetime(end_text)
        if isinstance(start, str) or isinstance(end, str):
            raise ValueError("Indique fechas de inicio y final históricas.")
        cadence = float(cadence_hours)
        if cadence <= 0:
            raise ValueError("La cadencia debe ser positiva.")
        times = iter_times(start, end, timedelta(hours=cadence))
        if len(times) > 300:
            raise ValueError("El período genera más de 300 pasos. Divídalo en bloques para evitar descargas excesivas.")
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        selected_models = models or []
        selected_nwp = nwp_maps or []
        selected_sat = satellite_maps or []
        fxxs = parse_hours(fxx_text)
        if not selected_models and not selected_sat:
            raise ValueError("Seleccione modelos/mapas de modelo o productos satelitales.")
        needs_gfs = any(item.startswith("gfs_") for item in selected_nwp)
        needs_ecmwf = any(item.startswith("ecmwf_") for item in selected_nwp)
        has_gfs = any(item in selected_models for item in ("gfs025", "gfs050"))
        missing_models = []
        if needs_gfs and not has_gfs:
            missing_models.append("GFS 0.50° (requerido por FRAM)" if "gfs_fram" in selected_nwp else "GFS 0.25° o GFS 0.50°")
        elif "gfs_fram" in selected_nwp and "gfs050" not in selected_models:
            missing_models.append("GFS 0.50° (requerido por FRAM)")
        if needs_ecmwf and "ecmwf" not in selected_models:
            missing_models.append("ECMWF IFS")
        if missing_models:
            raise ValueError(
                "Los mapas de modelo seleccionados requieren " + " y ".join(missing_models)
                + ". Marque ese modelo en la selección para continuar."
            )
        if output_mode not in {"Mapas", "Datos fuente", "Mapas y datos fuente"}:
            output_mode = "Mapas"
        make_maps = output_mode in {"Mapas", "Mapas y datos fuente"}
        include_sources = output_mode in {"Datos fuente", "Mapas y datos fuente"}
        logs.append(f"Período: {start:%Y-%m-%d %H:%M} → {end:%Y-%m-%d %H:%M} UTC | pasos: {len(times)} | cadencia: {cadence:g} h")
        logs.append("Alta resolución B02/sandwich/GeoColor: " + ("sí" if high_resolution else "no"))
        run_dir = make_run_dir("historico")
        source_cache: dict[tuple[Any, ...], str] = {}
        # Agrupa la galería por producto; las fechas y plazos van dentro de cada grupo.
        for map_id in selected_sat:
            logs.append(f"\\n[Producto satelital] {SATELLITE_LABELS[map_id]}")
            for index, timestamp in enumerate(times, start=1):
                try:
                    logs.append(f"  [{index}/{len(times)}] {timestamp:%Y-%m-%d %H:%M} UTC")
                    image, raw = render_satellite_map(
                        map_id, timestamp, satellite, scope, extent,
                        run_dir, source_cache, logs,
                        high_resolution=high_resolution,
                    )
                    if make_maps:
                        images.append(image)
                    elif image and Path(image).exists():
                        Path(image).unlink()
                    sources.extend(raw)
                except Exception as exc:
                    logs.append(f"  ERROR satélite {map_id} {timestamp:%Y-%m-%d %H:%M}: {type(exc).__name__}: {exc}")

        for model_id in selected_models:
            model_maps = [
                item for item in selected_nwp
                if NWP_LABELS.get(item, "").startswith("ECMWF") == (model_id == "ecmwf")
                and (item != "gfs_fram" or model_id == "gfs050")
            ]
            for map_id in model_maps:
                logs.append(f"\\n[Producto de modelo] {NWP_LABELS[map_id]}")
                for index, timestamp in enumerate(times, start=1):
                    for fxx in fxxs:
                        try:
                            context = (
                                ECMWFContext(timestamp, fxx, extent)
                                if model_id == "ecmwf"
                                else GFSContext(timestamp, fxx, model_id, extent)
                            )
                            logs.append(f"  [{index}/{len(times)}] {timestamp:%Y-%m-%d %H:%M} UTC · +{fxx} h")
                            image, raw = render_nwp_map(
                                map_id, context, run_dir, logs,
                                include_sources=include_sources,
                            )
                            if make_maps:
                                images.append(image)
                            elif image and Path(image).exists():
                                Path(image).unlink()
                            sources.extend(raw)
                        except Exception as exc:
                            logs.append(f"  ERROR modelo {map_id} {timestamp:%Y-%m-%d %H:%M} +{fxx} h: {type(exc).__name__}: {exc}")
        archive = write_bundle(
            run_dir,
            images,
            list(dict.fromkeys(sources)) if include_sources else [],
            logs,
            {"modo": "histórico/reanálisis", "periodo_utc": [start, end], "cadencia_horas": cadence, "horas_pronostico": fxxs, "modelos": selected_models, "mapas_modelo": selected_nwp, "satellite": satellite, "mapas_satelite": selected_sat, "scope": scope, "region": extent, "alta_resolucion": high_resolution, "salida": output_mode},
            "procesamiento_historico_meteorologia_sinoptica",
        )
        logs.append(f"\nMapas generados: {len(images)}")
        if archive:
            logs.append(f"Paquete: {archive}")
        return result_tuple(images, archive, logs)
    except Exception as exc:
        logs.append(f"ERROR: {type(exc).__name__}: {exc}")
        return result_tuple([], None, logs)





def run_eumetsat_download(collection_id, start_text, end_text, consumer_key, consumer_secret, max_products):
    """Descarga productos nativos de EUMETSAT Data Store y los entrega en ZIP."""
    import itertools
    import shutil
    import eumdac

    logs = ["=== DESCARGA EUMETSAT DATA STORE ==="]
    try:
        key = str(consumer_key or "").strip()
        secret = str(consumer_secret or "").strip()
        if not key or not secret:
            raise ValueError("Introduzca Consumer Key y Consumer Secret de EUMETSAT.")
        start = parse_datetime(start_text)
        end = parse_datetime(end_text)
        if isinstance(start, str) or isinstance(end, str) or start >= end:
            raise ValueError("El intervalo UTC debe tener inicio anterior al final.")
        start = start.replace(tzinfo=UTC)
        end = end.replace(tzinfo=UTC)
        limit = max(1, min(int(max_products), 5))

        logs.append(f"Colección: {collection_id}")
        logs.append(f"Intervalo UTC: {start:%Y-%m-%d %H:%M} → {end:%Y-%m-%d %H:%M}")
        logs.append(f"Límite solicitado: {limit} producto(s)")
        token = eumdac.AccessToken((key, secret))
        datastore = eumdac.DataStore(token)
        collection = datastore.get_collection(collection_id)
        products = list(itertools.islice(collection.search(dtstart=start, dtend=end), limit))
        if not products:
            logs.append("No se encontraron productos para ese intervalo. Revise fecha, colección y disponibilidad.")
            return None, "\n".join(logs)

        run_tag = f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
        download_dir = OUTPUT_DIR / "eumetsat" / run_tag
        download_dir.mkdir(parents=True, exist_ok=True)
        downloaded, product_ids = [], []
        for index, product in enumerate(products, start=1):
            product_id = str(product)
            try:
                with product.open() as source:
                    source_name = Path(str(getattr(source, "name", ""))).name
                    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", product_id).strip("._")
                    filename = source_name or safe_id or f"eumetsat_{index:03d}.bin"
                    if not Path(filename).suffix:
                        filename += ".bin"
                    target = download_dir / f"{index:03d}_{filename}"
                    temporary = target.with_name(target.name + ".part")
                    with temporary.open("wb") as destination:
                        shutil.copyfileobj(source, destination)
                    temporary.replace(target)
                if target.stat().st_size == 0:
                    target.unlink(missing_ok=True)
                    raise IOError("EUMETSAT devolvió un archivo vacío.")
                downloaded.append(target)
                product_ids.append(product_id)
                logs.append(f"OK {index}: {target.name} · {target.stat().st_size / 1024**2:.1f} MB")
            except Exception as product_error:
                logs.append(f"ERROR producto {product_id}: {type(product_error).__name__}: {product_error}")

        if not downloaded:
            logs.append("No se pudo descargar ningún producto; revise credenciales y colección.")
            return None, "\n".join(logs)

        archive = download_dir / f"EUMETSAT_{run_tag}.zip"
        manifest = {
            "provider": "EUMETSAT Data Store",
            "collection_id": collection_id,
            "collection_title": str(getattr(collection, "title", collection_id)),
            "start_utc": start.isoformat(),
            "end_utc": end.isoformat(),
            "product_ids": product_ids,
            "notice": "Productos descargados en su formato nativo; consultar licencia y metadatos EUMETSAT.",
        }
        with zipfile.ZipFile(archive, mode="w", compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
            bundle.writestr("EUMETSAT_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
            for item in downloaded:
                bundle.write(item, arcname=item.name)
        logs.append(f"\nProductos descargados: {len(downloaded)}")
        logs.append("Las claves API no se muestran ni se guardan en el ZIP.")
        return str(archive), "\n".join(logs)
    except Exception as exc:
        logs.append(f"ERROR: {type(exc).__name__}: {exc}")
        logs.append("Si el error es 401/403, genere/active las claves API de EUMETSAT y acepte los términos de la colección en el portal.")
        return None, "\n".join(logs)


In [ ]:
# ============================================================================
# ASCAT L2 (OSI SAF): importación de NetCDF local desde el navegador
# No se solicita ningún token y no se monta el Google Drive del usuario.
# Las direcciones NetCDF OSI SAF son OCEANOGRÁFICAS (hacia donde sopla).
# ============================================================================

ASCAT_DIR = DATA_DIR / "ascat"
ASCAT_DIR.mkdir(parents=True, exist_ok=True)


def _ascat_variable(ds: xr.Dataset, names: tuple[str, ...]) -> xr.DataArray:
    for name in names:
        if name in ds.variables:
            return ds[name]
    raise KeyError(
        f"No se encontró {names}. Variables disponibles: {list(ds.variables)}"
    )


def _ascat_swath(nc_path: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Lee variables de OSI SAF ASCAT y aplica el filtrado WVC básico.

    Dirección NetCDF: convención oceanográfica (0° significa movimiento al norte).
    Calidad: descarta VarQC, KNMI QC y monitoring (máscaras bits 16–18).
    """
    with xr.open_dataset(nc_path, mask_and_scale=True, decode_times=False) as ds:
        latitude = np.asarray(_ascat_variable(ds, ("lat", "latitude", "wvc_lat")).values, dtype=float)
        longitude = np.asarray(_ascat_variable(ds, ("lon", "longitude", "wvc_lon")).values, dtype=float)
        speed = np.asarray(_ascat_variable(ds, ("wind_speed", "wind_speed_selection")).values, dtype=float)
        direction = np.asarray(_ascat_variable(ds, ("wind_dir", "wind_direction")).values, dtype=float)
        start_time = str(
            ds.attrs.get("start_date") or ds.attrs.get("time_coverage_start")
            or ds.attrs.get("start_time") or "Hora de paso no especificada"
        )
        flags = None
        if "wvc_quality_flag" in ds:
            flags = np.asarray(ds["wvc_quality_flag"].values)

    # Reducción de dimensiones triviales. En L2 la pasada usa filas x celdas.
    latitude, longitude, speed, direction = (
        np.squeeze(a) for a in (latitude, longitude, speed, direction)
    )
    if latitude.ndim == longitude.ndim == 1 and speed.ndim == 2:
        if speed.shape == (len(latitude), len(longitude)):
            longitude, latitude = np.meshgrid(longitude, latitude)
    if not (latitude.shape == longitude.shape == speed.shape == direction.shape):
        raise ValueError(
            "Dimensiones de lat/lon/viento no compatibles: "
            f"lat={latitude.shape}, lon={longitude.shape}, "
            f"velocidad={speed.shape}, dirección={direction.shape}"
        )
    # Normalizar longitudes a -180..180 para las regiones del notebook.
    longitude = ((longitude + 180) % 360) - 180
    valid = np.isfinite(latitude) & np.isfinite(longitude)
    valid &= np.isfinite(speed) & np.isfinite(direction)
    valid &= (speed >= 0) & (speed <= 60)
    rejected = 0
    if flags is not None:
        flags = np.squeeze(flags)
        if flags.shape == speed.shape:
            # NetCDF OSI SAF: QC bits 16, 17 y 18; no confundir con el fondo ECMWF.
            bad = (np.nan_to_num(flags, nan=0).astype(np.int64)
                   & ((1 << 16) | (1 << 17) | (1 << 18))) != 0
            rejected = int(np.count_nonzero(valid & bad))
            valid &= ~bad
    return latitude[valid], longitude[valid], speed[valid], direction[valid], start_time, rejected


def run_ascat_uploaded(
    uploaded_file: str | None,
    region: str,
    min_lon: float, max_lon: float, min_lat: float, max_lat: float,
    include_file: bool,
):
    logs = ["=== ASCAT OSI SAF — VIENTOS OCEÁNICOS OBSERVADOS ==="]
    try:
        if not uploaded_file:
            raise ValueError("Seleccione un NetCDF ASCAT (.nc o .nc4) antes de ejecutar.")
        path = Path(uploaded_file)
        if not path.is_file():
            raise FileNotFoundError("El archivo seleccionado no está disponible en Colab.")
        if path.suffix.lower() not in (".nc", ".nc4", ".cdf"):
            raise ValueError("Se admite NetCDF ASCAT (.nc, .nc4, .cdf); no se admite BUFR directamente.")
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        lat, lon, speed, direction, time_text, rejected = _ascat_swath(str(path))
        region_mask = ((lon >= extent[0]) & (lon <= extent[1]) &
                       (lat >= extent[2]) & (lat <= extent[3]))
        lat, lon, speed, direction = [a[region_mask] for a in (lat, lon, speed, direction)]
        if speed.size == 0:
            raise ValueError("El archivo no contiene vientos ASCAT válidos dentro de esta región. "
                             "Pruebe con América para verificar la huella satelital.")
        run_dir = make_run_dir("ascat")
        # Puntos de dispersión y vectores con muestreo para evitar saturar Colab.
        pts = max(1, int(np.ceil(speed.size / 25000)))
        vec = max(1, int(np.ceil(speed.size / 400)))
        fig, ax = _base_map(extent)
        plotted = ax.scatter(lon[::pts], lat[::pts], c=speed[::pts],
                             s=4, vmin=0, vmax=30, cmap="viridis",
                             linewidths=0, transform=ccrs.PlateCarree())
        # Dirección ASCAT NetCDF OSI SAF: hacia dónde sopla el viento.
        theta = np.deg2rad(direction[::vec])
        u = speed[::vec] * np.sin(theta)
        v = speed[::vec] * np.cos(theta)
        ax.quiver(lon[::vec], lat[::vec], u, v,
                  scale=350, width=0.0018, color="black",
                  transform=ccrs.PlateCarree())
        fig.colorbar(plotted, ax=ax, pad=0.03, shrink=0.72, label="Viento ASCAT 10 m (m/s)")
        ax.set_title("ASCAT / Metop — Vientos oceánicos observados", loc="left", fontweight="bold")
        ax.set_title(f"{time_text} | OSI SAF L2", loc="right", fontsize=8)
        out_file = _save_figure(fig, run_dir / "ascat_viento_oceanico.png")
        logs.append(f"Archivo: {path.name}")
        logs.append(f"Puntos válidos en la región: {speed.size:,} | WVC descartados por calidad: {rejected:,}")
        logs.append("Dirección: convención oceanográfica NetCDF (hacia donde sopla).")
        logs.append("ASCAT = observación de pasada; ECMWF IFS = pronóstico en grilla. No tienen la misma hora/resolución automáticamente.")
        archive = write_bundle(
            run_dir, [out_file], [str(path)] if include_file else [], logs,
            {"modo":"observación ASCAT", "origen":"NetCDF OSI SAF L2", "region":extent,
             "fecha_producto":time_text, "datos_fuente_incluidos":include_file},
            "ascat_vientos_oceanicos",
        )
        return result_tuple([out_file], archive, logs)
    except Exception as exc:
        logs.append(f"ERROR: {type(exc).__name__}: {exc}")
        return result_tuple([], None, logs)




In [ ]:
# ============================================================================
# DESCARGA AUTOMÁTICA ASCAT
# A) NOAA CoastWatch ERDDAP: composición 1 día, pública, sin credenciales.
# B) NOAA STAR: respaldo e histórico por fecha, público y sin claves (en siguiente celda).
# No se guardan credenciales, no se monta Drive, no se hace descarga al iniciar.
# ============================================================================

ASCAT_ERDDAP_ID = "erdQCwindproducts1day"
ASCAT_ERDDAP = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/"


def _ascat_auto_error(logs, exc):
    logs.append(f"ERROR: {type(exc).__name__}: {exc}")
    return result_tuple([], None, logs)


def run_ascat_noaa(region, min_lon, max_lon, min_lat, max_lat, include_file):
    """Obtiene el ÚLTIMO compuesto diario disponible (no pasada L2)."""
    logs = ["=== ASCAT AUTOMÁTICO — NOAA COASTWATCH / ERDDAP ===",
            "Producto: compuesto diario de Metop-C (L4), no huella individual L2."]
    try:
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        west, east, south, north = [float(v) for v in extent]
        # ERDDAP requiere restricciones [tiempo][altura][latitud][longitud].
        # [last] selecciona el último tiempo realmente DISPONIBLE, no el de hoy.
        selector = f"[(last)][(10.0)][({south}):({north})][({west}):({east})]"
        query = f"wind_u{selector},wind_v{selector}"
        url = f"{ASCAT_ERDDAP}{ASCAT_ERDDAP_ID}.nc?{query}"
        run_dir = make_run_dir("ascat_noaa")
        nc_path = run_dir / "ascat_c_ultimo_compuesto_diario.nc"
        logs.append(f"Solicitando región W/E/S/N: {extent}")
        # No descargar el archivo global; solo la región requerida y dos variables.
        with requests.get(url, timeout=(20, 120), stream=True,
                          headers={"User-Agent": "AplicacionMeteorologiaSinoptica-ASCAT/1.0"}) as resp:
            resp.raise_for_status()
            received = 0
            with nc_path.open("wb") as out:
                for block in resp.iter_content(chunk_size=256 * 1024):
                    if block:
                        received += len(block)
                        if received > 50 * 1024 * 1024:
                            raise ValueError("El subconjunto ASCAT excede el límite de 50 MB.")
                        out.write(block)
        with xr.open_dataset(nc_path, mask_and_scale=True) as ds:
            if not all(v in ds for v in ("wind_u", "wind_v")):
                raise ValueError("El servidor devolvió un archivo sin wind_u y wind_v.")
            u_da = ds["wind_u"].squeeze(drop=True)
            v_da = ds["wind_v"].squeeze(drop=True)
            u = np.asarray(u_da.values, dtype=float)
            v = np.asarray(v_da.values, dtype=float)
            lons = np.asarray(ds["longitude"].values, dtype=float)
            lats = np.asarray(ds["latitude"].values, dtype=float)
            time_vals = np.asarray(ds["time"].values).reshape(-1)
            if len(time_vals) != 1:
                raise ValueError("Se esperaba un único tiempo en el producto ASCAT.")
            sample_time = np.datetime64(time_vals[0], "s")
            utc_time = datetime.fromisoformat(np.datetime_as_string(sample_time, unit="s")).replace(tzinfo=UTC)
        if u.ndim != 2 or u.shape != v.shape or u.shape != (len(lats), len(lons)):
            raise ValueError(f"La grilla ASCAT no coincide: u={u.shape}, v={v.shape}, lat={len(lats)}, lon={len(lons)}")
        age_hours = (datetime.now(UTC) - utc_time).total_seconds() / 3600
        logs.append(f"Fecha efectiva de los datos: {utc_time:%Y-%m-%d %H:%M} UTC")
        logs.append(f"Antigüedad del compuesto: {age_hours:.1f} horas")
        stale = age_hours > 72
        if stale:
            logs.append("ADVERTENCIA: el compuesto NOAA está desactualizado (>72 h); NO es situación actual.")
        valid = np.isfinite(u) & np.isfinite(v) & (np.hypot(u, v) <= 60)
        if valid.sum() < 2:
            raise ValueError("Sin vectores ASCAT válidos en la región/tiempo. Intente Caribe o América.")
        speed = np.where(valid, np.hypot(u, v), np.nan)
        xx, yy = np.meshgrid(lons, lats)
        fig, ax = _base_map(extent)
        mesh = ax.pcolormesh(xx, yy, speed, cmap="viridis", vmin=0, vmax=25,
                             shading="auto", transform=ccrs.PlateCarree())
        stride = max(1, int(max(u.shape) / 24))
        ax.quiver(xx[::stride, ::stride], yy[::stride, ::stride],
                  np.where(valid, u, np.nan)[::stride, ::stride],
                  np.where(valid, v, np.nan)[::stride, ::stride],
                  color="black", scale=320, width=0.002, transform=ccrs.PlateCarree())
        fig.colorbar(mesh, ax=ax, shrink=0.72, pad=0.03, label="Velocidad viento ASCAT (m/s)")
        ax.set_title("ASCAT / Metop-C — Viento sobre el océano (compuesto 1 día)",
                     loc="left", fontweight="bold")
        label = f"{utc_time:%Y-%m-%d %H:%M} UTC — NOAA / ERDDAP"
        if stale:
            label += " — DATOS ANTIGUOS"
        ax.set_title(label, loc="right", fontsize=8)
        image_path = _save_figure(fig, run_dir / "ascat_noaa_viento_10m.png")
        logs.append(f"Píxeles con viento válido: {int(valid.sum()):,}")
        logs.append("NOAA: producto combinado diario derivado de ASCAT, NO observación de órbita individual.")
        archive = write_bundle(run_dir, [image_path], [str(nc_path)] if include_file else [], logs,
               {"fuente":"NOAA CoastWatch ERDDAP", "dataset":ASCAT_ERDDAP_ID,
                "producto":"compuesto diario ASCAT Metop-C", "hora_datos_utc":utc_time.isoformat(),
                "advertencia_desactualizado":stale, "region":extent}, "ascat_noaa_auto")
        return result_tuple([image_path], archive, logs)
    except Exception as exc:
        return _ascat_auto_error(logs, exc)



In [ ]:
# ===========================================================================
# NOAA / ASCAT: resiliencia de red + respaldo oficial NOAA/NESDIS/STAR
# Una caída del ERDDAP NO implica que la observación ASCAT no exista.
# El respaldo se rechaza si el último archivo es demasiado antiguo.
# ===========================================================================
import re
from urllib.parse import urljoin

ASCAT_STAR_MIRRORS = (
    'https://www.star.nesdis.noaa.gov/pub/socd7/coastwatch/metop/ascat/netcdf/day/',
    'https://coastwatch.noaa.gov/pub/socd7/coastwatch/metop/ascat/netcdf/day/',
)
ASCAT_MAX_DIAS_OPERATIVOS = 5
ASCAT_STAR_REGEX = re.compile(r'AS(20\d{5})([ABC])(as|ds)_WW\.nc', re.IGNORECASE)


def _ascat_red_diagnostico():
    """Chequea acceso HTTP sin descargar grandes archivos (evita exponer credenciales)."""
    endpoints = (
        ('Internet general', 'https://www.google.com/generate_204'),
        ('NOAA ERDDAP', 'https://coastwatch.pfeg.noaa.gov/erddap/version'),
        ('NOAA STAR', 'https://www.star.nesdis.noaa.gov/'),
            )
    lines = ['=== DIAGNÓSTICO DE CONECTIVIDAD ASCAT ===']
    for name, url in endpoints:
        try:
            r = requests.get(url, timeout=(5, 9), allow_redirects=True, stream=True)
            # HTTP 401/403/404 sigue probando que se pudo llegar al servidor
            lines.append(f'{name}: conexión HTTP establecida (HTTP {r.status_code})')
            r.close()
        except requests.exceptions.RequestException as exc:
            lines.append(f'{name}: SIN CONEXIÓN — {type(exc).__name__}: {str(exc)[:170]}')
    lines.append('Si Internet general responde pero NOAA ERDDAP falla, el problema es de acceso a ese host.')
    lines.append('Si tampoco responde Internet general, revise red/restricciones del entorno Colab.')
    return '\n'.join(lines)


def _ascat_star_catalog():
    """Obtiene enlaces del directorio HTML público; no asume que está actualizado."""
    errors = []
    for base in ASCAT_STAR_MIRRORS:
        try:
            with requests.get(base, timeout=(9, 32), stream=True,
                    headers={'User-Agent':'MeteorologiaSinoptica-ASCAT/1.1'}) as r:
                r.raise_for_status()
                content = bytearray()
                for part in r.iter_content(chunk_size=131072):
                    if part:
                        content.extend(part)
                    if len(content) > 18 * 1024 * 1024:
                        raise ValueError('El índice NOAA STAR excedió 18 MB; se omitió para proteger Colab.')
            names = set(m.group(0) for m in ASCAT_STAR_REGEX.finditer(content.decode('utf-8', 'replace')))
            dated = []
            for name in names:
                match = ASCAT_STAR_REGEX.fullmatch(name)
                try:
                    day = datetime.strptime(match.group(1), '%Y%j').replace(tzinfo=UTC)
                except (ValueError, AttributeError):
                    continue
                # Priorizar Metop-C (y después B), nunca Metop-A retirado.
                dated.append((day, match.group(2).upper()=='C', name))
            dated.sort(reverse=True)
            if not dated:
                raise ValueError('Directorio accesible pero no contiene archivos diarios ASCAT reconocibles.')
            return base, dated
        except (requests.exceptions.RequestException, ValueError) as e:
            errors.append(f'{base.split("/")[2]}: {type(e).__name__}: {str(e)[:150]}')
    raise ConnectionError('No fue posible consultar el catálogo NOAA STAR. ' + '; '.join(errors))


def _ascat_star_download(base: str, name: str, dst: Path):
    if not ASCAT_STAR_REGEX.fullmatch(name):
        raise ValueError('Nombre de archivo NOAA STAR no permitido.')
    url = urljoin(base, name)
    with requests.get(url, stream=True, timeout=(9, 75),
                      headers={'User-Agent':'MeteorologiaSinoptica-ASCAT/1.1'}) as r:
        r.raise_for_status()
        n=0
        with dst.open('wb') as out:
            for block in r.iter_content(chunk_size=131072):
                if block:
                    n += len(block)
                    if n > 24*1024*1024:
                        raise ValueError('NetCDF NOAA STAR excede 24 MB; descarga cancelada.')
                    out.write(block)
    if dst.stat().st_size < 100:
        raise ValueError('Archivo NOAA STAR vacío o inválido.')


def _ascat_star_esquema(nc_path):
    """Resumen de cabecera útil si STAR cambia nombres o estructura de NetCDF."""
    with xr.open_dataset(nc_path, decode_times=False, mask_and_scale=False) as ds:
        dimensions = ', '.join(f'{k}={v}' for k,v in ds.sizes.items())
        variables = '; '.join(f'{k} {tuple(v.dims)}'
                              for k,v in list(ds.variables.items())[:35])
        metas = ('geospatial_lat_min','geospatial_lat_max',
                 'geospatial_lon_min','geospatial_lon_max',
                 'latitude_min','latitude_max','longitude_min','longitude_max',
                 'geospatial_lat_resolution','geospatial_lon_resolution',
                 'spatial_resolution', 'projection', 'Conventions')
        attributes = '; '.join(f'{k}={str(ds.attrs[k])[:70]}' for k in metas if k in ds.attrs)
        return f'Esquema NetCDF: dimensiones: {dimensions} | variables: {variables[:700]} | atributos: {attributes[:360]}'


def _ascat_star_dataset(nc_path):
    """Lee NetCDF NOAA STAR sin suponer nombres únicos de coordenadas/variables.

    Coordenadas válidas: variables 1D/2D explícitas CF o grilla recuperable
    únicamente con límites espaciales y espaciado explícitos en metadatos.
    Retorna (lon, lat, u, v, speed, procedencia); u/v son NaN si el archivo
    contiene dirección sin convención meteorológica explícita.
    """
    import re as _re

    def norm(s):
        return _re.sub(r'[^a-z0-9]', '', str(s).lower())

    def pick(ds, names, standard=(), units=(), axis=None, exclude=()):
        names_norm={norm(n) for n in names}
        # Preferir coincidencia explícita del nombre.
        for name,var in ds.variables.items():
            if norm(name) in names_norm and not any(norm(t) in norm(name) for t in exclude):
                return ds[name]
        for name,var in ds.variables.items():
            if any(norm(t) in norm(name) for t in exclude):
                continue
            sn=norm(var.attrs.get('standard_name',''))
            if sn and sn in {norm(x) for x in standard}:
                return ds[name]
        for name,var in ds.variables.items():
            if any(norm(t) in norm(name) for t in exclude):
                continue
            unit=norm(var.attrs.get('units',''))
            axes=norm(var.attrs.get('axis',''))
            if units and (unit in {norm(x) for x in units}) and (axis is None or axes==axis.lower()):
                return ds[name]
        return None

    def squeeze2(da):
        a=da.squeeze(drop=True)
        if a.ndim > 2:
            raise ValueError(f'Variable {da.name} contiene {a.ndim} dimensiones no triviales; '
                             'se requiere conocer sus coordenadas antes de seleccionar una capa.')
        return np.asarray(a.values,dtype=float),a.dims

    def find_coord(ds, typ):
        if typ=='lat':
            names=('latitude','lat','wvc_lat','wvc_latitude','grid_lat','grid_latitude','nav_lat','latitudes','Latitude','LATITUDE')
            standard=('latitude',); units=('degrees_north','degree_north','degrees_n','degree_n')
            ax='Y'
        else:
            names=('longitude','lon','wvc_lon','wvc_longitude','grid_lon','grid_longitude','nav_lon','longitudes','Longitude','LONGITUDE')
            standard=('longitude',); units=('degrees_east','degree_east','degrees_e','degree_e')
            ax='X'
        out=pick(ds,names,standard)
        if out is not None:
            return out
        # Identificar por unidades o eje: los nombres pueden ser 'x', 'y'.
        for name,v in ds.variables.items():
            unit=norm(v.attrs.get('units',''))
            axis=norm(v.attrs.get('axis',''))
            sn=norm(v.attrs.get('standard_name',''))
            if unit in {norm(x) for x in units} or sn==typ+'itude' or axis==ax.lower():
                if v.ndim in (1,2) and np.issubdtype(v.dtype,np.number):
                    # No reinterpretar coordenadas proyectadas en metros como grados.
                    if axis==ax.lower() and unit not in {norm(x) for x in units}:
                        continue
                    return ds[name]
        return None

    def known_geobounds(ds, nrow, ncol):
        # Reconstrucción SOLO si el archivo declara explícitamente extensión
        # espacial y resolución (centros vs extremos se verifica).
        def attr(keys):
            for k in keys:
                if k in ds.attrs:
                    try:return float(ds.attrs[k])
                    except (TypeError,ValueError): pass
            return None
        la0=attr(('geospatial_lat_min','latitude_min','southmost_latitude','Southernmost_Northing'))
        la1=attr(('geospatial_lat_max','latitude_max','northmost_latitude','Northernmost_Northing'))
        lo0=attr(('geospatial_lon_min','longitude_min','westernmost_longitude','Westernmost_Easting'))
        lo1=attr(('geospatial_lon_max','longitude_max','easternmost_longitude','Easternmost_Easting'))
        dlat=attr(('geospatial_lat_resolution','latitude_resolution','lat_resolution'))
        dlon=attr(('geospatial_lon_resolution','longitude_resolution','lon_resolution'))
        if None in (la0,la1,lo0,lo1,dlat,dlon) or dlat<=0 or dlon<=0:
            return None
        if not (-90<=la0<la1<=90 and -360<=lo0<lo1<=360):
            return None
        if abs((la1-la0)-dlat*(nrow-1)) > .03 or abs((lo1-lo0)-dlon*(ncol-1)) > .03:
            return None
        return np.meshgrid(np.linspace(lo0,lo1,ncol),np.linspace(la0,la1,nrow))

    with xr.open_dataset(nc_path, mask_and_scale=True) as ds:
        u_da=pick(ds,('wind_u','uwnd','eastward_wind','u10','u_wind','wind_velocity_u','uwnd_ascending','uwnd_descending'),('eastward_wind',),exclude=('model','gfs','forecast'))
        v_da=pick(ds,('wind_v','vwnd','northward_wind','v10','v_wind','wind_velocity_v','vwnd_ascending','vwnd_descending'),('northward_wind',),exclude=('model','gfs','forecast'))
        speed_da=pick(ds,('wind_speed','windspeed','wind_spd','wspd','wspd_ascending','wspd_descending','speed','windvelocity','surface_wind_speed','ascat_wind_speed'),('wind_speed',),exclude=('model','gfs','forecast','difference'))
        dir_da=pick(ds,('wind_direction','wind_dir','winddir','wdir','wind_from_direction','direction','surface_wind_direction'),('wind_from_direction','wind_to_direction'),exclude=('model','gfs','forecast'))
        if u_da is not None and v_da is not None:
            u, dims=squeeze2(u_da)
            v, dims_v=squeeze2(v_da)
            if dims != dims_v or u.shape != v.shape:
                raise ValueError('Componentes U/V no comparten dimensiones.')
            speed=np.hypot(u,v)
            procedure='componentes U/V observadas'
        elif speed_da is not None:
            speed,dims=squeeze2(speed_da)
            u=np.full(speed.shape,np.nan);v=np.full(speed.shape,np.nan)
            procedure='rapidez observada (dirección no declarada)'
            if dir_da is not None:
                direction, ddims=squeeze2(dir_da)
                if speed.shape==direction.shape and dims==ddims:
                    descriptors=' '.join(str(dir_da.attrs.get(k,'')) for k in
                            ('standard_name','long_name','description','comment','convention')).lower()
                    # Si dirección es FROM, convertir a movimiento cartesiano; si es TO,
                    # usar la convención oceanográfica sin invertir.
                    if 'wind_from_direction' in descriptors or 'direction from' in descriptors or 'from which' in descriptors:
                        u=-speed*np.sin(np.deg2rad(direction));v=-speed*np.cos(np.deg2rad(direction))
                        procedure='rapidez/dirección meteorológica FROM'
                    elif 'wind_to_direction' in descriptors or 'direction to' in descriptors:
                        u=speed*np.sin(np.deg2rad(direction));v=speed*np.cos(np.deg2rad(direction))
                        procedure='rapidez/dirección oceanográfica TO'
        else:
            raise ValueError('Sin viento observacional reconocido (U/V o rapidez). Variables: '+', '.join(ds.data_vars))

        if speed.ndim not in (1,2):
            raise ValueError('Dimensiones de viento no compatibles: '+str(speed.shape))
        latitude=find_coord(ds,'lat');longitude=find_coord(ds,'lon')
        xx=yy=None
        if latitude is not None and longitude is not None:
            la, ladims=squeeze2(latitude);lo,lodims=squeeze2(longitude)
            # Lat/lon como coordenadas 2D de huella o swath.
            if la.ndim==lo.ndim==2 and la.shape==lo.shape==speed.shape:
                yy,xx=la,lo
            elif speed.ndim==2 and la.ndim==lo.ndim==1:
                if ladims[0] in dims and lodims[0] in dims and ladims[0]!=lodims[0]:
                    # Alinear mediante nombres de dimensión, no suponer orientación.
                    xx,yy=np.meshgrid(lo,la,indexing='xy')
                    if dims==(lodims[0],ladims[0]):
                        xx=xx.T;yy=yy.T
                elif speed.shape==(len(la),len(lo)):
                    xx,yy=np.meshgrid(lo,la)
                elif speed.shape==(len(lo),len(la)):
                    xx,yy=np.meshgrid(lo,la);xx=xx.T;yy=yy.T
            elif speed.ndim==1 and la.ndim==lo.ndim==1 and la.shape==lo.shape==speed.shape:
                yy,xx=la,lo
        if xx is None and speed.ndim==2:
            geo=known_geobounds(ds,*speed.shape)
            if geo is not None:
                xx,yy=geo
                procedure+='; malla desde límites y resolución explícitos'
        if xx is None:
            raise ValueError('Sin coordenadas geográficas suficientes. '+_ascat_star_esquema(nc_path))
        if xx.shape!=speed.shape or yy.shape!=speed.shape:
            raise ValueError('Coordenadas y viento incompatibles. '+_ascat_star_esquema(nc_path))
        # NOAA STAR puede almacenar longitudes 0..360.
        xx=np.where(np.isfinite(xx),(xx+180)%360-180,np.nan)
        valid_geo=np.isfinite(xx)&np.isfinite(yy)&(yy>=-90)&(yy<=90)
        xx=np.where(valid_geo,xx,np.nan); yy=np.where(valid_geo,yy,np.nan)
        speed=np.where((speed>=0)&(speed<60),speed,np.nan)
        uv_valid=np.isfinite(u)&np.isfinite(v)&(np.hypot(u,v)<60)
        u=np.where(uv_valid,u,np.nan);v=np.where(uv_valid,v,np.nan)
        return xx, yy, u, v, speed, procedure

def _ascat_noaa_star(region, min_lon, max_lon, min_lat, max_lat, include_file):
    logs=['=== ASCAT AUTOMÁTICO — RESPALDO NOAA STAR ===',
          'Producto: grilla diaria L3 (NO es una pasada L2 de 12.5 km).']
    try:
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        base, dated = _ascat_star_catalog()
        latest_date = dated[0][0]
        age_days = (datetime.now(UTC) - latest_date).total_seconds()/86400
        logs.append(f'Fecha del último archivo publicado: {latest_date:%Y-%m-%d} UTC')
        logs.append(f'Antigüedad del último archivo: {age_days:.1f} días')
        if age_days > ASCAT_MAX_DIAS_OPERATIVOS:
            raise ValueError(f'NOAA STAR no posee datos recientes para operación: '
                             f'último archivo {latest_date:%Y-%m-%d}, más de '
                             f'{ASCAT_MAX_DIAS_OPERATIVOS} días. No se graficará como actual.')
        # Usar las grillas Metop-C más recientes (ascendente/descendente).
        latest = [rec for rec in dated if rec[0]==latest_date and rec[1]][:2]
        if not latest:
            latest = [rec for rec in dated if rec[0]==latest_date][:2]
        run_dir = make_run_dir('ascat_star')
        figures=[]; ncs=[]
        for day, metopc, name in latest:
            path = run_dir / name
            try:
                _ascat_star_download(base, name, path)
                logs.append(f'{name}: '+_ascat_star_esquema(path))
                xx,yy,u,v,sp,procedure=_ascat_star_dataset(path)
                # Sin rellenar observaciones inexistentes con pronósticos.
                valid=np.isfinite(xx)&np.isfinite(yy)&np.isfinite(sp)
                valid &= (xx>=extent[0])&(xx<=extent[1])&(yy>=extent[2])&(yy<=extent[3])
                if np.count_nonzero(valid)<2:
                    logs.append(f'{name}: sin observaciones válidas para la región seleccionada.')
                    continue
                sp=np.where(valid,sp,np.nan)
                fig,ax=_base_map(extent)
                # Puntos muestreados dentro de la región: evita saltos de longitud
                # en mallas globales 0..360 normalizadas a -180..180.
                indexes=np.flatnonzero(valid)
                indexes=indexes[::max(1,int(np.ceil(len(indexes)/55000)))]
                mesh=ax.scatter(xx.flat[indexes],yy.flat[indexes],c=sp.flat[indexes],s=6,
                                cmap='viridis',vmin=0,vmax=25,linewidths=0,
                                transform=ccrs.PlateCarree())
                vmask=valid&np.isfinite(u)&np.isfinite(v)
                if vmask.sum()>0:
                    vecids=np.flatnonzero(vmask)[::max(1,int(np.ceil(vmask.sum()/400)))]
                    ax.quiver(xx.flat[vecids],yy.flat[vecids],u.flat[vecids],v.flat[vecids],
                              color='black',scale=320,width=0.002,
                              transform=ccrs.PlateCarree())
                else:
                    logs.append(f'{name}: solo rapidez; no se inventaron vectores de dirección.')
                fig.colorbar(mesh,ax=ax,shrink=0.72,pad=0.03,label='Viento ASCAT (m/s)')
                logs.append(f'{name}: método: {procedure}')
                ax.set_title(f'ASCAT NOAA STAR — {name}',loc='left',fontsize=10,fontweight='bold')
                ax.set_title(f'Fecha archivo: {day:%Y-%m-%d} UTC (L3)',loc='right',fontsize=8)
                figures.append(_save_figure(fig,run_dir / f'{name[:-3]}.png'))
                ncs.append(str(path)); logs.append(f'{name}: {np.count_nonzero(valid)} puntos válidos.')
            except (OSError, ValueError, KeyError, requests.exceptions.RequestException) as e:
                logs.append(f'{name}: {type(e).__name__}: {e}')
        if not figures:
            raise ValueError('No se pudo generar un mapa NOAA STAR para esta región. '
                             'Revise el registro o cargue un NetCDF local.')
        archive=write_bundle(run_dir,figures,ncs if include_file else [],logs,
                {'fuente':'NOAA CoastWatch STAR','tipo':'ASCAT L3 compuesto diario',
                 'fecha_archivo':latest_date.isoformat(),'region':extent},'ascat_star_auto')
        return result_tuple(figures,archive,logs)
    except Exception as e:
        return _ascat_auto_error(logs,e)

# Usar ERDDAP primero, luego NOAA STAR si falla la conexión o la petición.
_ascat_noaa_erddap_original = run_ascat_noaa

def run_ascat_noaa(region, min_lon, max_lon, min_lat, max_lat, include_file):
    a, b, report = _ascat_noaa_erddap_original(region, min_lon, max_lon, min_lat, max_lat, include_file)
    if a:
        return a,b,report
    # Si ERDDAP no devuelve datos válidos, explorar la otra fuente oficial.
    c,d,report2 = _ascat_noaa_star(region, min_lon, max_lon, min_lat, max_lat, include_file)
    combined = report + '\n\n--- RESPALDO NOAA STAR ---\n' + report2
    if not c:
        combined += ('\n\nSOLUCIONES: use «Diagnosticar red ASCAT», revise la conexión '
                     'o utilice NOAA STAR histórico / cargue un NetCDF local. '
                     'No se sustituyen observaciones faltantes por vientos pronosticados ECMWF.')
    return c,d,combined


# ============================================================================
# CONSULTA HISTÓRICA ASCAT NOAA STAR — SIN API KEY
# Sólo se consultan las fechas presentes en el directorio público NOAA.
# Respetar períodos: no reutilizar el archivo "último" como dato histórico.
# ============================================================================
ASCAT_HIST_MAX_DIAS = 7


def _ascat_star_historical_selection(catalog, start_day, end_day, satellite='Auto', orbit='Ambas'):
    """Selecciona nombres efectivamente publicados en el catálogo público.

    Permite varias fechas (máx. ASCAT_HIST_MAX_DIAS), hasta dos asc/desc
    por día. Prioriza Metop-C, luego B y A únicamente cuando están listados.
    No interpreta datos faltantes como observaciones sin viento.
    """
    from datetime import date as _date
    if isinstance(start_day, str):
        start_day = _date.fromisoformat(start_day)
    if isinstance(end_day, str):
        end_day = _date.fromisoformat(end_day)
    if not start_day <= end_day:
        raise ValueError('Fecha inicial posterior a la final.')
    if (end_day - start_day).days + 1 > ASCAT_HIST_MAX_DIAS:
        raise ValueError(f'Seleccione un máximo de {ASCAT_HIST_MAX_DIAS} días por solicitud para proteger Colab.')
    if end_day > datetime.now(UTC).date():
        raise ValueError('No se permiten fechas futuras en el historial ASCAT.')
    if satellite not in ('Auto', 'A', 'B', 'C'):
        raise ValueError('Satélite no reconocido. Utilice Auto, A, B o C.')
    if orbit not in ('Ambas', 'Ascendente', 'Descendente'):
        raise ValueError('Tipo de pasada inválido.')

    indexed = {}
    for dt, _, name in catalog:
        m = ASCAT_STAR_REGEX.fullmatch(name)
        if not m or not (start_day <= dt.date() <= end_day):
            continue
        sat = m.group(2).upper()
        orb = m.group(3).lower()
        if orbit == 'Ascendente' and orb != 'as':
            continue
        if orbit == 'Descendente' and orb != 'ds':
            continue
        indexed.setdefault(dt.date(), []).append((sat,orb,name))

    chosen = []
    missing = []
    for i in range((end_day-start_day).days+1):
        target = start_day + timedelta(days=i)
        available = indexed.get(target, [])
        if satellite == 'Auto':
            selected_sat = next((s for s in ('C','B','A') if any(rec[0]==s for rec in available)), None)
        else:
            selected_sat = satellite
        files = [rec for rec in available if rec[0] == selected_sat]
        files.sort(key=lambda item: (item[1] != 'as', item[2]))
        if not files:
            missing.append(target.isoformat())
            continue
        for sat,orb,name in files[:2]:
            chosen.append((target,sat,orb,name))
    return chosen, missing


def run_ascat_noaa_historico(mode, start, end, satellite, orbit, region,
                              min_lon, max_lon, min_lat, max_lat, include_file):
    """Descarga y grafica ASCAT L3 de NOAA STAR según fecha/rango, sin credenciales."""
    logs = ['=== ASCAT HISTÓRICO — NOAA STAR / SIN API KEY ===',
            'Producto: compuesto diario L3 de ASCAT por pasada ascendente/descendente, NO L2.',
            'Cada mapa corresponde a la FECHA DEL ARCHIVO, no a una corrida ECMWF.']
    try:
        if mode not in ('Fecha única','Rango histórico'):
            raise ValueError('Modo temporal desconocido.')
        start_day = datetime.strptime(str(start).strip(),'%Y-%m-%d').date()
        end_day = start_day if mode == 'Fecha única' else datetime.strptime(str(end).strip(),'%Y-%m-%d').date()
        # Validar rango antes de descargar el catálogo.
        if start_day > end_day:
            raise ValueError('La fecha inicial debe ser anterior o igual a la final.')
        if (end_day-start_day).days+1 > ASCAT_HIST_MAX_DIAS:
            raise ValueError(f'Por solicitud se permiten como máximo {ASCAT_HIST_MAX_DIAS} días; divida los períodos largos.')
        if end_day > datetime.now(UTC).date():
            raise ValueError('La fecha solicitada está en el futuro.')
        extent = resolve_extent(region, min_lon, max_lon, min_lat, max_lat)
        logs.append(f'Fecha solicitada: {start_day} a {end_day} (UTC); satélite: {satellite}; pasada: {orbit}.')
        logs.append(f'Región W/E/S/N: {extent}')
        base, catalog = _ascat_star_catalog()
        logs.append(f'Catálogo NOAA STAR: {catalog[-1][0]:%Y-%m-%d} a {catalog[0][0]:%Y-%m-%d} '
                    f'({len(catalog)} archivos enumerados).')
        chosen, missing = _ascat_star_historical_selection(catalog, start_day, end_day, satellite, orbit)
        if missing:
            logs.append('Días sin archivo publicado para la selección: '+', '.join(missing))
        if not chosen:
            raise ValueError('No hay archivos ASCAT NOAA STAR en el catálogo para esas fechas/satélite/pasada. '
                             'Pruebe otro día o seleccione satélite Auto. No se generan datos ficticios.')
        logs.append(f'Archivos seleccionados: {len(chosen)} (límite de 2 por día).')
        run_dir = make_run_dir('ascat_star_historico')
        figures, source_files, success_dates = [], [], []
        for day,sat,orb,name in chosen:
            path = run_dir / name
            try:
                _ascat_star_download(base, name, path)
                xx,yy,u,v,sp,method = _ascat_star_dataset(path)
                valid = np.isfinite(xx) & np.isfinite(yy) & np.isfinite(sp)
                valid &= ((xx>=extent[0]) & (xx<=extent[1]) &
                          (yy>=extent[2]) & (yy<=extent[3]))
                if np.count_nonzero(valid) < 2:
                    logs.append(f'{name}: archivo descargado; sin observaciones válidas sobre la región.');
                    continue
                fig,ax = _base_map(extent)
                good_ids = np.flatnonzero(valid)
                point_ids = good_ids[::max(1,int(np.ceil(len(good_ids)/55000)))]
                mesh = ax.scatter(xx.flat[point_ids], yy.flat[point_ids], c=sp.flat[point_ids],
                                  s=6, cmap='viridis', vmin=0, vmax=25, linewidths=0,
                                  transform=ccrs.PlateCarree())
                vmask = valid & np.isfinite(u) & np.isfinite(v)
                if np.count_nonzero(vmask):
                    vecids = np.flatnonzero(vmask)
                    vecids = vecids[::max(1,int(np.ceil(len(vecids)/400)))]
                    ax.quiver(xx.flat[vecids], yy.flat[vecids], u.flat[vecids], v.flat[vecids],
                              color='black',scale=320,width=0.002,transform=ccrs.PlateCarree())
                else:
                    logs.append(f'{name}: sin componentes U/V confiables; no se inventan vectores.')
                fig.colorbar(mesh,ax=ax,shrink=0.72,pad=0.03,label='Viento ASCAT (m/s)')
                passing = 'Ascendente' if orb=='as' else 'Descendente'
                ax.set_title(f'ASCAT NOAA STAR — Metop-{sat} · {passing}',
                             loc='left',fontsize=10,fontweight='bold')
                ax.set_title(f'{day:%Y-%m-%d} UTC — HISTÓRICO L3',loc='right',fontsize=8)
                fig_path = _save_figure(fig,run_dir/f'{name[:-3]}_historico.png')
                figures.append(fig_path)
                success_dates.append(day.isoformat())
                if include_file:
                    source_files.append(str(path))
                logs.append(f'{name}: {np.count_nonzero(valid)} puntos válidos. Método: {method}')
            except (OSError, ValueError, KeyError, requests.exceptions.RequestException) as exc:
                logs.append(f'{name}: {type(exc).__name__}: {exc}')
        if not figures:
            raise ValueError('Se identificaron archivos históricos pero no hubo observaciones que se '
                             'pudieran representar en la región. Consulte el registro.')
        extra = {'fuente':'NOAA NESDIS STAR', 'producto':'ASCAT L3 grilla diaria por pasada',
                 'modo':'historico', 'fecha_inicial':start_day.isoformat(),
                 'fecha_final':end_day.isoformat(), 'satelite_seleccionado':satellite,
                 'pasada':orbit, 'fechas_graficadas':success_dates, 'region':extent,
                 'advertencia':'Compuesto gridded diario; no corresponde a huellas L2.'}
        bundle=write_bundle(run_dir,figures,source_files,logs,extra,'ascat_noaa_historico')
        return result_tuple(figures,bundle,logs)
    except Exception as exc:
        return _ascat_auto_error(logs,exc)


## ENDVI10 — NDVI, países de América y climatología seleccionable

Selector de 35 países, con puntos de referencia y límites geográficos precargados; serie completa de 1–180 décadas comparada por fecha con la media/P10–P90 de 3, 5, 8, 10, 15 años o período histórico personalizado. Se rechazan observaciones con STM inválido; los años con datos ausentes no se inventan.


In [ ]:
# ============================================================================
# LSA SAF / EUMETSAT — ENDVI10 (Metop-AVHRR, NetCDF4, OpenDAP)
# Compositos decádicos UTC; fuente oficial EPS/ENDVI10/NETCDF.
# Módulo autocontenido, apto para integrar como celda en el cuaderno principal.
# ============================================================================

from __future__ import annotations

import calendar
import csv
import io
import math
import re
import zipfile
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
from matplotlib.patches import Patch
import numpy as np
import xarray as xr

ENDVI_ROOT = 'https://datalsasaf.lsasvcs.ipma.pt/PRODUCTS/EPS/ENDVI10/NETCDF'
ENDVI_DAP = 'https://thredds.lsasvcs.ipma.pt/thredds/dodsC/EPS/ENDVI10/NETCDF'
ENDVI_PRODUCT_MIN = -0.08
ENDVI_PRODUCT_MAX = 0.92
ENDVI_LIMIT_PIXELS_AXIS = 1600
NDVI_PALETTE = ['#6d3d33', '#b95634', '#e6a64b', '#f1df7b', '#b4da71', '#58b666', '#217842']
NDVI_FIXED_BOUNDS = [-0.08, 0.0, 0.2, 0.35, 0.5, 0.65, 0.8, 0.92]
NDVI_CMAP = ListedColormap(NDVI_PALETTE, name='ENDVI10_vegetacion')
NDVI_CMAP.set_bad('#d4d8db')
NDVI_NORM = BoundaryNorm(NDVI_FIXED_BOUNDS, NDVI_CMAP.N, clip=True)
# Países soberanos de América (35). Punto inicial representativo EN TIERRA y
# extensión geográfica continental/insular aproximada [oeste, este, sur, norte].
# No son fronteras nacionales ni un polígono de recorte administrativo.
# Países con territorios ultramarinos/islas remotas: se prioriza el territorio principal.
ENDVI_COUNTRIES = {
    'Antigua y Barbuda': (-61.92, -61.65, 16.98, 17.75, 17.06, -61.79),
    'Argentina': (-73.60, -53.60, -55.10, -21.70, -34.61, -58.38),
    'Bahamas': (-79.40, -72.65, 20.80, 27.50, 25.07, -77.35),
    'Barbados': (-59.68, -59.42, 13.02, 13.35, 13.10, -59.61),
    'Belice': (-89.25, -87.40, 15.80, 18.55, 17.25, -88.77),
    'Bolivia': (-69.65, -57.35, -22.95, -9.60, -16.49, -68.12),
    'Brasil': (-74.00, -34.70, -34.00, 5.40, -15.78, -47.93),
    'Canadá': (-141.00, -52.50, 41.50, 75.00, 45.42, -75.69),
    'Chile': (-75.90, -66.20, -56.00, -17.20, -33.45, -70.67),
    'Colombia': (-79.20, -66.80, -4.30, 13.55, 4.61, -74.08),
    'Costa Rica': (-86.20, -82.40, 7.70, 11.50, 9.93, -84.08),
    'Cuba': (-85.20, -74.00, 19.70, 23.30, 23.13, -82.38),
    'Dominica': (-61.53, -61.19, 15.18, 15.68, 15.30, -61.39),
    'Ecuador': (-81.20, -74.80, -5.10, 1.50, -0.23, -78.52),
    'El Salvador': (-90.15, -87.65, 13.10, 14.50, 13.69, -89.19),
    'Estados Unidos': (-125.00, -66.00, 24.00, 49.40, 38.91, -77.04),
    'Granada': (-61.85, -61.35, 11.95, 12.58, 12.05, -61.75),
    'Guatemala': (-92.30, -88.10, 13.70, 17.85, 14.63, -90.52),
    'Guyana': (-61.45, -56.45, 1.10, 8.60, 6.80, -58.16),
    'Haití': (-74.60, -71.60, 17.80, 20.10, 18.54, -72.34),
    'Honduras': (-89.40, -82.95, 12.95, 16.55, 14.08, -87.20),
    'Jamaica': (-78.45, -76.15, 17.65, 18.60, 17.97, -76.79),
    'México': (-118.50, -86.50, 14.50, 32.80, 19.43, -99.13),
    'Nicaragua': (-87.80, -82.60, 10.70, 15.10, 12.14, -86.25),
    'Panamá': (-83.15, -77.10, 7.10, 9.75, 8.98, -79.52),
    'Paraguay': (-62.70, -54.10, -27.65, -19.25, -25.28, -57.64),
    'Perú': (-81.45, -68.60, -18.40, -0.05, -12.05, -77.04),
    'República Dominicana': (-72.10, -68.25, 17.45, 20.10, 18.49, -69.93),
    'San Cristóbal y Nieves': (-62.92, -62.52, 17.08, 17.47, 17.30, -62.72),
    'San Vicente y las Granadinas': (-61.50, -61.05, 12.55, 13.42, 13.16, -61.23),
    'Santa Lucía': (-61.13, -60.83, 13.69, 14.12, 14.01, -60.99),
    'Surinam': (-58.10, -53.85, 1.80, 6.10, 5.85, -55.20),
    'Trinidad y Tobago': (-61.95, -60.45, 10.00, 11.45, 10.66, -61.51),
    'Uruguay': (-58.55, -53.00, -35.15, -30.05, -34.90, -56.16),
    'Venezuela': (-73.40, -59.70, 0.55, 12.25, 10.49, -66.88),
}
ENDVI_MACROREGIONS = {
    'Centroamérica': [-93.0, -77.0, 6.5, 19.0],
    'Centroamérica y Caribe': [-99.0, -59.0, 1.0, 27.0],
    'América del Sur': [-82.0, -34.0, -56.0, 13.0],
    'América del Norte (continental)': [-141.0, -52.5, 20.0, 75.0],
    'Zona del punto (±0.8°)': None,
    'Personalizada': None,
}
ENDVI_REGIONS = {**{name:list(spec[:4]) for name,spec in ENDVI_COUNTRIES.items()}, **ENDVI_MACROREGIONS}


def endvi_region_defaults(region, lat=None, lon=None):
    """Coordenada en tierra y límites de visualización para el país seleccionado.

    Usado por el callback Dropdown.change para actualizar seis controles.
    """
    if region in ENDVI_COUNTRIES:
        w, e, s, n, y, x = ENDVI_COUNTRIES[region]
        return float(y), float(x), float(w), float(e), float(s), float(n)
    if region == 'Zona del punto (±0.8°)':
        y, x = float(lat if lat is not None else 9.93), float(lon if lon is not None else -84.08)
        return y, x, x-.8, x+.8, y-.8, y+.8
    if region == 'Personalizada':
        y, x = float(lat if lat is not None else 9.93), float(lon if lon is not None else -84.08)
        return y, x, x-1., x+1., y-1., y+1.
    if region in ENDVI_MACROREGIONS:
        w,e,s,n=ENDVI_MACROREGIONS[region]
        centres={'Centroamérica':(13.7,-86.3),'Centroamérica y Caribe':(13.7,-86.3),
                 'América del Sur':(-12.,-66.),'América del Norte (continental)':(40.,-100.)}
        y,x=centres[region]
        return float(y), float(x), float(w), float(e), float(s), float(n)
    raise ValueError('País o región no encontrada.')


def endvi_climate_years(fecha, climate_mode, start=None, end=None):
    """Años de base fijos para TODOS los meses/décadas del gráfico.

    3/5/8/10...: últimos N años completos ANTERIORES al año de fecha.
    Personalizado: inicio-fin inclusive, con opción de inclusión del año actual.
    """
    selection = str(climate_mode)
    if selection in ('Sin climatología', 'No comparar', '0', 'None', ''):
        return []
    if selection == 'Personalizado':
        if start is None or end is None:
            raise ValueError('Especifique año inicial y final para climatología personalizada.')
        lo,hi = int(start),int(end)
    else:
        try: length=int(selection)
        except (TypeError, ValueError): raise ValueError('Seleccione 3, 5, 8, 10, 15 años o Personalizado.')
        if length < 3 or length > 30:
            raise ValueError('La climatología exige de 3 a 30 años; el archivo inicia en 2007.')
        lo,hi = fecha.year-length,fecha.year-1
    if not 2007 <= lo <= hi <= fecha.year or hi-lo+1 > 30:
        raise ValueError('Período climatológico inválido. Use años 2007 hasta el año elegido y un máximo de 30 años.')
    if hi-lo+1 < 3:
        raise ValueError('Se requieren al menos tres años para mostrar la climatología seleccionada.')
    return list(range(lo,hi+1))

ENDVI_OUTPUT_FOLDER = Path('/content/meteorologia_sinoptica_app/salidas/endvi10') if Path('/content').exists() else Path.cwd() / 'salidas_endvi10'


def endvi_fecha(year, month, decade):
    """Inicio exacto de la década: día 1, 11 o 21; sin fechas futuras inventadas."""
    y, m, d = int(year), int(month), int(decade)
    if not (2007 <= y <= 2100 and 1 <= m <= 12 and d in (1, 11, 21)):
        raise ValueError('Indique año >=2007, mes entre 1-12 y década 1, 11 o 21.')
    return datetime(y, m, d)


def endvi_periodo(fecha):
    final = 10 if fecha.day == 1 else (20 if fecha.day == 11 else calendar.monthrange(fecha.year, fecha.month)[1])
    return f'{fecha:%d/%m/%Y}–{final:02d}/{fecha:%m/%Y}'


def endvi_filename(fecha):
    return f'NETCDF4_LSASAF_AVHRR_ENDVI_GLOBE_{fecha:%Y%m%d}0000.nc'


def endvi_urls(fecha, user='', password=''):
    path = f'{fecha:%Y/%m/%d}/{endvi_filename(fecha)}'
    archive = f'{ENDVI_ROOT}/{path}'
    if user:
        # La forma user:pass@host corresponde al ejemplo oficial de LSA SAF.
        credentials = f'{quote(str(user), safe="")}:{quote(str(password), safe="")}@'
        dap = ENDVI_DAP.replace('https://', f'https://{credentials}', 1)
    else:
        dap = ENDVI_DAP
    return f'{dap}/{path}', archive


def endvi_dias_retro(fecha, n):
    """Últimos n inicios de década, en orden temporal, incluido fecha."""
    result = [fecha]
    for _ in range(n - 1):
        pivot = result[-1] - timedelta(days=1)
        dd = 21 if pivot.day >= 21 else (11 if pivot.day >= 11 else 1)
        result.append(datetime(pivot.year, pivot.month, dd))
    return list(reversed(result))


def endvi_recorte(region, left, right, bottom, top, lat, lon):
    if region == 'Zona del punto (±0.8°)':
        box = [lon - .8, lon + .8, lat - .8, lat + .8]
    elif region == 'Personalizada':
        box = [float(left), float(right), float(bottom), float(top)]
    elif region in ENDVI_REGIONS:
        box = list(ENDVI_REGIONS[region])
    else:
        raise ValueError('La región seleccionada no está definida.')
    w, e, s, n = box
    if not (-180 <= w < e <= 180 and -90 <= s < n <= 90):
        raise ValueError('Límites geográficos inválidos; use oeste < este y sur < norte.')
    if e - w > 110 or n - s > 75:
        raise ValueError('Reduzca el dominio para limitar las consultas OpenDAP.')
    return box


def endvi_coord(ds, axis):
    options = ('lat', 'latitude', 'Latitude', 'y') if axis == 'lat' else ('lon', 'longitude', 'Longitude', 'x')
    for name in options:
        if name in ds.coords and ds.coords[name].ndim == 1:
            return name
    for name in options:
        if name in ds.variables and ds[name].ndim == 1:
            return name
    raise ValueError(f'No se encontró una coordenada 1D de {axis}.')


def endvi_var(ds, key):
    for n in [key, key.lower(), key.upper(), 'ndvi' if key == 'NDV' else 'status_map']:
        if n in ds.data_vars:
            return ds[n]
    raise ValueError(f'El archivo no contiene la variable {key}. Variables: {list(ds.data_vars)}')


def endvi_un_time(var):
    # Compuestos NetCDF: normalmente una fecha por archivo.
    extras = [d for d in var.dims if d.lower() in ('time', 't')]
    return var.isel({d: 0 for d in extras}, drop=True) if extras else var


def endvi_extraer_ndvi(da):
    """Devuelve el NDVI físico: CF automáticamente o escala ENDVI10 documentada.

    Producto v2: NDV sin escalar V=0..250 => -0.08 + 0.004*V; 255 = no dato.
    No realiza doble escalado cuando xarray decodificó scale_factor/add_offset.
    """
    v = np.asarray(da.values, dtype='float32').copy()
    v[~np.isfinite(v)] = np.nan
    has_cf_decode = ('scale_factor' in da.encoding or 'add_offset' in da.encoding)
    a = da.attrs
    raw_range = np.asarray(a.get('valid_range', []), dtype='float64').reshape(-1)
    raw_flag = len(raw_range) == 2 and float(raw_range[1]) > 1.5
    if not has_cf_decode and (raw_flag or (np.any(np.isfinite(v)) and np.nanmax(v) > 1.01)):
        v[(v < 0) | (v > 250)] = np.nan
        v = ENDVI_PRODUCT_MIN + .004 * v
    # Si CF ya aplicó el factor, el rango es físico. Descartar flags/valores imposibles.
    v[(v < ENDVI_PRODUCT_MIN - 0.00001) | (v > ENDVI_PRODUCT_MAX + 0.00001)] = np.nan
    return v


def endvi_mascara_stm(stm, modo):
    """Interpretación Tabla 5, PUM ENDVIv2:
    bit7 Tierra, bit6 observación válida, bit4 AOT elevado,
    bit3 good/acceptable, bit2 nube/sombra, bit1 nube, bit0 nieve.
    """
    v = np.asarray(stm, dtype='float64')
    valid_num = np.isfinite(v) & (v >= 0) & (v <= 255)
    status = np.where(valid_num, v, 0).astype('uint16')
    on_land = (status & 128) != 0
    valid_obs = (status & 64) != 0
    mask = valid_num & on_land & valid_obs
    if modo == 'Estricto (cielo despejado, sin nieve ni AOT alto)':
        mask &= (status & (16 | 4 | 2 | 1)) == 0
    elif modo == 'Sin nube y sin nieve':
        mask &= (status & (4 | 2 | 1)) == 0
    elif modo == 'Solo observaciones válidas':
        pass
    else:
        raise ValueError('Modo de calidad desconocido.')
    return mask


def endvi_recortar_2d(ds, varname, box, max_axis=ENDVI_LIMIT_PIXELS_AXIS):
    latn, lonn = endvi_coord(ds, 'lat'), endvi_coord(ds, 'lon')
    latv, lonv = ds[latn].values, ds[lonn].values
    if latv.ndim != 1 or lonv.ndim != 1:
        raise ValueError('Se requiere una malla 1D geográfica regular.')
    west, east, south, north = box
    if float(lonv[0]) >= 0 and west < 0 and float(lonv[-1]) > 180:
        west = (west + 360) % 360
        east = (east + 360) % 360
    if west >= east:
        raise ValueError('La zona cruza la longitud 180° en esta cuadrícula; divídala en dos.')
    sel_lat = slice(north, south) if latv[0] > latv[-1] else slice(south, north)
    sel_lon = slice(west, east) if lonv[0] < lonv[-1] else slice(east, west)
    region = ds.sel({latn: sel_lat, lonn: sel_lon})
    ny, nx = region.sizes.get(latn, 0), region.sizes.get(lonn, 0)
    if ny == 0 or nx == 0:
        raise ValueError('La región no contiene píxeles del archivo NetCDF.')
    skip = max(1, math.ceil(max(ny, nx) / max_axis))
    reg = region.isel({latn: slice(None, None, skip), lonn: slice(None, None, skip)})
    a = endvi_un_time(endvi_var(reg, varname))
    if tuple(a.dims) != (latn, lonn):
        a = a.transpose(latn, lonn)
    return a, reg, skip


def endvi_map_data(ds, box, calidad):
    da, sub, step = endvi_recortar_2d(ds, 'NDV', box)
    latn, lonn = endvi_coord(sub, 'lat'), endvi_coord(sub, 'lon')
    ndvi = endvi_extraer_ndvi(da)
    if 'STM' in sub.data_vars:
        stm = endvi_un_time(sub['STM']).transpose(latn, lonn).values
        quality = endvi_mascara_stm(stm, calidad)
    else:
        raise ValueError('STM no está presente; no es posible garantizar el filtrado de nubosidad.')
    out = np.where(quality, ndvi, np.nan)
    return {'ndvi': out, 'lat': np.asarray(sub[latn].values), 'lon': np.asarray(sub[lonn].values),
            'step': step, 'n_total': ndvi.size, 'n_valid': int(np.isfinite(out).sum()),
            'percent_valid': 100*np.isfinite(out).mean(), 'units': 'adimensional'}


def endvi_punto_data(ds, lat, lon, pixeles, calidad):
    latn, lonn = endvi_coord(ds, 'lat'), endvi_coord(ds, 'lon')
    lats = np.asarray(ds[latn].values)
    lons = np.asarray(ds[lonn].values)
    xlon = (lon + 360) % 360 if lons.min() >= 0 and lon < 0 else lon
    i = int(np.argmin(np.abs(lats - lat)))
    j = int(np.argmin(np.abs(lons - xlon)))
    if abs(float(lats[i]) - lat) > .06 or abs(float(lons[j]) - xlon) > .06:
        raise ValueError('El punto queda fuera del dominio del producto.')
    r = int(pixeles) // 2
    selection = {latn: slice(max(0, i-r), min(len(lats), i+r+1)),
                 lonn: slice(max(0, j-r), min(len(lons), j+r+1))}
    v = endvi_un_time(endvi_var(ds.isel(selection), 'NDV')).transpose(latn, lonn)
    raw = endvi_extraer_ndvi(v)
    if 'STM' not in ds:
        raise ValueError('STM ausente: no se puede controlar la calidad en el punto.')
    stm = endvi_un_time(ds['STM'].isel(selection)).transpose(latn, lonn).values
    ok = np.isfinite(raw) & endvi_mascara_stm(stm, calidad)
    value = float(np.nanmean(raw[ok])) if np.any(ok) else float('nan')
    return {'value': value, 'count': int(ok.sum()), 'sample_size': int(raw.size),
            'lat': float(lats[i]), 'lon': float(lons[j]) if float(lons[j]) <= 180 else float(lons[j]-360)}


def endvi_figure_map(data, fecha, lat, lon, tipo, out):
    ndvi = data['ndvi']
    x, y = data['lon'], data['lat']
    # En ambos mapas el significado geográfico es el mismo; solo cambia la escala.
    if tipo == 'NDVI original':
        field = ndvi
        cmap, norm = NDVI_CMAP, NDVI_NORM
        clabel = 'NDVI físico (adimensional)'
        ticks = NDVI_FIXED_BOUNDS
        sub = 'Escala fija ENDVI10: −0.08 a 0.92; gris = sin dato o calidad insuficiente'
    else:
        field = np.clip((ndvi - ENDVI_PRODUCT_MIN)/(ENDVI_PRODUCT_MAX - ENDVI_PRODUCT_MIN), 0, 1)
        cmap = NDVI_CMAP
        norm = BoundaryNorm(np.linspace(0, 1, 8), cmap.N)
        clabel = 'NDVI reescalado al rango del producto [0,1]'
        ticks = np.linspace(0, 1, 6)
        sub = 'Escala lineal de presentación, no es índice de sequía ni humedad del suelo'
    fig, ax = plt.subplots(figsize=(11, 6.5), dpi=135)
    fig.patch.set_facecolor('white'); ax.set_facecolor('#eff2f1')
    img = ax.pcolormesh(x, y, np.ma.masked_invalid(field), shading='nearest',
                        cmap=cmap, norm=norm, rasterized=True)
    ax.plot([lon], [lat], marker='o', markersize=8, color='#ffffff',
            mec='#111111', mew=1.6, zorder=10)
    ax.set_title(f'ENDVI10 · {tipo} · {endvi_periodo(fecha)}', weight='bold', fontsize=15, pad=14)
    ax.set_xlabel('Longitud (°)'); ax.set_ylabel('Latitud (°)')
    ax.grid(alpha=.18)
    ax.set_aspect('auto')
    cb = fig.colorbar(img, ax=ax, pad=.02, shrink=.9, ticks=ticks)
    cb.set_label(clabel, fontsize=10)
    fig.text(.09, .035, sub, fontsize=9, color='#444444')
    fig.text(.91, .035, 'Fuente: EUMETSAT LSA SAF / AVHRR–Metop', fontsize=8.5, ha='right')
    fig.savefig(out, bbox_inches='tight', facecolor='white'); plt.close(fig)
    return str(out)


def endvi_figure_distribution(data, fecha, out):
    x = data['ndvi']; valid = x[np.isfinite(x)]
    if not valid.size:
        raise ValueError('No hay observaciones válidas para graficar la distribución.')
    bounds = [-.08, .2, .4, .6, .8, .92000001]
    tags = ['Muy bajo (< 0.20)', 'Bajo (0.20–0.40)', 'Moderado (0.40–0.60)',
            'Alto (0.60–0.80)', 'Muy alto (≥ 0.80)']
    colors = ['#b95634', '#e6a64b', '#f1df7b', '#58b666', '#217842']
    categories = np.clip(np.digitize(valid, bounds[1:-1]), 0, 4)
    counts = np.bincount(categories, minlength=5)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.4), dpi=135, gridspec_kw={'width_ratios':[1,1.2]})
    valid_pos = counts > 0
    ax.pie(counts[valid_pos], colors=np.asarray(colors)[valid_pos], startangle=90,
           wedgeprops={'width':.33, 'edgecolor':'white', 'linewidth':2})
    ax.text(0, .06, f'{np.nanmean(valid):.3f}', ha='center', va='center', fontsize=23, weight='bold')
    ax.text(0, -.21, 'NDVI medio', ha='center', fontsize=11)
    ax.set_title('Distribución de vigor espectral', fontsize=12, weight='bold')
    pct = 100*counts/valid.size
    ax2.barh(tags[::-1], pct[::-1], color=colors[::-1])
    for i, n in enumerate(pct[::-1]):
        ax2.text(n + 0.4, i, f'{n:.1f} %', va='center', fontsize=9)
    ax2.set_xlim(0, max(105, max(pct)*1.17)); ax2.set_xlabel('Porcentaje de píxeles válidos (%)')
    ax2.grid(axis='x', alpha=.15)
    fig.suptitle(f'ENDVI10 · Composición del área · {endvi_periodo(fecha)}', weight='bold', fontsize=15)
    fig.text(.5, .025, f'Datos válidos tras filtro STM: {data["percent_valid"]:.1f}% · Clasificación ilustrativa (no humedad del suelo)',
             ha='center', fontsize=9)
    fig.tight_layout(rect=[0, .05, 1, .95]); fig.savefig(out, bbox_inches='tight', facecolor='white'); plt.close(fig)
    return str(out), dict(zip(tags, pct.round(2).tolist()))


def endvi_climatology_for_series(rows, years, get_point):
    """Promedio/P10/P90 para CADA década del intervalo, con años de base fijos.

    Retorna histórico usado con fechas, incluso faltantes para auditoría.
    Nunca interpola, rellena ni calcula estadísticos con <3 años válidos.
    """
    keys = sorted({(r['date'].month, r['date'].day) for r in rows})
    clim = {}
    provenance = []
    for month, day in keys:
        values=[]
        for year in years:
            d=datetime(year,month,day)
            pt=get_point(d)
            v=float(pt['value'])
            provenance.append({'date':d, 'month':month, 'decade':day, 'year':year,
                               'ndvi':v, 'valid_pixels':int(pt['count'])})
            if np.isfinite(v): values.append(v)
        k=(month,day)
        n=len(values)
        if n>=3:
            a=np.asarray(values,dtype=float)
            clim[k]={'clim_mean':float(np.mean(a)), 'clim_median':float(np.median(a)),
                     'clim_p10':float(np.percentile(a,10)),
                     'clim_p90':float(np.percentile(a,90)),
                     'clim_min':float(np.min(a)), 'clim_max':float(np.max(a)),
                     'clim_std':float(np.std(a,ddof=1)), 'clim_n':n,
                     'clim_expected':len(years)}
        else:
            clim[k]={'clim_mean':float('nan'), 'clim_median':float('nan'),
                     'clim_p10':float('nan'),'clim_p90':float('nan'),
                     'clim_min':float('nan'),'clim_max':float('nan'),
                     'clim_std':float('nan'),'clim_n':n,'clim_expected':len(years)}
    for r in rows:
        c=clim[(r['date'].month,r['date'].day)]
        r.update(c)
        v=float(r['ndvi'])
        ave=c['clim_mean']
        r['anom']=(v-ave) if np.isfinite(v) and np.isfinite(ave) else float('nan')
        std=c['clim_std']
        # Sólo estimar desviaciones estandarizadas cuando >=5 años y varianza >0.
        r['anom_z']=(v-ave)/std if (c['clim_n']>=5 and np.isfinite(v) and
                                     np.isfinite(std) and std>1e-6) else float('nan')
    return clim, provenance


def endvi_coord_label(lat, lon):
    """Coordenadas en notación de hemisferio, sin longitud negativa E."""
    return f'{abs(lat):.4f}° {"N" if lat >= 0 else "S"}, {abs(lon):.4f}° {"E" if lon >= 0 else "O"}'


def _endvi_is_finite(rows, key):
    return np.asarray([r.get(key,float('nan')) for r in rows],dtype=float)


def endvi_figure_series(rows, lat, lon, out, reference_years=None):
    xs = [r['date'] for r in rows]
    fig,ax=plt.subplots(figsize=(13.4,6.3),dpi=135)
    ax.set_facecolor('white')
    observed=_endvi_is_finite(rows,'ndvi')
    climmean=_endvi_is_finite(rows,'clim_mean')
    p10=_endvi_is_finite(rows,'clim_p10')
    p90=_endvi_is_finite(rows,'clim_p90')
    if reference_years and np.any(np.isfinite(climmean)):
        ax.fill_between(xs,p10,p90,where=np.isfinite(p10)&np.isfinite(p90),
                        alpha=.20,color='#9865bd',label='Referencia histórica · P10–P90',
                        interpolate=False)
        ax.plot(xs,climmean,color='#79349e',lw=2.1,ls='--',marker='.',ms=4,
                label=f'Media histórica {reference_years[0]}–{reference_years[-1]}')
    ax.plot(xs,observed,color='#137e43',marker='o',ms=5,lw=2.2,
            label='ENDVI10 observado (control STM)',zorder=6)
    ax.set_ylim(-.10,.98)
    ax.set_ylabel('NDVI físico (adimensional)')
    ax.set_xlabel('Inicio de la década')
    ax.grid(alpha=.17)
    ax.set_title(f'ENDVI10 · Serie temporal y climatología · {endvi_coord_label(lat,lon)}',
                 fontsize=14,weight='bold')
    if len(xs)>6: fig.autofmt_xdate(rotation=28)
    ax.legend(loc='best',fontsize=9)
    if reference_years:
        caption=f'Base: {len(reference_years)} años seleccionados; se necesitan ≥3 válidos POR década. P10–P90 no es intervalo de confianza.'
    else:
        caption='Sin climatología seleccionada. Se respetan observaciones faltantes; no hay interpolación.'
    fig.text(.5,.005,caption,ha='center',fontsize=8.7,color='#444444')
    fig.tight_layout(rect=[0,.045,1,1])
    fig.savefig(out,bbox_inches='tight',facecolor='white');plt.close(fig)
    return str(out)


def endvi_figure_series_normalized(rows, lat, lon, out, reference_years=None):
    xs=[r['date'] for r in rows]
    observed=_endvi_is_finite(rows,'ndvi_01')
    avg=np.clip(_endvi_is_finite(rows,'clim_mean') - ENDVI_PRODUCT_MIN,0,1)
    p10=np.clip(_endvi_is_finite(rows,'clim_p10') - ENDVI_PRODUCT_MIN,0,1)
    p90=np.clip(_endvi_is_finite(rows,'clim_p90') - ENDVI_PRODUCT_MIN,0,1)
    fig,ax=plt.subplots(figsize=(13.4,5.5),dpi=135)
    if reference_years and np.any(np.isfinite(avg)):
        ax.fill_between(xs,p10,p90,color='#9477c3',alpha=.19,label='P10–P90 histórico')
        ax.plot(xs,avg,color='#79349e',lw=2,ls='--',label='Media histórica normalizada')
    ax.plot(xs,observed,color='#21865f',marker='o',ms=4.5,lw=2.1,label='NDVI observado normalizado')
    ax.set_ylim(0,1.02);ax.set_yticks(np.linspace(0,1,11))
    ax.set_xlabel('Inicio de la década')
    ax.set_ylabel('NDVI escalado linealmente a [0, 1]')
    ax.set_title(f'ENDVI10 · NDVI reescalado (0–1) · {endvi_coord_label(lat,lon)}',weight='bold')
    ax.grid(alpha=.18);ax.legend(loc='best',fontsize=9)
    if len(xs)>6:fig.autofmt_xdate(rotation=28)
    fig.text(.5,.005,'0–1 = (NDVI físico + 0.08)/1.00. NO es anomalía, VCI ni porcentaje de humedad.',
             ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.05,1,1]);fig.savefig(out,bbox_inches='tight',facecolor='white');plt.close(fig)
    return str(out)


def endvi_figure_anomalies(rows, lat, lon, out, reference_years):
    xs=[r['date'] for r in rows]
    values=_endvi_is_finite(rows,'anom')
    fig,ax=plt.subplots(figsize=(13.4,5.4),dpi=135)
    valid=np.isfinite(values)
    if np.any(valid):
        cols=np.where(values[valid]>=0,'#21855e','#bd6147')
        ax.bar(np.asarray(xs,dtype=object)[valid],values[valid],width=7,color=cols,
               alpha=.9,label='Observado − media histórica')
    ax.axhline(0,color='#323232',lw=1)
    ax.set_xlabel('Inicio de la década');ax.set_ylabel('Anomalía absoluta NDVI (adimensional)')
    ax.set_title(f'Anomalías decádicas ENDVI10 frente a {reference_years[0]}–{reference_years[-1]}',weight='bold')
    ax.grid(axis='y',alpha=.18)
    ax.set_xlim(min(xs)-timedelta(days=6),max(xs)+timedelta(days=6))
    if len(xs)>6:fig.autofmt_xdate(rotation=28)
    fig.text(.5,.005,'Anomalía = observado − media de la MISMA década del calendario. Vacíos = sin referencia/datos.',
             ha='center',fontsize=9)
    fig.tight_layout(rect=[0,.05,1,1]);fig.savefig(out,bbox_inches='tight',facecolor='white');plt.close(fig)
    return str(out)


def _endvi_fecha_desde_ds(ds, filepath, fallback=None):
    if 'time' in ds.coords:
        try:
            date = np.datetime64(ds['time'].values[0] if ds['time'].ndim else ds['time'].values, 'D')
            return datetime.strptime(str(date), '%Y-%m-%d')
        except (ValueError, TypeError, IndexError):
            pass
    for attr in ('time_coverage_start', 'image_reference_time'):
        val = ds.attrs.get(attr)
        if val:
            try: return datetime.fromisoformat(str(val)[:10])
            except ValueError: pass
    mm = re.search(r'(20\d{6})', Path(str(filepath)).name)
    if mm:
        try: return datetime.strptime(mm.group(1), '%Y%m%d')
        except ValueError: pass
    return fallback


def run_endvi10(year, month, decade, point_lat, point_lon, region, west, east, south, north,
                quality_mode, point_pixels, n_dekads, climate_mode, climate_start, climate_end,
                username, password, uploaded_files):
    """Consulta geográfica y climatología POR CADA década, con trazabilidad CSV.

    Nunca descarga el NetCDF global: usa cortes OPeNDAP. En modo local sólo
    se incluyen las fechas realmente cargadas, sin sustituir faltantes.
    """
    folder=ENDVI_OUTPUT_FOLDER / uuid4().hex[:10]
    try:
        fecha=endvi_fecha(year,month,decade)
        lat,lon=float(point_lat),float(point_lon)
        if not -56 <= lat <= 75 or not -180 <= lon <= 180:
            raise ValueError('ENDVI10 cubre aproximadamente 56°S a 75°N; ajuste el punto.')
        pixels=int(point_pixels)
        if pixels not in (1,3,5):raise ValueError('Seleccione 1, 3 o 5 píxeles de muestreo.')
        box=endvi_recorte(region,west,east,south,north,lat,lon)
        if not (box[0]<=lon<=box[1] and box[2]<=lat<=box[3]):
            raise ValueError('El punto está fuera de los límites del mapa; elija nuevamente el país o ajuste el punto.')
        n=int(n_dekads)
        if n not in (1,3,6,9,18,36,72,108,180):
            raise ValueError('Período temporal no admitido (1–180 décadas).')
        dates=endvi_dias_retro(fecha,n)
        years=endvi_climate_years(fecha,climate_mode,climate_start,climate_end)
        folder.mkdir(parents=True,exist_ok=True)
        notices=[]; gallery=[]; archive_files=[]
        if region in ENDVI_COUNTRIES:
            notices.append(f'País: {region}; límites aproximados para visualizar (no frontera administrativa).')
        notices.append(f'Fecha elegida: {endvi_periodo(fecha)} · Punto: {lat:.4f}, {lon:.4f}; muestreo {pixels}×{pixels}.')
        notices.append(f'Período observado: {dates[0]:%Y-%m-%d} a {dates[-1]:%Y-%m-%d} ({len(dates)} décadas).')
        if years:
            notices.append(f'Climatología seleccionada: {years[0]}–{years[-1]} ({len(years)} años); media/P10/P90 para TODAS las décadas de la serie.')
            notices.append('Se exige al menos tres años válidos para cada década (no interpolar).')
        else:
            notices.append('Climatología desactivada.')
        local=uploaded_files or []
        if isinstance(local,(str,Path)):local=[local]
        local=[str(getattr(f,'name',f)) for f in local if f]
        filemap={}
        for f in local:
            if not Path(f).is_file():
                notices.append(f'Archivo local no encontrado: {Path(f).name}.')
                continue
            try:
                # netCDF4 es el motor definido en Colab y el del producto original.
                with xr.open_dataset(f,engine='netcdf4',mask_and_scale=True) as ds:
                    dt=_endvi_fecha_desde_ds(ds,f)
                if dt is None:
                    notices.append(f'Sin fecha reconocible: {Path(f).name}.')
                else:
                    k=dt.replace(hour=0,minute=0,second=0,microsecond=0)
                    if k not in filemap:filemap[k]=f
            except Exception as err:
                notices.append(f'Error leyendo {Path(f).name}: {type(err).__name__}.')
        if local and not filemap:
            raise ValueError('Ningún NetCDF local válido con fecha reconocible.')
        if filemap:
            notices.append(f'Modo local: {len(filemap)} archivos datados; NO se consultan fechas ausentes en línea.')
            if fecha not in filemap:
                raise ValueError(f'No se cargó NetCDF para {fecha:%Y-%m-%d}; selecciónelo o cambie la fecha.')
        else:
            notices.append('Modo OpenDAP: únicamente lee los píxeles solicitados; no descarga el planeta completo.')

        def abrir(d):
            if filemap:
                return xr.open_dataset(filemap[d],engine='netcdf4',mask_and_scale=True)
            dap,_=endvi_urls(d,username or '',password or '')
            return xr.open_dataset(dap,engine='netcdf4',mask_and_scale=True)

        with abrir(fecha) as ds:
            mapdata=endvi_map_data(ds,box,quality_mode)
            current=endvi_punto_data(ds,lat,lon,pixels,quality_mode)
        if not mapdata['n_valid']:
            raise ValueError('No hay píxeles válidos en el mapa con el filtro STM elegido.')
        stem=f'endvi10_{fecha:%Y%m%d}'
        for label,slug in [('NDVI original','ndvi'),('NDVI reescalado 0-1','normalizado')]:
            path=folder / f'{stem}_{slug}.png'
            endvi_figure_map(mapdata,fecha,lat,lon,label,path)
            gallery.append((str(path),label));archive_files.append(path)
        path=folder / f'{stem}_distribucion.png'
        endvi_figure_distribution(mapdata,fecha,path)
        gallery.append((str(path),'Distribución espacial'));archive_files.append(path)
        notices.append(f'Mapa: {mapdata["n_valid"]:,}/{mapdata["n_total"]:,} píxeles válidos ({mapdata["percent_valid"]:.1f}%), submuestreo {mapdata["step"]}.')
        if np.isfinite(current['value']):
            notices.append(f'NDVI punto ({fecha:%Y-%m-%d}): {current["value"]:.3f}.')
        else:notices.append('El punto seleccionado no tiene observaciones válidas en esa década.')

        # Cache de esta ejecución: misma fecha sirve para series y climatología.
        cache={fecha:current}
        failures={}
        def fetch_point(d):
            if d in cache:return cache[d]
            empty={'value':float('nan'),'count':0,'sample_size':pixels*pixels}
            if filemap and d not in filemap:
                failures[d]='No se proporcionó archivo local'
                cache[d]=empty
                return empty
            try:
                with abrir(d) as ds:
                    value=endvi_punto_data(ds,lat,lon,pixels,quality_mode)
            except Exception as err:
                failures[d]=type(err).__name__
                value=empty
            cache[d]=value
            return value

        rows=[]
        for d in dates:
            pt=fetch_point(d)
            value=float(pt['value'])
            rows.append({'date':d,'ndvi':value,'valid_pixels':int(pt['count']),
                         'total_pixels':int(pt['sample_size']),
                         'ndvi_01':float(np.clip(value-ENDVI_PRODUCT_MIN,0,1)) if np.isfinite(value) else float('nan')})

        clim={};prov=[]
        if years:
            clim,prov=endvi_climatology_for_series(rows,years,fetch_point)
            eligible=sum(np.isfinite(r['clim_mean']) for r in rows)
            notices.append(f'Comparación histórica: {eligible}/{len(rows)} décadas tienen al menos 3 años válidos de referencia.')
            if not eligible:notices.append('Sin datos climatológicos suficientes: no se dibujan medias ni bandas históricas ficticias.')
        else:
            for r in rows:
                r.update({'clim_mean':float('nan'),'clim_median':float('nan'),
                          'clim_p10':float('nan'),'clim_p90':float('nan'),
                          'clim_min':float('nan'),'clim_max':float('nan'),
                          'clim_std':float('nan'),'clim_n':0,'clim_expected':0,
                          'anom':float('nan'),'anom_z':float('nan')})

        for slug,label,func in [
                ('serie_vs_climatologia','Serie observada vs. media y P10–P90',endvi_figure_series),
                ('serie_normalizada','Serie NDVI normalizada vs. climatología',endvi_figure_series_normalized)]:
            path=folder / f'{stem}_{slug}.png'
            func(rows,lat,lon,path,reference_years=years)
            gallery.append((str(path),label));archive_files.append(path)
        if years and np.isfinite(_endvi_is_finite(rows,'anom')).any():
            path=folder / f'{stem}_anomalias.png'
            endvi_figure_anomalies(rows,lat,lon,path,years)
            gallery.append((str(path),'Anomalía NDVI por década'));archive_files.append(path)

        def csvval(v,digits=5):
            if v is None:return ''
            try:
                return f'{float(v):.{digits}f}' if np.isfinite(float(v)) else ''
            except (TypeError,ValueError):return ''
        csv_path=folder / f'{stem}_serie_climatologia.csv'
        fields=['fecha_inicio_utc','mes','decada','latitud','longitud','ndvi_observado',
                'ndvi_normalizado_0_1','media_historica','mediana_historica',
                'p10_historico','p90_historico','desv_std_historica','min_historico','max_historico',
                'anomalia_ndvi','anomalia_estandarizada_z','anios_validos','anios_esperados',
                'pixeles_validos','pixeles_totales']
        with csv_path.open('w',newline='',encoding='utf-8-sig') as file:
            writer=csv.writer(file);writer.writerow(fields)
            for r in rows:
                d=r['date']
                writer.writerow([d.strftime('%Y-%m-%d'),d.month,d.day,
                    csvval(lat),csvval(lon),csvval(r['ndvi']),csvval(r['ndvi_01']),
                    csvval(r['clim_mean']),csvval(r['clim_median']),csvval(r['clim_p10']),
                    csvval(r['clim_p90']),csvval(r['clim_std']),csvval(r['clim_min']),
                    csvval(r['clim_max']),csvval(r['anom']),csvval(r['anom_z']),
                    r['clim_n'],r['clim_expected'],r['valid_pixels'],r['total_pixels']])
        archive_files.append(csv_path)
        if years:
            histfile=folder / f'{stem}_detalle_datos_historicos.csv'
            with histfile.open('w',newline='',encoding='utf-8-sig') as file:
                writer=csv.writer(file)
                writer.writerow(['fecha_historia','anio','mes','decada','ndvi_historico','pixeles_validos'])
                for r in prov:
                    writer.writerow([r['date'].strftime('%Y-%m-%d'),r['year'],r['month'],
                                     r['decade'],csvval(r['ndvi']),r['valid_pixels']])
            archive_files.append(histfile)
        nvalid=sum(np.isfinite(r['ndvi']) for r in rows)
        notices.append(f'Serie observada: {nvalid}/{len(rows)} décadas válidas. Faltantes = NaN, sin interpolar.')
        if failures:
            sample=', '.join(f'{d:%Y-%m-%d} ({reason})' for d,reason in list(sorted(failures.items()))[:8])
            notices.append(f'Archivos/datos ausentes: {len(failures)}; ejemplos: {sample}.')
        notesfile=folder/'METODOLOGIA_Y_FUENTES.txt'
        notesfile.write_text(
            'ENDVI10 LSA SAF — Metodología, escala y climatología\n'
            f'Fecha de producto: {fecha:%Y-%m-%d}; período observado {dates[0]:%Y-%m-%d}–{dates[-1]:%Y-%m-%d}.\n'
            f'País/región: {region}. Región rectangular aproximada {box}.\n'
            f'Punto: {lat},{lon}, promedio {pixels}x{pixels} píxeles. Filtro STM: {quality_mode}.\n'
            f'Base climatológica elegida: {years[0] if years else "Sin"}–{years[-1] if years else "Sin"} ({len(years)} años).\n'
            'Cada fecha observada compara la misma década del calendario (mes y día 1, 11 o 21)\n'
            'contra observaciones de los mismos meses/décadas de TODOS los años de base.\n'
            'Se requieren mínimo 3 años válidos por década para graficar media, P10 y P90.\n'
            'El año base de los preajustes N años es [año de la fecha - N, año de la fecha - 1].\n'
            'No hay relleno temporal, interpolación, ni equivalencia con la climatología de 30 años WMO.\n'
            'Climatología 3/5/8/10 años = referencia histórica parcial definida por usuario.\n'
            'NDVI físico válido: -0.08..0.92; si V=0..250, NDVI = -0.08 + 0.004*V.\n'
            'NDVI reescalado 0..1: (NDVI físico + 0.08)/1.00, no indicador de agua.\n'
            'Anomalía = NDVI observado - media de la misma década; z solo si >=5 años y desvío > 0.\n'
            'STM: tierra y observación válida, excluye nube/nieve según filtro configurado.\n'
            'ENDVI10 utiliza corrección atmosférica propia; no se reitera artificialmente.\n'
            'P10–P90 son percentiles empíricos (no intervalo de confianza).\n'
            'Una serie de NDVI NO demuestra por sí sola estrés hídrico ni necesidad de riego.\n'
            f'Fuente: {ENDVI_ROOT}/{fecha:%Y/%m/%d}/\n'
            'PUM: https://lsa-saf.eumetsat.int/en/data/products/vegetation/\n',
            encoding='utf-8')
        archive_files.append(notesfile)
        zip_path=folder / f'ENDVI10_{fecha:%Y%m%d}_serie_y_climatologia.zip'
        with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
            for path in archive_files:zf.write(path,arcname=path.name)
        notices.append('Referencia corta (3/5/8/10 años) no equivale a normal climatológica estándar de 30 años.')
        notices.append('NDVI no mide directamente disponibilidad hídrica ni requerimiento de riego.')
        return gallery,str(zip_path),'\n'.join(notices)
    except Exception as exc:
        msg=str(exc)
        if (username and str(username) in msg) or (password and str(password) in msg):
            msg='No se pudo abrir el servidor OpenDAP. Revise cuenta, permisos y conexión.'
        if any(x in msg.lower() for x in ('https://','thredds.','credentials')):
            msg='No se pudo leer el NetCDF remoto. Compruebe fecha publicada, cuenta LSA SAF y conexión.'
        return [],None,f'ENDVI10 — {type(exc).__name__}: {msg}\nLas otras secciones del visor permanecen intactas.'


In [ ]:
# ============================================================================
# Interfaz Gradio para Colab
# ============================================================================


def _make_dashboard_previews() -> dict[str, tuple[str, str]]:
    """Crea tarjetas esquemáticas; no representan datos meteorológicos reales."""
    preview_dir = APP_DIR / "vistas_previas"
    preview_dir.mkdir(parents=True, exist_ok=True)
    x = np.linspace(-3.0, 3.0, 150)
    y = np.linspace(-1.25, 1.25, 76)
    xx, yy = np.meshgrid(x, y)
    base = np.sin(xx * 1.1) + 0.7 * np.cos(yy * 2.1) + 0.25 * np.sin(xx + yy)
    outputs = {}
    definitions = [
        ("fronts", "GFS · análisis frontal", "RdYlBu_r"),
        ("gfs_barbs", "GFS · barbas a 850 hPa", "GnBu"),
        ("dmw", "GOES B13 · barbas DMW por presión", "gray"),
        ("geocolor", "GOES · imagen satelital", "viridis"),
    ]
    for key, title, cmap in definitions:
        path = preview_dir / f"preview_{key}.png"
        if path.exists():
            outputs[key] = (str(path), f"{title} · vista esquemática")
            continue
        fig, ax = plt.subplots(figsize=(6.4, 2.6), dpi=95)
        ax.set_facecolor("#0b1f34")
        field = base.copy()
        if key == "dmw":
            field = np.exp(-((xx + 0.5) ** 2 / 1.8 + (yy - 0.15) ** 2 / 0.25))
            field += 0.8 * np.exp(-((xx - 1.5) ** 2 / 0.7 + (yy + 0.25) ** 2 / 0.18))
        elif key == "geocolor":
            field = 0.5 + 0.5 * np.sin(xx * 1.5) * np.cos(yy * 2.0)
        ax.imshow(field, extent=[x.min(), x.max(), y.min(), y.max()],
                  origin="lower", cmap=cmap, aspect="auto", alpha=0.96)
        ax.contour(xx, yy, base, levels=8, colors="#f0f7ff", linewidths=0.5, alpha=0.45)
        if key == "fronts":
            line_x = np.linspace(-2.8, 2.8, 25)
            line_y = 0.32 * np.sin(line_x * 1.1) - 0.05
            ax.plot(line_x, line_y, color="#42d2ff", linewidth=3.0)
            ax.scatter(line_x[::2], line_y[::2], marker="v", s=28,
                       color="#42d2ff", edgecolors="white", linewidths=0.35, zorder=5)
        elif key in {"gfs_barbs", "dmw"}:
            bx, by = np.meshgrid(np.linspace(-2.6, 2.6, 10), np.linspace(-0.9, 0.9, 5))
            uu = 7.0 + 1.8 * np.sin(by * 2)
            vv = 2.0 * np.cos(bx)
            if key == "gfs_barbs":
                ax.barbs(bx, by, uu, vv, length=5.2, linewidth=0.65,
                         color="#f5fbff", zorder=5)
            else:
                colors = ["#3155d9", "#27a8e0", "#13a66b", "#d4bd14", "#ed6b32", "#d43d8f"]
                for i, color in enumerate(colors):
                    rows = slice(i % 5, None, 5)
                    ax.barbs(bx[rows], by[rows], uu[rows], vv[rows],
                             length=5.0, linewidth=0.7, color=color, zorder=5)
        ax.text(0.035, 0.88, title, transform=ax.transAxes, color="white",
                fontsize=11, fontweight="bold",
                bbox={"facecolor": "#071a2c", "alpha": 0.82, "pad": 5})
        ax.text(0.035, 0.06, "VISTA PREVIA · ESQUEMA, NO SON DATOS ACTUALES",
                transform=ax.transAxes, color="white", fontsize=7.2,
                bbox={"facecolor": "#071a2c", "alpha": 0.78, "pad": 3})
        ax.set_xlim(-3, 3)
        ax.set_ylim(-1.25, 1.25)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color("#dcecf6")
            spine.set_linewidth(0.6)
        fig.savefig(path, bbox_inches="tight", facecolor="#0b1f34")
        plt.close(fig)
        outputs[key] = (str(path), f"{title} · vista esquemática")
    return outputs


def _gallery_index(event) -> int:
    index = getattr(event, "index", 0)
    if isinstance(index, (tuple, list)):
        index = index[-1] if index else 0
    return int(index)


def _region_controls(gr, prefix: str, default_region: str = "Centroamérica y Caribe"):
    default_extent = REGIONS.get(default_region, REGIONS["Centroamérica y Caribe"])
    with gr.Accordion("Área de interés", open=False, elem_classes=["panel"]):
        region = gr.Dropdown(list(REGIONS), value=default_region, label="Región")
        with gr.Row():
            min_lon = gr.Number(value=default_extent[0], label="Longitud oeste", precision=2)
            max_lon = gr.Number(value=default_extent[1], label="Longitud este", precision=2)
            min_lat = gr.Number(value=default_extent[2], label="Latitud sur", precision=2)
            max_lat = gr.Number(value=default_extent[3], label="Latitud norte", precision=2)
        gr.Markdown("Los límites se usan al seleccionar **Personalizada**. Fechas y horas en UTC.")
    return region, min_lon, max_lon, min_lat, max_lat


def build_app():
    import gradio as gr

    nwp_groups = {
        "Frentes fríos": ["gfs_fram"],
        "Presión y temperatura": [
            "gfs_prmsl", "gfs_thickness", "gfs_delta_t", "gfs_td1000",
            "ecmwf_msl", "ecmwf_t2m",
        ],
        "Humedad y convección": [
            "gfs_pwat", "gfs_rh925", "gfs_gdi", "gfs_mfc", "ecmwf_pwat",
        ],
        "Viento y dinámica": [
            "gfs_wind500", "gfs_wind925", "gfs_wind850", "gfs_wind_mean",
            "gfs_shear", "gfs_vort925", "gfs_div925", "gfs_divmean",
            "ecmwf_wind10",
        ],
    }
    satellite_groups = {
        "Canales e imágenes GOES": [
            "abi_b13", "abi_b08", "abi_b09", "abi_b10", "abi_b02",
            "sandwich", "geocolor", "rgb_day", "rgb_day_glm", "rgb_night",
        ],
        "Agua atmosférica": ["mimic_tpw", "alpw"],
        "Rayos e incendios": ["glm_instant", "glm_accum", "fires"],
        "Viento satelital": ["dmw"],
    }

    def nwp_options(keys):
        return [(NWP_LABELS[key], key) for key in keys]

    def satellite_options(keys):
        return [(SATELLITE_LABELS[key], key) for key in keys]

    def selection_group(*args, **kwargs):
        # No dependemos de la inferencia automática de interactividad de Gradio.
        kwargs.setdefault("interactive", True)
        kwargs.setdefault("type", "value")
        return gr.CheckboxGroup(*args, **kwargs)

    def selected_from_groups(*groups):
        return list(dict.fromkeys(item for group in groups for item in (group or [])))

    def run_forecast_ui(
        date_text, cycle, fxx, models, fronts, thermodynamics, moisture,
        dynamics, region, min_lon, max_lon, min_lat, max_lat,
        include_sources, auto_ecmwf,
    ):
        selected_maps = selected_from_groups(fronts, thermodynamics, moisture, dynamics)
        selected_models = list(dict.fromkeys(models or []))
        return run_forecast(
            date_text, cycle, fxx, selected_models, selected_maps, region,
            min_lon, max_lon, min_lat, max_lat, include_sources, auto_ecmwf,
        )

    def run_diagnostic_ui(
        time_mode, start, end, hours_back, cadence, satellite, scope,
        channels, water, lightning, winds, region, min_lon, max_lon,
        min_lat, max_lat, include_sources, high_resolution,
    ):
        selected_maps = selected_from_groups(channels, water, lightning, winds)
        return run_diagnostic(
            time_mode, start, end, hours_back, cadence, satellite, scope,
            selected_maps, region, min_lon, max_lon, min_lat, max_lat,
            include_sources, high_resolution,
        )

    def run_history_ui(
        start, end, cadence, fxx, models, nwp_fronts, nwp_thermo,
        nwp_moisture, nwp_dynamics, sat_channels, sat_water, sat_lightning,
        sat_wind, satellite, scope, region, min_lon, max_lon, min_lat,
        max_lat, output_mode, high_resolution,
    ):
        selected_nwp = selected_from_groups(
            nwp_fronts, nwp_thermo, nwp_moisture, nwp_dynamics
        )
        selected_sat = selected_from_groups(
            sat_channels, sat_water, sat_lightning, sat_wind
        )
        selected_models = list(dict.fromkeys(models or []))
        return run_reanalysis(
            start, end, cadence, fxx, selected_models, selected_nwp,
            satellite, scope, selected_sat, region, min_lon, max_lon,
            min_lat, max_lat, output_mode, high_resolution,
        )

    now = utc_now_naive()
    today = now.strftime("%Y-%m-%d")
    now_text = now.strftime("%Y-%m-%d %H:%M")
    recent_text = (now - timedelta(hours=6)).strftime("%Y-%m-%d %H:%M")

    with gr.Blocks(title="Meteorología sinóptica · Centroamérica") as demo:
        gr.Markdown(
            "<p class='eyebrow'>CENTRO DE OPERACIONES · CENTROAMÉRICA</p>"
            "# Meteorología sinóptica\n"
            "GFS · ECMWF · GOES · ASCAT · ENDVI10 — mapas, diagnóstico, viento y vegetación",
            elem_classes=["app-header"],
        )

        with gr.Row(elem_classes=["nav-bar"]):
            nav_home = gr.Button("⌂  Inicio", variant="primary", elem_classes=["nav-item"])
            nav_forecast = gr.Button("↗  Pronóstico", variant="secondary", elem_classes=["nav-item"])
            nav_satellite = gr.Button("◉  Satélite y viento", variant="secondary", elem_classes=["nav-item"])
            nav_ascat = gr.Button("≋  ASCAT", variant="secondary", elem_classes=["nav-item"])
            nav_history = gr.Button("◷  Histórico", variant="secondary", elem_classes=["nav-item"])
            nav_eumetsat = gr.Button("🇪🇺  EUMETSAT", variant="secondary", elem_classes=["nav-item"])
            nav_endvi = gr.Button("🌱  NDVI · ENDVI10", variant="secondary", elem_classes=["nav-item"])

        with gr.Column(visible=True, elem_classes=["page-shell"]) as home_page:
            gr.Markdown(
                "## Panel de trabajo\n"
                "Elija una sección. Los datos se descargan cuando pulsa **Generar mapas**.",
                elem_classes=["page-heading"],
            )
            with gr.Row(elem_classes=["dashboard-grid"]):
                home_forecast = gr.Button(
                    "📈  Pronóstico numérico\nGFS, ECMWF y Frentes fríos",
                    elem_classes=["dashboard-card"],
                )
                home_satellite = gr.Button(
                    "🛰️  Satélite y viento\nGOES, GeoColor, DMW y rayos",
                    elem_classes=["dashboard-card"],
                )
            with gr.Row(elem_classes=["dashboard-grid"]):
                home_ascat = gr.Button(
                    "🌬️  Viento oceánico\nASCAT reciente e histórico NOAA",
                    elem_classes=["dashboard-card"],
                )
                home_history = gr.Button(
                    "🗂️  Histórico y reanálisis\nProductos organizados por tipo",
                    elem_classes=["dashboard-card"],
                )
            with gr.Row(elem_classes=["dashboard-grid"]):
                home_eumetsat = gr.Button(
                    "🇪🇺  Satélites europeos\nEUMETSAT · MTG · Metop · Sentinel-3",
                    elem_classes=["dashboard-card"],
                )
            with gr.Row(elem_classes=["dashboard-grid"]):
                home_endvi = gr.Button(
                    "🌿  Vigor vegetativo · ENDVI10\nNDVI, nubes, mapas y series decádicas LSA SAF",
                    elem_classes=["dashboard-card"],
                )
            gr.Markdown(
                "**Área inicial:** Centroamérica y Caribe · **Zona horaria:** UTC · "
                "**Visor público:** la última celda imprime el enlace cuando Colab inicia el túnel."
            )

        with gr.Column(visible=False, elem_classes=["page-shell"]) as forecast_page:
            gr.Markdown(
                "## Pronóstico numérico\n"
                "Seleccione fecha, ciclo y plazos. 21 productos numéricos disponibles; todas las variables aparecen en tarjetas agrupadas por tipo.",
                elem_classes=["page-heading"],
            )
            with gr.Row(elem_classes=["panel"]):
                forecast_date = gr.DateTime(
                    value=today, include_time=False, type="string", timezone="UTC",
                    label="Fecha de corrida",
                )
                forecast_cycle = gr.Dropdown(
                    ["00", "06", "12", "18"], value="00", label="Ciclo (UTC)"
                )
                forecast_fxx = selection_group(
                    [(f"+{h} h", str(h)) for h in (0, 6, 12, 18, 24, 36, 48, 72, 96, 120)],
                    value=["6"], label="Plazos de pronóstico",
                    elem_classes=["choice-cards"],
                    info="Puede marcar varios plazos; el mapa se genera para cada uno.",
                )
            with gr.Row(elem_classes=["panel"]):
                forecast_models = selection_group(
                    MODEL_CHOICES, value=["gfs050"],
                    label="Modelos numéricos · marque uno o varios",
                    elem_classes=["choice-cards"],
                    info="FRAM usa GFS 0.50°. Los demás mapas requieren GFS o ECMWF según su etiqueta.",
                )
                forecast_auto_ecmwf = gr.Checkbox(
                    value=True,
                    label="ECMWF automático: usar la última corrida disponible",
                )

            with gr.Accordion("Frentes fríos", open=True, elem_classes=["product-group"]):
                forecast_fronts = selection_group(
                    nwp_options(nwp_groups["Frentes fríos"]),
                    value=[], label="Producto FRAM", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Presión y temperatura", open=True, elem_classes=["product-group"]):
                forecast_thermo = selection_group(
                    nwp_options(nwp_groups["Presión y temperatura"]), value=[],
                    label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Humedad y convección", open=True, elem_classes=["product-group"]):
                forecast_moisture = selection_group(
                    nwp_options(nwp_groups["Humedad y convección"]), value=[],
                    label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Viento y dinámica", open=True, elem_classes=["product-group"]):
                forecast_dynamics = selection_group(
                    nwp_options(nwp_groups["Viento y dinámica"]),
                    value=[], label="Productos", elem_classes=["product-tiles"],
                )

            forecast_region, forecast_min_lon, forecast_max_lon, forecast_min_lat, forecast_max_lat = _region_controls(gr, "forecast", "Frentes fríos — dominio de referencia")
            with gr.Accordion("Opciones avanzadas", open=False, elem_classes=["panel"]):
                forecast_sources = gr.Checkbox(
                    value=False, label="Incluir archivos fuente en el ZIP (pueden ser grandes)"
                )
            forecast_button = gr.Button("Generar mapas", variant="primary")
            forecast_gallery = gr.Gallery(
                label="Resultados · agrupados por producto", columns=2,
                height="auto", object_fit="contain", elem_classes=["output-gallery"],
            )
            forecast_zip = gr.File(label="Descargar ZIP de resultados")
            with gr.Accordion("Registro técnico", open=False, elem_classes=["panel"]):
                forecast_log = gr.Textbox(label="Ejecución", lines=14, max_lines=28)
            forecast_button.click(
                run_forecast_ui,
                inputs=[
                    forecast_date, forecast_cycle, forecast_fxx, forecast_models,
                    forecast_fronts, forecast_thermo, forecast_moisture,
                    forecast_dynamics, forecast_region, forecast_min_lon,
                    forecast_max_lon, forecast_min_lat, forecast_max_lat,
                    forecast_sources, forecast_auto_ecmwf,
                ],
                outputs=[forecast_gallery, forecast_zip, forecast_log],
            )

        with gr.Column(visible=False, elem_classes=["page-shell"]) as satellite_page:
            gr.Markdown(
                "## Satélite y viento\n"
                "GOES-R, vapor de agua, rayos, fuego y vientos derivados DMW. 16 productos disponibles; todas las variables aparecen en tarjetas seleccionables.",
                elem_classes=["page-heading"],
            )
            diagnostic_mode = gr.Radio(
                ["Últimas horas (automático)", "Periodo histórico"],
                value="Últimas horas (automático)", label="Período",
            )
            with gr.Row(elem_classes=["panel"]):
                diagnostic_start = gr.DateTime(
                    value=recent_text, include_time=True, type="string", timezone="UTC",
                    label="Inicio (UTC) · para período histórico",
                )
                diagnostic_end = gr.DateTime(
                    value=now_text, include_time=True, type="string", timezone="UTC",
                    label="Fin (UTC) · para período histórico",
                )
            with gr.Row(elem_classes=["panel"]):
                diagnostic_hours = gr.Slider(
                    minimum=1, maximum=72, step=1, value=6,
                    label="Horas recientes",
                )
                diagnostic_cadence = gr.Dropdown(
                    ["10", "20", "30", "60"], value="30", label="Intervalo (minutos)"
                )
            gr.Markdown("Accesos rápidos para las últimas horas")
            with gr.Row(elem_classes=["quick-date"]):
                recent_6 = gr.Button("Últimas 6 h", variant="secondary")
                recent_12 = gr.Button("Últimas 12 h", variant="secondary")
                recent_24 = gr.Button("Últimas 24 h", variant="secondary")
            recent_6.click(lambda: 6, inputs=[], outputs=[diagnostic_hours])
            recent_12.click(lambda: 12, inputs=[], outputs=[diagnostic_hours])
            recent_24.click(lambda: 24, inputs=[], outputs=[diagnostic_hours])
            with gr.Row(elem_classes=["panel"]):
                diagnostic_satellite = gr.Dropdown(
                    ["G16", "G17", "G18", "G19"], value="G19", label="Satélite GOES"
                )
                diagnostic_scope = gr.Dropdown(
                    SCOPE_CHOICES, value="full", label="Cobertura ABI"
                )

            with gr.Accordion("Canales e imágenes GOES", open=True, elem_classes=["product-group"]):
                satellite_channels = selection_group(
                    satellite_options(satellite_groups["Canales e imágenes GOES"]),
                    value=["geocolor"], label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Agua atmosférica", open=True, elem_classes=["product-group"]):
                satellite_water = selection_group(
                    satellite_options(satellite_groups["Agua atmosférica"]),
                    value=[], label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Rayos e incendios", open=True, elem_classes=["product-group"]):
                satellite_lightning = selection_group(
                    satellite_options(satellite_groups["Rayos e incendios"]),
                    value=[], label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Viento satelital", open=True, elem_classes=["product-group"]):
                satellite_winds = selection_group(
                    satellite_options(satellite_groups["Viento satelital"]),
                    value=["dmw"], label="Producto", elem_classes=["product-tiles"],
                )
            diagnostic_high_resolution = gr.Checkbox(
                value=False,
                label="Alta resolución para B02, sandwich y GeoColor",
                info="Aumenta el detalle y el uso de memoria.",
            )
            diagnostic_region, diagnostic_min_lon, diagnostic_max_lon, diagnostic_min_lat, diagnostic_max_lat = _region_controls(gr, "diagnostic")
            with gr.Accordion("Opciones avanzadas", open=False, elem_classes=["panel"]):
                diagnostic_sources = gr.Checkbox(
                    value=False, label="Incluir archivos fuente en el ZIP (pueden ser grandes)"
                )
            diagnostic_button = gr.Button("Generar mapas satelitales", variant="primary")
            diagnostic_gallery = gr.Gallery(
                label="Resultados · agrupados por producto", columns=2,
                height="auto", object_fit="contain", elem_classes=["output-gallery"],
            )
            diagnostic_zip = gr.File(label="Descargar ZIP de resultados")
            with gr.Accordion("Registro técnico", open=False, elem_classes=["panel"]):
                diagnostic_log = gr.Textbox(label="Ejecución", lines=14, max_lines=28)
            diagnostic_button.click(
                run_diagnostic_ui,
                inputs=[
                    diagnostic_mode, diagnostic_start, diagnostic_end,
                    diagnostic_hours, diagnostic_cadence, diagnostic_satellite,
                    diagnostic_scope, satellite_channels, satellite_water,
                    satellite_lightning, satellite_winds, diagnostic_region,
                    diagnostic_min_lon, diagnostic_max_lon, diagnostic_min_lat,
                    diagnostic_max_lat, diagnostic_sources, diagnostic_high_resolution,
                ],
                outputs=[diagnostic_gallery, diagnostic_zip, diagnostic_log],
            )

        with gr.Column(visible=False, elem_classes=["page-shell"]) as ascat_page:
            gr.Markdown(
                "## ASCAT · Viento oceánico\n"
                "Vientos observados sobre el océano, con consulta reciente o histórica.",
                elem_classes=["page-heading"],
            )
            gr.Markdown(
                "**Reciente:** NOAA ERDDAP con respaldo NOAA STAR. "
                "**Histórico:** archivos diarios disponibles en NOAA STAR. "
                "Productos L3 diarios; también puede cargar un NetCDF L2."
            )
            ascat_region, ascat_min_lon, ascat_max_lon, ascat_min_lat, ascat_max_lat = _region_controls(gr, "ascat")
            ascat_sources = gr.Checkbox(
                value=False, label="Incluir NetCDF fuente en el ZIP"
            )
            ascat_noaa_button = gr.Button(
                "Consultar ASCAT reciente · NOAA", variant="primary"
            )
            with gr.Accordion("Consulta histórica NOAA STAR", open=True, elem_classes=["product-group"]):
                ascat_hist_mode = gr.Radio(
                    ["Fecha única", "Rango histórico"], value="Fecha única", label="Período"
                )
                with gr.Row():
                    ascat_hist_start = gr.DateTime(
                        value=(now - timedelta(days=3)).strftime("%Y-%m-%d"),
                        include_time=False, type="string", timezone="UTC",
                        label="Fecha UTC / inicio",
                    )
                    ascat_hist_end = gr.DateTime(
                        value=(now - timedelta(days=3)).strftime("%Y-%m-%d"),
                        include_time=False, type="string", timezone="UTC",
                        label="Fin del rango",
                    )
                with gr.Row():
                    ascat_hist_sat = gr.Dropdown(
                        [("Automático (Metop-C → B → A)", "Auto"),
                         ("Metop-C", "C"), ("Metop-B", "B"),
                         ("Metop-A (si existe)", "A")],
                        value="Auto", label="Satélite",
                    )
                    ascat_hist_orbit = gr.Dropdown(
                        ["Ambas", "Ascendente", "Descendente"],
                        value="Ambas", label="Pasada",
                    )
                gr.Markdown("Máximo **7 días por solicitud**. Solo se usan archivos históricos existentes.")
                ascat_hist_button = gr.Button(
                    "Consultar ASCAT histórico", variant="secondary"
                )
            ascat_upload = gr.File(
                label="Cargar NetCDF ASCAT L2 (.nc, .nc4 o .cdf)",
                file_types=[".nc", ".nc4", ".cdf"], type="filepath",
            )
            with gr.Row():
                ascat_net_button = gr.Button("Diagnosticar conexión NOAA", variant="secondary")
                ascat_button = gr.Button("Graficar archivo cargado", variant="secondary")
            ascat_gallery = gr.Gallery(
                label="Mapa ASCAT", columns=1, object_fit="contain",
                elem_classes=["output-gallery"],
            )
            ascat_zip = gr.File(label="Descargar ZIP de resultados")
            with gr.Accordion("Registro ASCAT", open=False, elem_classes=["panel"]):
                ascat_log = gr.Textbox(label="Fecha real y estado de los datos", lines=12)
            common_inputs = [
                ascat_region, ascat_min_lon, ascat_max_lon, ascat_min_lat,
                ascat_max_lat, ascat_sources,
            ]
            ascat_net_button.click(_ascat_red_diagnostico, inputs=[], outputs=[ascat_log])
            ascat_noaa_button.click(
                run_ascat_noaa, inputs=common_inputs,
                outputs=[ascat_gallery, ascat_zip, ascat_log],
            )
            ascat_hist_button.click(
                run_ascat_noaa_historico,
                inputs=[
                    ascat_hist_mode, ascat_hist_start, ascat_hist_end,
                    ascat_hist_sat, ascat_hist_orbit,
                ] + common_inputs,
                outputs=[ascat_gallery, ascat_zip, ascat_log],
            )
            ascat_button.click(
                run_ascat_uploaded, inputs=[ascat_upload] + common_inputs,
                outputs=[ascat_gallery, ascat_zip, ascat_log],
            )

        with gr.Column(visible=False, elem_classes=["page-shell"]) as history_page:
            gr.Markdown(
                "## Histórico y reanálisis\n"
                "Seleccione el rango de fechas y productos. 21 productos numéricos y 16 satelitales agrupados por tipo; todos aparecen en tarjetas y los resultados se ordenan por producto.",
                elem_classes=["page-heading"],
            )
            with gr.Row(elem_classes=["panel"]):
                history_start = gr.DateTime(
                    value=(now - timedelta(days=1)).strftime("%Y-%m-%d %H:%M"),
                    include_time=True, type="string", timezone="UTC",
                    label="Inicio (UTC)",
                )
                history_end = gr.DateTime(
                    value=now_text, include_time=True, type="string", timezone="UTC",
                    label="Fin (UTC)",
                )
            with gr.Row(elem_classes=["panel"]):
                history_cadence = gr.Dropdown(
                    ["1", "3", "6", "12", "24"], value="6",
                    label="Intervalo temporal (horas)",
                )
                history_fxx = selection_group(
                    [(f"+{h} h", str(h)) for h in (0, 6, 12, 24, 48)],
                    value=["0"], label="Plazos de pronóstico",
                    elem_classes=["choice-cards"],
                    info="Marque uno o varios plazos de pronóstico.",
                )
                history_models = selection_group(
                    MODEL_CHOICES, value=[],
                    label="Modelos numéricos · marque uno o varios",
                    elem_classes=["choice-cards"],
                    info="Los mapas GFS requieren GFS; los mapas ECMWF requieren ECMWF IFS.",
                )

            with gr.Accordion("Frentes fríos", open=True, elem_classes=["product-group"]):
                history_nwp_fronts = selection_group(
                    nwp_options(nwp_groups["Frentes fríos"]), value=[],
                    label="Producto FRAM", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Presión y temperatura", open=True, elem_classes=["product-group"]):
                history_nwp_thermo = selection_group(
                    nwp_options(nwp_groups["Presión y temperatura"]), value=[],
                    label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Humedad y convección", open=True, elem_classes=["product-group"]):
                history_nwp_moisture = selection_group(
                    nwp_options(nwp_groups["Humedad y convección"]), value=[],
                    label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Viento y dinámica", open=True, elem_classes=["product-group"]):
                history_nwp_dynamics = selection_group(
                    nwp_options(nwp_groups["Viento y dinámica"]), value=[],
                    label="Productos", elem_classes=["product-tiles"],
                )
            gr.Markdown("### Productos satelitales")
            with gr.Row(elem_classes=["panel"]):
                history_satellite = gr.Dropdown(
                    ["G16", "G17", "G18", "G19"], value="G19", label="Satélite GOES",
                )
                history_scope = gr.Dropdown(
                    SCOPE_CHOICES, value="full", label="Cobertura ABI",
                )
                history_output_mode = gr.Dropdown(
                    ["Mapas", "Datos fuente", "Mapas y datos fuente"],
                    value="Mapas", label="Salida",
                )
            with gr.Accordion("Canales e imágenes GOES", open=True, elem_classes=["product-group"]):
                history_sat_channels = selection_group(
                    satellite_options(satellite_groups["Canales e imágenes GOES"]),
                    value=[], label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Agua atmosférica", open=True, elem_classes=["product-group"]):
                history_sat_water = selection_group(
                    satellite_options(satellite_groups["Agua atmosférica"]),
                    value=[], label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Rayos e incendios", open=True, elem_classes=["product-group"]):
                history_sat_lightning = selection_group(
                    satellite_options(satellite_groups["Rayos e incendios"]),
                    value=[], label="Productos", elem_classes=["product-tiles"],
                )
            with gr.Accordion("Viento satelital", open=True, elem_classes=["product-group"]):
                history_sat_winds = selection_group(
                    satellite_options(satellite_groups["Viento satelital"]),
                    value=[], label="Producto", elem_classes=["product-tiles"],
                )
            history_high_resolution = gr.Checkbox(
                value=False, label="Alta resolución para B02, sandwich y GeoColor",
            )
            history_region, history_min_lon, history_max_lon, history_min_lat, history_max_lat = _region_controls(gr, "history")
            history_button = gr.Button("Procesar período", variant="primary")
            history_gallery = gr.Gallery(
                label="Resultados · agrupados por producto", columns=2,
                height="auto", object_fit="contain", elem_classes=["output-gallery"],
            )
            history_zip = gr.File(label="Descargar ZIP de resultados")
            with gr.Accordion("Registro técnico", open=False, elem_classes=["panel"]):
                history_log = gr.Textbox(label="Ejecución", lines=14, max_lines=28)
            history_button.click(
                run_history_ui,
                inputs=[
                    history_start, history_end, history_cadence, history_fxx,
                    history_models, history_nwp_fronts, history_nwp_thermo,
                    history_nwp_moisture, history_nwp_dynamics,
                    history_sat_channels, history_sat_water,
                    history_sat_lightning, history_sat_winds,
                    history_satellite, history_scope, history_region,
                    history_min_lon, history_max_lon, history_min_lat,
                    history_max_lat, history_output_mode, history_high_resolution,
                ],
                outputs=[history_gallery, history_zip, history_log],
            )


        with gr.Column(visible=False, elem_classes=["page-shell"]) as eumetsat_page:
            gr.Markdown(
                "## Satélites europeos · EUMETSAT\n"
                "Descargue los archivos nativos del Data Store para analizarlos en el visor o en otras herramientas.",
                elem_classes=["page-heading"],
            )
            gr.Markdown(
                "**Acceso:** requiere una cuenta EUMETSAT y sus claves API Consumer Key/Consumer Secret. "
                "Genérelas en el [portal de claves API](https://api.eumetsat.int/api-key). "
                "Las claves se solicitan al descargar y no se escriben en el registro ni dentro del ZIP.\n\n"
                "**Cobertura:** MTG/SEVIRI de 0° son discos completos centrados en Europa, África y el Atlántico; "
                "para Centroamérica use especialmente Metop/ASCAT, Metop/AVHRR y Sentinel-3, que son satélites de órbita polar."
            )
            with gr.Row(elem_classes=["panel"]):
                eum_start = gr.DateTime(
                    value=(now - timedelta(hours=6)).strftime("%Y-%m-%d %H:%M"),
                    include_time=True, type="string", timezone="UTC", label="Inicio UTC",
                )
                eum_end = gr.DateTime(
                    value=now_text, include_time=True, type="string", timezone="UTC",
                    label="Fin UTC",
                )
            with gr.Row(elem_classes=["panel"]):
                eum_key = gr.Textbox(label="Consumer Key", type="password")
                eum_secret = gr.Textbox(label="Consumer Secret", type="password")
                eum_limit = gr.Slider(minimum=1, maximum=5, step=1, value=1, label="Máximo de productos")

            with gr.Accordion("MTG / Meteosat · geostacionarios", open=True, elem_classes=["product-group"]):
                eum_geo_collection = gr.Dropdown(
                    [
                        ("MTG FCI L1c · resolución normal · disco 0°", "EO:EUM:DAT:0662"),
                        ("MTG FCI L1c · alta resolución · disco 0°", "EO:EUM:DAT:0665"),
                        ("Meteosat SEVIRI L1.5 · disco 0°", "EO:EUM:DAT:MSG:HRSEVIRI"),
                    ], value="EO:EUM:DAT:0662", label="Producto geostacionario",
                )
                eum_geo_button = gr.Button("Descargar datos MTG / Meteosat", variant="primary")
            with gr.Accordion("Metop / Sentinel-3 · órbita polar y cobertura global", open=True, elem_classes=["product-group"]):
                eum_polar_collection = gr.Dropdown(
                    [
                        ("Metop ASCAT · vientos costeros 12.5 km", "EO:EUM:DAT:METOP:OSI-104"),
                        ("Metop AVHRR · radiancias L1B globales", "EO:EUM:DAT:METOP:AVHRRL1"),
                        ("Sentinel-3 SLSTR · radiancias y temperaturas L1B", "EO:EUM:DAT:0411"),
                        ("Sentinel-3 OLCI · imágenes L1B", "EO:EUM:DAT:0409"),
                    ], value="EO:EUM:DAT:METOP:OSI-104", label="Producto polar",
                )
                eum_polar_button = gr.Button("Descargar datos Metop / Sentinel-3", variant="primary")

            eum_file = gr.File(label="Descargar ZIP EUMETSAT")
            with gr.Accordion("Registro EUMETSAT", open=True, elem_classes=["panel"]):
                eum_log = gr.Textbox(label="Estado de la búsqueda y descarga", lines=10, max_lines=24)
            eum_inputs = [eum_start, eum_end, eum_key, eum_secret, eum_limit]
            eum_geo_button.click(
                run_eumetsat_download,
                inputs=[eum_geo_collection] + eum_inputs,
                outputs=[eum_file, eum_log],
            )
            eum_polar_button.click(
                run_eumetsat_download,
                inputs=[eum_polar_collection] + eum_inputs,
                outputs=[eum_file, eum_log],
            )


        # ================================================================
        # ENDVI10 — mapas de NDVI y seguimiento decádico por coordenadas
        # ================================================================
        # Solo sugerir décadas completas + al menos tres días de latencia nominal.
        default_endvi = endvi_dias_retro(datetime(now.year, now.month, (21 if now.day >= 21 else 11 if now.day >= 11 else 1)), 1)[0]
        # La fecha sugerida es la última década cuyo final transcurrió antes de la latencia.
        while True:
            final_dia = 10 if default_endvi.day == 1 else 20 if default_endvi.day == 11 else calendar.monthrange(default_endvi.year, default_endvi.month)[1]
            if datetime(default_endvi.year, default_endvi.month, final_dia) <= (now - timedelta(days=3)):
                break
            default_endvi = endvi_dias_retro(default_endvi, 2)[0]
        with gr.Column(visible=False, elem_classes=["page-shell"]) as endvi_page:
            gr.Markdown(
                "## Vigor vegetativo · ENDVI10 (LSA SAF)\n"
                "AVHRR/Metop · composiciones de 10 días · NDVI físico, reescalado y calidad STM. "
                "El NDVI representa verdor vegetal, no humedad del suelo ni riego directo.",
                elem_classes=["page-heading"],
            )
            gr.Markdown(
                "**Acceso:** el servidor oficial OpenDAP puede solicitar usuario y contraseña de LSA SAF. "
                "[Registro LSA SAF](https://datalsasaf.lsasvcs.ipma.pt/) · "
                "[Carpeta original ENDVI10](https://datalsasaf.lsasvcs.ipma.pt/PRODUCTS/EPS/ENDVI10/NETCDF/) · "
                "[Ejemplo oficial OpenDAP](https://gitlab.com/helpdesk.landsaf/lsasaf_data_access/-/blob/main/examples/thredds/example_ndvi_thredds.ipynb). "
                "Si carga archivos NetCDF locales se utilizan esos archivos y no la red."
            )
            with gr.Row(elem_classes=["panel"]):
                endvi_year = gr.Number(label="Año", value=default_endvi.year, precision=0, minimum=2007, maximum=2100)
                endvi_month = gr.Dropdown(
                    choices=[(nombre + f' ({i:02d})', i) for i, nombre in enumerate(
                        ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio',
                         'Agosto','Septiembre','Octubre','Noviembre','Diciembre'], 1)],
                    value=default_endvi.month, label="Mes",
                )
                endvi_decade = gr.Dropdown(
                    choices=[("Primera · días 1–10", 1), ("Segunda · días 11–20", 11),
                             ("Tercera · día 21 a fin de mes", 21)],
                    value=default_endvi.day, label="Década",
                )
            with gr.Row(elem_classes=["panel"]):
                endvi_lat = gr.Number(label="Latitud del punto (°)", value=9.93, precision=5)
                endvi_lon = gr.Number(label="Longitud del punto (°)", value=-84.08, precision=5)
                endvi_sample = gr.Dropdown(
                    [("Píxel más cercano",1),("Promedio 3 × 3 píxeles",3),("Promedio 5 × 5 píxeles",5)],
                    value=3, label="Muestreo local",
                )
            with gr.Accordion("Región geográfica del mapa", open=True, elem_classes=["panel"]):
                gr.Markdown(
                    "**35 países de América disponibles:** al cambiar de país se actualizan la latitud, "
                    "la longitud del punto de referencia y los cuatro límites del mapa. "
                    "Puede modificar el punto manualmente después. "
                    "Las extensiones son recuadros geográficos aproximados, no máscaras de fronteras."
                )
                endvi_region = gr.Dropdown(
                    choices=[(f"{pais} · área principal" if pais in ("Canadá", "Chile", "Ecuador", "Estados Unidos") else pais, pais)
                             for pais in sorted(ENDVI_COUNTRIES)] +
                            [(f"Región · {zona}",zona) for zona in ENDVI_MACROREGIONS],
                    value="Costa Rica", label="País de América / región geográfica", filterable=True,
                )
                with gr.Row():
                    endvi_west = gr.Number(label="Oeste (editable para Personalizada)", value=-86.2, precision=3)
                    endvi_east = gr.Number(label="Este (editable para Personalizada)", value=-82.4, precision=3)
                    endvi_south = gr.Number(label="Sur (editable para Personalizada)", value=7.7, precision=3)
                    endvi_north = gr.Number(label="Norte (editable para Personalizada)", value=11.5, precision=3)
                endvi_region.change(
                    fn=endvi_region_defaults,
                    inputs=[endvi_region, endvi_lat, endvi_lon],
                    outputs=[endvi_lat,endvi_lon,endvi_west,endvi_east,endvi_south,endvi_north],
                    show_progress="hidden",
                )
            with gr.Row(elem_classes=["panel"]):
                endvi_quality = gr.Dropdown(
                    choices=["Estricto (cielo despejado, sin nieve ni AOT alto)",
                             "Sin nube y sin nieve", "Solo observaciones válidas"],
                    value="Estricto (cielo despejado, sin nieve ni AOT alto)",
                    label="Filtro de calidad STM",
                )
                endvi_n = gr.Dropdown(
                    choices=[("Década elegida",1), ("3 décadas",3),
                             ("6 décadas",6), ("9 décadas",9),
                             ("18 décadas · medio año",18), ("36 décadas · 1 año",36),
                             ("72 décadas · 2 años",72), ("108 décadas · 3 años",108),
                             ("180 décadas · 5 años",180)],
                    value=18, label="Período de la serie temporal (todas las décadas)",
                )
            with gr.Row(elem_classes=["panel"]):
                endvi_history = gr.Dropdown(
                    choices=[("No comparar", "Sin climatología"),
                             ("3 años anteriores", "3"), ("5 años anteriores", "5"),
                             ("8 años anteriores", "8"), ("10 años anteriores", "10"),
                             ("15 años anteriores", "15"),
                             ("Período personalizado (desde / hasta)", "Personalizado")],
                    value="5", label="Climatología de referencia por década",
                )
                endvi_hist_start = gr.Number(
                    label="Desde año (solo para Personalizado)",
                    value=max(2007,default_endvi.year-10), precision=0,
                    minimum=2007, maximum=2100,
                )
                endvi_hist_end = gr.Number(
                    label="Hasta año (solo para Personalizado)",
                    value=default_endvi.year-1, precision=0,
                    minimum=2007, maximum=2100,
                )
            gr.Markdown(
                "La curva histórica se calcula **para cada una de las décadas** seleccionadas, "
                "utilizando los mismos días del calendario (1, 11 o 21) de todos los años de base. "
                "Grafica media, P10–P90 y anomalías, sin inventar valores faltantes. "
                "**Ejemplo:** 36 décadas y 10 años pueden requerir unas 396 lecturas OpenDAP; "
                "los tiempos de respuesta dependen de LSA SAF. "
                "Los períodos cortos de 3–10 años son referencias históricas elegidas, no una normal de 30 años."
            )
            with gr.Accordion("Cuenta LSA SAF o archivos NetCDF locales", open=False, elem_classes=["panel"]):
                gr.Markdown(
                    "Las credenciales son de **LSA SAF**, no las API Consumer Key/Secret del EUMETSAT Data Store. "
                    "No quedan incluidas en los resultados. Al usar el enlace público de Gradio, "
                    "evite introducir contraseñas si no confía en quienes puedan acceder a la sesión."
                )
                with gr.Row():
                    endvi_user = gr.Textbox(label="Usuario LSA SAF", type="password")
                    endvi_pass = gr.Textbox(label="Contraseña LSA SAF", type="password")
                endvi_upload = gr.File(
                    label="NetCDF ENDVI10 .nc/.nc4/.cdf (opcional; uno o varios)",
                    file_types=[".nc", ".nc4", ".cdf"], file_count="multiple", type="filepath",
                )
            endvi_generate = gr.Button("Generar NDVI, climatología y anomalías", variant="primary")
            endvi_gallery = gr.Gallery(label="ENDVI10 · Mapas y evolución temporal", columns=1, object_fit="contain", height="auto")
            endvi_zip = gr.File(label="Descargar mapas PNG, serie CSV y metodología (ZIP)")
            with gr.Accordion("Diagnóstico de lectura y calidad", open=True, elem_classes=["panel"]):
                endvi_log = gr.Textbox(label="Estado de los NetCDF y filtros", lines=10, max_lines=25)
            endvi_generate.click(
                run_endvi10,
                inputs=[endvi_year,endvi_month,endvi_decade,endvi_lat,endvi_lon,
                        endvi_region,endvi_west,endvi_east,endvi_south,endvi_north,
                        endvi_quality,endvi_sample,endvi_n,endvi_history,
                        endvi_hist_start,endvi_hist_end,
                        endvi_user,endvi_pass,endvi_upload],
                outputs=[endvi_gallery,endvi_zip,endvi_log],
            )

        pages = {
            "home": home_page,
            "forecast": forecast_page,
            "satellite": satellite_page,
            "ascat": ascat_page,
            "history": history_page,
            "eumetsat": eumetsat_page,
            "endvi": endvi_page,
        }
        nav_buttons = {
            "home": nav_home,
            "forecast": nav_forecast,
            "satellite": nav_satellite,
            "ascat": nav_ascat,
            "history": nav_history,
            "eumetsat": nav_eumetsat,
            "endvi": nav_endvi,
        }
        nav_outputs = list(pages.values()) + list(nav_buttons.values())

        def navigate(target):
            return (
                [gr.update(visible=(name == target)) for name in pages]
                + [
                    gr.update(variant="primary" if name == target else "secondary")
                    for name in nav_buttons
                ]
            )

        for name, button in nav_buttons.items():
            button.click(
                lambda selected=name: navigate(selected),
                inputs=[], outputs=nav_outputs,
            )
        home_forecast.click(lambda: navigate("forecast"), inputs=[], outputs=nav_outputs)
        home_satellite.click(lambda: navigate("satellite"), inputs=[], outputs=nav_outputs)
        home_ascat.click(lambda: navigate("ascat"), inputs=[], outputs=nav_outputs)
        home_history.click(lambda: navigate("history"), inputs=[], outputs=nav_outputs)
        home_eumetsat.click(lambda: navigate("eumetsat"), inputs=[], outputs=nav_outputs)
        home_endvi.click(lambda: navigate("endvi"), inputs=[], outputs=nav_outputs)

    return demo


def launch_app(*, debug_colab=False, mostrar_integrado=True, compartir=True):
    """Inicia el visor, imprime el enlace público y mantiene vivo el servidor."""

    import sys
    import traceback
    from threading import Event

    global _ACTIVE_DEMO
    if _ACTIVE_DEMO is not None:
        try:
            _ACTIVE_DEMO.close()
        except Exception:
            pass
        _ACTIVE_DEMO = None

    en_colab = "google.colab" in sys.modules
    print("Entorno:", "Google Colab" if en_colab else "Jupyter / Python", flush=True)
    print("Preparando el visor; todavía no se descargan datos meteorológicos.", flush=True)
    try:
        _ACTIVE_DEMO = build_app()
        config = _ACTIVE_DEMO.get_config_file()
        print(f"Interfaz construida: {len(config.get('components', []))} componentes.", flush=True)
        result = _ACTIVE_DEMO.launch(
            share=bool(compartir),
            inline=bool(mostrar_integrado and en_colab),
            inbrowser=False,
            debug=bool(debug_colab and en_colab),
            show_error=True,
            prevent_thread_lock=True,
            css=APP_CSS,
            height=900,
        )

        share_url = None
        if isinstance(result, tuple) and len(result) >= 3:
            share_url = result[2]
        share_url = share_url or getattr(_ACTIVE_DEMO, "share_url", None)
        if share_url:
            print("\nENLACE PÚBLICO PARA ACCEDER AL VISOR:", flush=True)
            print(str(share_url), flush=True)
        else:
            print(
                "\nNo se generó el enlace público. Use la vista integrada debajo de esta celda "
                "mientras el runtime permanezca activo.",
                flush=True,
            )
        print("Mantenga esta celda en ejecución mientras utiliza el visor.", flush=True)
        try:
            Event().wait()
        except KeyboardInterrupt:
            print("Cerrando el visor...", flush=True)
            try:
                _ACTIVE_DEMO.close()
            except Exception:
                pass
        return result
    except Exception as exc:
        print("\nERROR AL INICIAR LA INTERFAZ:", type(exc).__name__, str(exc), flush=True)
        traceback.print_exc()
        raise


def prueba_gradio_minima():
    """Prueba opcional de enlace externo, sin GOES, ECMWF ni ASCAT.

    Ejecutar solo para aislar errores de red y de Gradio.
    """
    import gradio as gr
    demo = gr.Interface(lambda x: f"Gradio funciona: {x}",
                        inputs=gr.Textbox(label="Prueba"),
                        outputs=gr.Textbox(label="Resultado"),
                        title="Verificación de Gradio / Colab")
    return demo.launch(share=True, inline=True, debug=True, show_error=True)

# En Colab, ejecutar launch_app() desde la última celda.
# Si el enlace no funciona, detener la última celda y ejecutar:
# prueba_gradio_minima()


## Navegación y acceso

1. Ejecute todas las celdas en orden.
2. En la última celda, espere el rótulo **ENLACE PÚBLICO PARA ACCEDER AL VISOR** y abra la dirección gradio.live.
3. En el visor, use los botones superiores o las tarjetas de inicio para cambiar de sección.
4. En Pronóstico, Satélite e Histórico, todas las variables están organizadas en tarjetas seleccionables por tipo. En la galería, cada mapa muestra el producto y luego sus fechas o plazos.
5. Para fechas, use el calendario; los horarios y períodos se interpretan en UTC. Deje la última celda ejecutándose mientras use el enlace.

Si Colab no logra crear el túnel público, el visor integrado seguirá disponible mientras el runtime esté activo.


### Datos europeos
La sección **EUMETSAT** descarga productos nativos MTG/SEVIRI, Metop y Sentinel-3. Para descargar, genere claves Consumer Key/Consumer Secret en el portal de EUMETSAT e introdúzcalas en el visor. Los archivos se entregan en un ZIP con manifiesto e identificadores de colección.

5. En **NDVI · ENDVI10**, seleccione año, mes, década, coordenadas y filtro STM. Para OpenDAP, use su cuenta LSA SAF, no las claves EUMETSAT Data Store; alternativamente cargue NetCDF propios. Las imágenes se producen únicamente después de seleccionar **Generar NDVI**.

6. **ENDVI10 América:** seleccione país; el punto y los límites se actualizan en el formulario. Escoja duración de serie y climatología de N años anteriores o el rango de años personalizado. El ZIP incluye mapas, evolución completa, bandas históricas por década y CSV con los años contribuyentes.


In [ ]:
# Mantiene activo el servidor, imprime el enlace público y muestra la interfaz integrada.
launch_app(debug_colab=False, mostrar_integrado=True, compartir=True)


Entorno: Google Colab
Preparando el visor; todavía no se descargan datos meteorológicos.
Interfaz construida: 285 componentes.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ced568836d22422c08.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



ENLACE PÚBLICO PARA ACCEDER AL VISOR:
https://ced568836d22422c08.gradio.live
Mantenga esta celda en ejecución mientras utiliza el visor.


/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.13/dist-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
